In [2]:
from google.colab import drive
import os

os.listdir('/content/drive')


['MyDrive']

In [6]:
import os

print(os.listdir('/content/drive/MyDrive/Firm_Clusters')[:20])


['pipeline']


In [18]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

BOOTSTRAP


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# COLAB CELL: survey the real Drive layout and pin the artefact paths
#
# Run this BEFORE colab_p3a_industrial.py or colab_p4a_lineage.py.
#
# Why it exists
# -------------
# The handoff documents paths that do not match the Drive:
#
#   handoff says                          drive actually has
#   ---------------------------           ---------------------------------
#   Firm_Clusters/full_2012_2026/clusters  Firm_Clusters/clusters        AND
#                                         Firm_Clusters/full_2012_2026/clusters
#   MyDrive/India_Firms/India_Firms.parquet Firm_Clusters/India_Firms.parquet
#
# `clusters/` and `edges/` exist in TWO places. Picking one silently would read
# whichever copy is stale or empty and produce plausible-looking nonsense, so
# this cell prints file counts, sizes and samples for both and lets the numbers
# decide.
#
# It writes nothing.
# ============================================================================

import json
from pathlib import Path

import colab_utils as cu

MAX_DEPTH = 3
SAMPLE_FILES = 3

print(cu.drive_report())
print()

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
print()


def survey(path, depth=0, max_depth=MAX_DEPTH):
    """Yield (relative_path, dir, n_files, bytes, samples) breadth-first."""
    root = Path(path)
    queue = [(root, depth)]
    while queue:
        current, d = queue.pop(0)
        files, dirs, total = [], 0, 0
        try:
            entries = sorted(current.iterdir())
        except OSError as exc:
            print(f"  !! cannot list {current}: {exc}")
            continue
        for entry in entries:
            try:
                if entry.is_dir():
                    dirs += 1
                else:
                    files.append(entry)
                    total += entry.stat().st_size
            except OSError:
                continue
        yield current, d, files, dirs, total
        if d < max_depth:
            for entry in entries:
                try:
                    if entry.is_dir():
                        queue.append((entry, d + 1))
                except OSError:
                    continue


def human(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}" if unit != "B" else f"{int(n)} B"
        n /= 1024.0
    return f"{n:.1f} TB"


# ---------------------------------------------------------------------------
# 1. Tree survey
# ---------------------------------------------------------------------------

print("=" * 96)
print(f"TREE SURVEY -- {PROJECT_ROOT}  (depth <= {MAX_DEPTH})")
print("=" * 96)
print(f"{'path':62s} {'files':>7s} {'dirs':>6s} {'size':>11s}")

inventory = {}
for current, depth, files, dirs, total in survey(PROJECT_ROOT):
    rel = current.relative_to(PROJECT_ROOT)
    label = str(rel) if str(rel) != "." else "."
    inventory[str(rel)] = {
        "n_files": len(files),
        "n_dirs": dirs,
        "bytes": total,
        "samples": [f.name for f in files[:SAMPLE_FILES]],
    }
    print(f"{label:62s} {len(files):>7d} {dirs:>6d} {human(total):>11s}")

# ---------------------------------------------------------------------------
# 2. Ambiguity report -- the part that actually matters
# ---------------------------------------------------------------------------

print()
print("=" * 96)
print("AMBIGUOUS FOLDER NAMES (same name at more than one depth)")
print("=" * 96)

by_name = {}
for rel, info in inventory.items():
    if rel == ".":
        continue
    leaf = Path(rel).name
    by_name.setdefault(leaf, []).append((rel, info))

ambiguous = {k: v for k, v in by_name.items() if len(v) > 1}
if not ambiguous:
    print("  none")
for leaf, entries in sorted(ambiguous.items()):
    print()
    print(f"  {leaf}/  appears {len(entries)} times:")
    for rel, info in sorted(entries, key=lambda t: t[1]["n_files"], reverse=True):
        verdict = ""
        if info["n_files"] == 0 and info["bytes"] == 0:
            verdict = "   <-- EMPTY, ignore this copy"
        print(
            f"    {rel:56s} files={info['n_files']:>6d} "
            f"size={human(info['bytes']):>10s}{verdict}"
        )
        if info["samples"]:
            print(f"      samples: {info['samples']}")

# ---------------------------------------------------------------------------
# 3. The specific artefacts the cells need
# ---------------------------------------------------------------------------

print()
print("=" * 96)
print("ARTEFACTS THE CELLS NEED")
print("=" * 96)

wanted = ("clusters", "edges", "events", "external")
resolved = cu.resolve_artefacts(PROJECT_ROOT, wanted=wanted, verbose=False)

for leaf in wanted:
    matches = cu.find_all_artefact(PROJECT_ROOT, leaf)
    print()
    print(f"  {leaf}/  ({cu.ARTEFACTS.get(leaf, '')})")
    if not matches:
        print("    NOT FOUND anywhere under the project root")
        continue
    for path, depth in matches:
        try:
            files = list(path.iterdir())
            n = len(files)
            size = sum(f.stat().st_size for f in files if f.is_file())
        except OSError:
            n, size = -1, 0
        print(f"    depth={depth}  {path}")
        print(f"      entries={n}  size={human(size)}")
        try:
            names = sorted(f.name for f in files)[:4]
            print(f"      sample names: {names}")
        except OSError:
            pass

# Specific files the cells open by name.
print()
print("  named files:")
for rel in (
    "full_2012_2026/events/canonical_event_base.parquet",
    "full_2012_2026/events/event_local_cluster_membership.parquet",
    "India_Firms.parquet",
    "full_2012_2026/external/osm_facilities",
):
    target = PROJECT_ROOT / rel
    state = "present" if cu.path_ok(target) else "ABSENT at this path"
    extra = ""
    if not cu.path_ok(target):
        # Search by leaf name, since the layout may differ.
        leaf = Path(rel).name
        hits = cu.find_all_artefact(PROJECT_ROOT, leaf)
        if hits:
            extra = "  found instead at: " + ", ".join(str(h[0]) for h in hits[:3])
    print(f"    {rel:62s} {state}{extra}")

# ---------------------------------------------------------------------------
# 4. Verdict
# ---------------------------------------------------------------------------

print()
print("=" * 96)
print("VERDICT")
print("=" * 96)

real_clusters = None
matches = cu.find_all_artefact(PROJECT_ROOT, "clusters")
if matches:
    scored = []
    for path, depth in matches:
        try:
            files = [f for f in path.iterdir() if f.is_file()]
        except OSError:
            files = []
        scored.append((len(files), depth, path))
    scored.sort(reverse=True)
    best = scored[0]
    real_clusters = best[2]
    print(f"  clusters/ : {len(scored)} copies found")
    for count, depth, path in scored:
        print(f"    {count:>6d} files  depth={depth}  {path}")
    print()
    print(f"  CANDIDATE for P4a: {real_clusters}")
    if len(scored) > 1 and scored[0][0] == scored[1][0]:
        print("  !! The top two copies have the SAME file count. Do not guess --")
        print("     open both and compare names/dates before committing.")
else:
    print("  clusters/ NOT FOUND")

print()
print("NEXT")
print("  Paste this output back. The P3a/P4a cells will be pinned to the paths")
print("  confirmed here, so they stop guessing and stop failing on FileNotFound.")
print("=" * 96)


In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = True          # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
EVENT_LON_COL = "centroid_lon"
EVENT_LAT_COL = "centroid_lat"

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")
    return files


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

for col in (EVENT_LON_COL, EVENT_LAT_COL):
    if col not in event_columns:
        print()
        print(f"!! {col} is not in the event table. Set EVENT_LON_COL /")
        print("   EVENT_LAT_COL above to the real centroid columns and re-run.")
        print("   Do not guess: a wrong centroid silently shifts every feature.")
        raise KeyError(col)

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("3. Then set INSPECT_ONLY = False and re-run the whole cell.")
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")
    raise SystemExit(0)

# ===========================================================================
# 5. COMPUTE  (only reached once INSPECT_ONLY = False)
# ===========================================================================

import shapely
from shapely import STRtree
from shapely.geometry import LineString, MultiLineString

print()
print("=" * 78)
print("P3a STEP 3 -- PROJECTION AND TREES")
print("=" * 78)

from pyproj import Transformer

to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


def project_points(lon, lat):
    """Vectorised lon/lat -> Equal Earth metres."""
    x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                               np.asarray(lat, dtype="float64"))
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    # Infinite transform results mean the point fell outside the projection.
    bad = ~np.isfinite(x) | ~np.isfinite(y)
    if bad.any():
        print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
        x[bad] = np.nan
        y[bad] = np.nan
    return x, y


# --- per-category point trees ----------------------------------------------
category_trees: dict = {}
for cat in ind.CATEGORIES:
    pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
    if pts:
        arr = np.array(pts, dtype="float64")
        catx, caty = project_points(arr[:, 0], arr[:, 1])
        good = np.isfinite(catx) & np.isfinite(caty)
        category_trees[cat] = np.column_stack([catx[good], caty[good]])
    else:
        category_trees[cat] = None
    n = 0 if category_trees[cat] is None else len(category_trees[cat])
    print(f"  {cat:14s} point tree: {n:>8,d}")

# --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
boundary_xy = None
if boundary_chunks:
    raw = np.concatenate(boundary_chunks, axis=0)
    bx, by = project_points(raw[:, 0], raw[:, 1])
    good = np.isfinite(bx) & np.isfinite(by)
    boundary_xy = np.column_stack([bx[good], by[good]])
    print(f"  boundary samples: {len(boundary_xy):,d} "
          f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
else:
    print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

# --- polygons for INSIDE_FACILITY -------------------------------------------
# The original geometries are reused here, not the densified ring samples: the
# point-in-polygon answer must come from valid closed rings.
polys = [rec["geom"] for rec in records if rec["geom"] is not None]
polys = [p for p in polys if p.is_valid and not p.is_empty]
inside_tree = STRtree(polys) if polys else None
print(f"  STRtree polygons: {len(polys)}")
if not polys:
    print("  no usable polygons -> INSIDE_FACILITY will be NaN")

print()
print("=" * 78)
print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
print("=" * 78)

OUT_DIR.mkdir(parents=True, exist_ok=True)
read_cols = [EVENT_LON_COL, EVENT_LAT_COL]
read_cols += list(present_flare.values())
read_cols += [c for c in ("event_id",) if c in event_columns]
read_cols = list(dict.fromkeys(read_cols))
print("reading columns:", read_cols)

need_cols = set(read_cols)
null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
rows_written = 0
chunk_files = []
events_seen = 0

batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
for batch_no, batch in enumerate(batch_iter):
    frame = batch.to_pandas()
    if frame.empty:
        continue
    if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
        break

    lon = pd.to_numeric(frame[EVENT_LON_COL], errors="coerce").to_numpy("float64")
    lat = pd.to_numeric(frame[EVENT_LAT_COL], errors="coerce").to_numpy("float64")
    x, y = project_points(lon, lat)
    pts = np.column_stack([x, y])
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        continue
    if not finite.all():
        # Unprojectable centroids still need their 18 keys emitted as NaN.
        pts = np.where(finite[:, None], pts, np.nan)

    inside = None
    if inside_tree is not None:
        geoms = shapely.points(x, y)
        hits = inside_tree.query(geoms, predicate="intersects")
        inside = np.zeros(len(pts), dtype="float64")
        inside[np.unique(hits[0])] = 1.0
        del geoms

    boundary = None
    if boundary_xy is not None:
        boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
        full = np.full(len(pts), np.nan)
        full[finite] = boundary
        boundary = full

    flare_site = None
    if inside_tree is not None:
        # A centroid inside any polygon is treated as flare-capable only when
        # the polygon belongs to a flare-capable category; the STRtree here is
        # over all polygons, so fall back to the union distance plus the
        # category trees for the per-category test.
        flare_site = np.zeros(len(pts), dtype=bool)
        flare_pts = []
        for cat in ind.FLARE_SITE_CATEGORIES:
            tree = category_trees.get(cat)
            if tree is None or len(tree) == 0:
                continue
            d = ind.nearest_distance(pts[finite], tree)
            sub = np.zeros(int(finite.sum()), dtype=bool)
            sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
            flare_pts.append(sub)
        if flare_pts:
            flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
        else:
            flare_site = None

    support = {}
    for key, col in present_flare.items():
        support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

    features = ind.compute_industrial_features(
        pts,
        category_trees,
        crs=TARGET_CRS,
        inside_flags=inside,
        boundary_distance_m=boundary,
        category_is_flare_site=flare_site,
        **support,
    )

    out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
    if "event_id" in frame.columns:
        out.insert(0, "event_id", frame["event_id"].to_numpy())

    for name in ind.INDUSTRIAL_FEATURES:
        null_counts[name] += int(out[name].isna().sum())

    path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
    out.to_parquet(path, index=False, compression="snappy")
    chunk_files.append({"file": path.name, "rows": len(out)})
    rows_written += len(out)
    events_seen += len(out)

    if batch_no % 5 == 0:
        print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
              f"total={rows_written:>10,d}")

print()
print(f"rows written: {rows_written:,}")

# ---------------------------------------------------------------------------
# 6. QC
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a QC")
print("=" * 78)
print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
summary = {}
for name in ind.INDUSTRIAL_FEATURES:
    nulls = null_counts[name]
    frac = 100.0 * nulls / rows_written if rows_written else float("nan")
    summary[name] = {"nan_fraction": frac, "nan_count": nulls}
    print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "event_base": str(EVENT_BASE),
    "events_total_in_table": event_pf.metadata.num_rows,
    "rows_written": rows_written,
    "target_crs": TARGET_CRS,
    "chunk_rows": CHUNK_ROWS,
    "max_events": MAX_EVENTS,
    "boundary_densify_m": BOUNDARY_DENSIFY_M,
    "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
    "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
    "feature_order": ind.INDUSTRIAL_FEATURES,
    "null_fraction": summary,
    "chunks": chunk_files,
}
with open(OUT_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print("Wrote", OUT_DIR / "manifest.json")

print()
print("Status to record in feature_ledger.py once you have inspected the")
print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
print("runtime_verified stays False until P2/P4 actually execute them.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = True          # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("3. Then set INSPECT_ONLY = False and re-run the whole cell.")
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")
    raise SystemExit(0)

# ===========================================================================
# 5. COMPUTE  (only reached once INSPECT_ONLY = False)
# ===========================================================================

import shapely
from shapely import STRtree
from shapely.geometry import LineString, MultiLineString

print()
print("=" * 78)
print("P3a STEP 3 -- PROJECTION AND TREES")
print("=" * 78)

from pyproj import Transformer

to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


def project_points(lon, lat):
    """Vectorised lon/lat -> Equal Earth metres."""
    x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                               np.asarray(lat, dtype="float64"))
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    # Infinite transform results mean the point fell outside the projection.
    bad = ~np.isfinite(x) | ~np.isfinite(y)
    if bad.any():
        print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
        x[bad] = np.nan
        y[bad] = np.nan
    return x, y


# --- per-category point trees ----------------------------------------------
category_trees: dict = {}
for cat in ind.CATEGORIES:
    pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
    if pts:
        arr = np.array(pts, dtype="float64")
        catx, caty = project_points(arr[:, 0], arr[:, 1])
        good = np.isfinite(catx) & np.isfinite(caty)
        category_trees[cat] = np.column_stack([catx[good], caty[good]])
    else:
        category_trees[cat] = None
    n = 0 if category_trees[cat] is None else len(category_trees[cat])
    print(f"  {cat:14s} point tree: {n:>8,d}")

# --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
boundary_xy = None
if boundary_chunks:
    raw = np.concatenate(boundary_chunks, axis=0)
    bx, by = project_points(raw[:, 0], raw[:, 1])
    good = np.isfinite(bx) & np.isfinite(by)
    boundary_xy = np.column_stack([bx[good], by[good]])
    print(f"  boundary samples: {len(boundary_xy):,d} "
          f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
else:
    print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

# --- polygons for INSIDE_FACILITY -------------------------------------------
# The original geometries are reused here, not the densified ring samples: the
# point-in-polygon answer must come from valid closed rings.
polys = [rec["geom"] for rec in records if rec["geom"] is not None]
polys = [p for p in polys if p.is_valid and not p.is_empty]
inside_tree = STRtree(polys) if polys else None
print(f"  STRtree polygons: {len(polys)}")
if not polys:
    print("  no usable polygons -> INSIDE_FACILITY will be NaN")

print()
print("=" * 78)
print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
print("=" * 78)

OUT_DIR.mkdir(parents=True, exist_ok=True)
read_cols = [X_COL, Y_COL]
read_cols += list(present_flare.values())
read_cols += [c for c in ("event_id",) if c in event_columns]
read_cols = list(dict.fromkeys(read_cols))
print("reading columns:", read_cols)

need_cols = set(read_cols)
null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
rows_written = 0
chunk_files = []
events_seen = 0

batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
for batch_no, batch in enumerate(batch_iter):
    frame = batch.to_pandas()
    if frame.empty:
        continue
    if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
        break

    col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
    col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
    if COORD_KIND == "projected metres":
        # Already Equal Earth metres. Projecting again would rescale every
        # distance, so this branch deliberately does nothing.
        x, y = col_x, col_y
    else:
        x, y = project_points(col_x, col_y)
    pts = np.column_stack([x, y])
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        continue
    if not finite.all():
        # Unprojectable centroids still need their 18 keys emitted as NaN.
        pts = np.where(finite[:, None], pts, np.nan)

    inside = None
    if inside_tree is not None:
        geoms = shapely.points(x, y)
        hits = inside_tree.query(geoms, predicate="intersects")
        inside = np.zeros(len(pts), dtype="float64")
        inside[np.unique(hits[0])] = 1.0
        del geoms

    boundary = None
    if boundary_xy is not None:
        boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
        full = np.full(len(pts), np.nan)
        full[finite] = boundary
        boundary = full

    flare_site = None
    if inside_tree is not None:
        # A centroid inside any polygon is treated as flare-capable only when
        # the polygon belongs to a flare-capable category; the STRtree here is
        # over all polygons, so fall back to the union distance plus the
        # category trees for the per-category test.
        flare_site = np.zeros(len(pts), dtype=bool)
        flare_pts = []
        for cat in ind.FLARE_SITE_CATEGORIES:
            tree = category_trees.get(cat)
            if tree is None or len(tree) == 0:
                continue
            d = ind.nearest_distance(pts[finite], tree)
            sub = np.zeros(int(finite.sum()), dtype=bool)
            sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
            flare_pts.append(sub)
        if flare_pts:
            flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
        else:
            flare_site = None

    support = {}
    for key, col in present_flare.items():
        support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

    features = ind.compute_industrial_features(
        pts,
        category_trees,
        crs=TARGET_CRS,
        inside_flags=inside,
        boundary_distance_m=boundary,
        category_is_flare_site=flare_site,
        **support,
    )

    out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
    if "event_id" in frame.columns:
        out.insert(0, "event_id", frame["event_id"].to_numpy())

    for name in ind.INDUSTRIAL_FEATURES:
        null_counts[name] += int(out[name].isna().sum())

    path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
    out.to_parquet(path, index=False, compression="snappy")
    chunk_files.append({"file": path.name, "rows": len(out)})
    rows_written += len(out)
    events_seen += len(out)

    if batch_no % 5 == 0:
        print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
              f"total={rows_written:>10,d}")

print()
print(f"rows written: {rows_written:,}")

# ---------------------------------------------------------------------------
# 6. QC
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a QC")
print("=" * 78)
print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
summary = {}
for name in ind.INDUSTRIAL_FEATURES:
    nulls = null_counts[name]
    frac = 100.0 * nulls / rows_written if rows_written else float("nan")
    summary[name] = {"nan_fraction": frac, "nan_count": nulls}
    print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "event_base": str(EVENT_BASE),
    "events_total_in_table": event_pf.metadata.num_rows,
    "rows_written": rows_written,
    "target_crs": TARGET_CRS,
    "chunk_rows": CHUNK_ROWS,
    "max_events": MAX_EVENTS,
    "boundary_densify_m": BOUNDARY_DENSIFY_M,
    "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
    "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
    "feature_order": ind.INDUSTRIAL_FEATURES,
    "null_fraction": summary,
    "chunks": chunk_files,
}
with open(OUT_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print("Wrote", OUT_DIR / "manifest.json")

print()
print("Status to record in feature_ledger.py once you have inspected the")
print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
print("runtime_verified stays False until P2/P4 actually execute them.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# COLAB CELL: project reconnaissance -- what actually exists, and cheaply
#
# Purpose
# -------
# One consolidated report of every input the 141 features need: real folder
# names, real column names, real dtypes, real geometry types. Stops all the
# guessing that has been producing wrong paths and wrong column names.
#
# Cost control -- this does NOT scan the data
# -------------------------------------------
#   * folder listings      : name-only, no file reads
#   * parquet schemas      : ParquetFile(...).schema_arrow -- FOOTER METADATA
#                            ONLY, no row group is decoded
#   * data samples         : exactly ONE file per *distinct schema signature*,
#                            read with a single row group
#   * cluster files        : schema only; dates come from the 5,337 filenames
#                            rather than from opening them
# So the whole report costs a few thousand file-header reads, not 15M rows.
#
# Output: a printed report plus qc/project_recon.json
# ============================================================================

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu

RECON_VERSION = "1.0"
SAMPLE_ROWS = 5
SAMPLE_MAX_FILES = 40          # cap on distinct schemas we open data for

print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
report = {
    "recon_version": RECON_VERSION,
    "project_root": str(PROJECT_ROOT),
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
}
t_start = time.time()


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} TB"


def schema_of(path):
    """Column names + types from the parquet footer. No data is decoded."""
    try:
        pf = pq.ParquetFile(path)
        return {
            "columns": list(pf.schema_arrow.names),
            "types": {f.name: str(f.type) for f in pf.schema_arrow.schema},
            "n_rows": pf.metadata.num_rows,
            "n_row_groups": pf.metadata.num_row_groups,
        }
    except Exception as exc:                        # noqa: BLE001
        return {"error": f"{exc.__class__.__name__}: {exc}"}


def signature(info):
    """Stable identity for 'files that look alike', used to sample once."""
    return tuple(sorted(info.get("columns", [])))


# ===========================================================================
# 1. FOLDER INVENTORY -- names only
# ===========================================================================

print()
print("=" * 92)
print("1. FOLDER INVENTORY (name listing only, no file reads)")
print("=" * 92)

folders = [
    ("events", cu.find_artefact(PROJECT_ROOT, "events")[0]),
    ("clusters", cu.find_artefact(PROJECT_ROOT, "clusters")[0]),
    ("edges", cu.find_artefact(PROJECT_ROOT, "edges")[0]),
    ("mappings", cu.find_artefact(PROJECT_ROOT, "mappings")[0]),
    ("external", cu.find_artefact(PROJECT_ROOT, "external")[0]),
    ("feature_audit", cu.find_artefact(PROJECT_ROOT, "feature_audit")[0]),
    ("qc", cu.path_ok(PROJECT_ROOT / "qc") and PROJECT_ROOT / "qc" or None),
    ("qc_canon", cu.path_ok(_CANON / "qc") and _CANON / "qc" or None),
]
report["folders"] = {}
print(f"{'label':16s} {'files':>7s} {'parquet':>8s} {'csv':>6s} "
      f"{'json':>6s} {'other':>6s}  path")
for label, path in folders:
    if path is None:
        print(f"{label:16s} {'-':>7s}  (not found)")
        report["folders"][label] = None
        continue
    try:
        names = [p.name for p in path.iterdir()]
    except OSError as exc:
        print(f"{label:16s}  !! {exc}")
        continue
    ext = Counter(Path(n).suffix.lower() for n in names)
    dirs = [n for n in names if (path / n).is_dir()]
    files = [n for n in names if not (path / n).is_dir()]
    fext = Counter(Path(n).suffix.lower() for n in files)
    other = len(files) - fext.get(".parquet", 0) - fext.get(".csv", 0) - fext.get(
        ".json", 0
    )
    print(f"{label:16s} {len(files):>7d} {fext.get('.parquet',0):>8d} "
          f"{fext.get('.csv',0):>6d} {fext.get('.json',0):>6d} {other:>6d}  "
          f"{path.name}/")
    if dirs:
        print(f"{'':16s} {'':7s} subdirs: {sorted(dirs)[:8]}")
    report["folders"][label] = {
        "path": str(path),
        "n_files": len(files),
        "n_dirs": len(dirs),
        "extensions": dict(fext),
        "subdirs": sorted(dirs)[:20],
        "file_samples": sorted(files)[:8],
    }

# ===========================================================================
# 2. EVENT BASE -- the frozen 7,265,849-row table
# ===========================================================================

print()
print("=" * 92)
print("2. EVENT BASE")
print("=" * 92)

events_dir = cu.find_artefact(PROJECT_ROOT, "events")[0]
if events_dir is None:
    raise FileNotFoundError("events/ not found under the project root")

report["event_tables"] = {}
event_files = sorted(p for p in events_dir.rglob("*.parquet") if p.is_file())
for path in event_files:
    info = schema_of(path)
    if "error" in info:
        print(f"  {path.name:48s} !! {info['error']}")
        continue
    print()
    print(f"  {path.name}   rows={info['n_rows']:>10,}  "
          f"row_groups={info['n_row_groups']:>4d}  "
          f"cols={len(info['columns'])}")
    print(f"    {'column':34s} type")
    for col, typ in info["types"].items():
        print(f"    {col:34s} {typ}")
    report["event_tables"][str(path.relative_to(events_dir))] = info

# Verify the frozen row count and inspect the coordinate magnitudes once.
for name in ("canonical_event_base.parquet", "event_base.parquet"):
    candidate = events_dir / name
    if cu.path_ok(candidate):
        info = schema_of(candidate)
        if info.get("n_rows") == 7_265_849:
            print()
            print(f"  FROZEN ROW COUNT CONFIRMED: {name} = "
                  f"{info['n_rows']:,}")
            report["event_base_rows"] = info["n_rows"]
            probe_cols = [
                c
                for c in (
                    "event_centroid_x", "event_centroid_y",
                    "event_centroid_lon", "event_centroid_lat",
                    "event_id", "start_time", "end_time",
                )
                if c in info["columns"]
            ]
            head = pd.read_parquet(candidate, columns=probe_cols).head(2000)
            for col in probe_cols:
                s = pd.to_numeric(head[col], errors="coerce")
                if s.notna().any():
                    print(f"    {col:26s} min={s.min():>16.3f} "
                          f"max={s.max():>16.3f}")
            print()
            print("  SAMPLE ROWS")
            for row in head.head(3).to_dict("records"):
                print("   ", {k: row[k] for k in list(row)[:6]})
            report["event_base_probe"] = {
                c: {
                    "min": float(pd.to_numeric(head[c], errors="coerce").min()),
                    "max": float(pd.to_numeric(head[c], errors="coerce").max()),
                }
                for c in probe_cols
                if pd.to_numeric(head[c], errors="coerce").notna().any()
            }
        break

# Membership: the event -> local cluster bridge.
print()
print("membership table:")
for path in sorted(events_dir.glob("*membership*.parquet")):
    info = schema_of(path)
    print(f"  {path.name}: rows={info.get('n_rows', 0):,} "
          f"cols={info.get('columns')}")
    report["membership"] = {"file": path.name, **info}
    head = pd.read_parquet(path).head(2000)
    for col in head.columns:
        print(f"    {col:32s} sample={list(head[col].unique()[:3])}")

# ===========================================================================
# 3. CLUSTERS / EDGES / MAPPINGS -- schema of ONE file, dates from filenames
# ===========================================================================

print()
print("=" * 92)
print("3. CLUSTERS / EDGES / MAPPINGS")
print("=" * 92)

import lineage_features as lf

report["daily_sets"] = {}
for label, leaf, prefix in (
    ("clusters", "clusters", "clusters"),
    ("edges", "edges", "edges"),
    ("mappings", "mappings", None),
):
    path = cu.find_artefact(PROJECT_ROOT, leaf)[0]
    if path is None:
        print(f"  {label}: NOT FOUND")
        continue
    index, stats = lf.index_daily_files(path, prefix=prefix)
    dates = sorted(index)
    print()
    print(f"  {label}/  {len(index):,} dated files  {index[dates[0]].name if dates else ''}"
          f" .. {index[dates[-1]].name if dates else ''}")
    print(f"    stats: {stats}")
    entry = {"path": str(path), "n_dated_files": len(index),
             "first": dates[0] if dates else None,
             "last": dates[-1] if dates else None,
             "index_stats": stats}
    if dates:
        info = schema_of(index[dates[0]])
        print(f"    schema of {index[dates[0]].name}: "
              f"rows={info.get('n_rows', 0):,}")
        for col, typ in info.get("types", {}).items():
            print(f"      {col:30s} {typ}")
        entry["schema"] = info
    report["daily_sets"][label] = entry

# ===========================================================================
# 4. GAZETTEER -- one file per distinct schema, geometry + CRS + categories
# ===========================================================================

print()
print("=" * 92)
print("4. EXTERNAL / GAZETTEER")
print("=" * 92)

external = cu.find_artefact(PROJECT_ROOT, "external")[0]
if external is None:
    print("  external/ NOT FOUND")
else:
    report["external"] = {}
    ext_subfolders = {}
    for child in sorted(external.iterdir()):
        if child.is_dir():
            try:
                ext_subfolders[child.name] = sum(
                    1 for _ in child.rglob("*") if _.is_file()
                )
            except OSError:
                ext_subfolders[child.name] = -1

    # Subfolder census: names + counts, no reads.
    print("  external/ subfolders:")
    for name, count in sorted(ext_subfolders.items()):
        print(f"    {name:42s} {count:>6d} files")
    report["external"]["subfolders"] = ext_subfolders

    # Gazetteer: dedupe by name (same as P3a) then sample ONE per schema.
    fac_dir = external / "osm_facilities"
    if cu.is_dir(fac_dir):
        seen = {}
        for p in sorted(fac_dir.rglob("*.geoparquet")):
            penalty = 1_000 * sum(
                1
                for part in p.relative_to(fac_dir).parts[:-1]
                if re.search(r"\s*\(\d+\)$|\b_v\d+\b", part)
            )
            if p.name not in seen or penalty < seen[p.name][0]:
                seen[p.name] = (penalty, p)
        gaz = [v[1] for v in seen.values()]
        print()
        print(f"  osm_facilities: {len(gaz)} unique files "
              f"(duplicates collapsed)")

        by_sig = {}
        for p in gaz:
            info = schema_of(p)
            if "error" in info:
                continue
            by_sig.setdefault(signature(info), []).append((p, info))

        gaz_report = {}
        for i, (sig, group) in enumerate(sorted(by_sig.items(),
                                                key=lambda kv: -len(kv[1]))):
            if i >= SAMPLE_MAX_FILES:
                break
            sample_path, info = group[0]
            print()
            print(f"    [{len(group)} file(s) share this schema] "
                  f"{sample_path.name}")
            for col, typ in info["types"].items():
                print(f"      {col:30s} {typ}")
            entry = {
                "n_files_sharing_schema": len(group),
                "sample": str(sample_path.relative_to(fac_dir)),
                "columns": info["columns"],
                "types": info["types"],
                "rows": info["n_rows"],
                "files": [str(q.relative_to(fac_dir)) for q, _ in group][:10],
            }
            # One data sample: geometry types, CRS, category values.
            try:
                head = pd.read_parquet(sample_path).head(SAMPLE_ROWS * 200)
                geom_col = next(
                    (c for c in ("geometry", "geom", "wkb_geometry")
                     if c in head.columns),
                    None,
                )
                if geom_col:
                    import shapely
                    from shapely import from_wkb

                    geoms = from_wkb(
                        np.asarray(head[geom_col].to_numpy(zero_copy_only=False))
                    )
                    entry["geometry_types"] = dict(
                        Counter(g.geom_type for g in geoms if g is not None)
                    )
                    entry["crs"] = (
                        head.attrs.get("crs")
                        or str(getattr(sample_path, "crs", ""))[:60]
                    )
                    print(f"      geometry types: {entry['geometry_types']}")
                for cand in ("category", "facility_type", "type", "class"):
                    if cand in head.columns:
                        vals = head[cand].dropna().astype(str)
                        entry["category_column"] = cand
                        entry["category_values"] = sorted(vals.unique())[:25]
                        print(f"      {cand}: "
                              f"{entry['category_values'][:12]}")
                        break
            except Exception as exc:                 # noqa: BLE001
                entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
            gaz_report[str(sample_path.relative_to(fac_dir))] = entry

        report["external"]["gazetteer"] = gaz_report

# ===========================================================================
# 5. FEATURE INPUT AVAILABILITY -- the point of the whole exercise
# ===========================================================================

print()
print("=" * 92)
print("5. FEATURE INPUT AVAILABILITY (141 features)")
print("=" * 92)

import feature_ledger as fl

clusters_cols = set()
if "clusters" in report.get("daily_sets", {}):
    clusters_cols = set(
        report["daily_sets"]["clusters"].get("schema", {}).get("columns", [])
    )
event_cols = set()
for info in report.get("event_tables", {}).values():
    event_cols |= set(info.get("columns", []))

# Which lineage inputs are genuinely present in the detection columns?
wanted_detection_cols = {
    "frp": ("frp", "FRP"),
    "i4": ("i4", "brightness_i4"),
    "i5": ("i5", "brightness_i5"),
    "longitude": ("longitude", "lon"),
    "latitude": ("latitude", "lat"),
    "subpixel_temp": ("subpixel_temp",),
    "source_area": ("source_area",),
    "confidence": ("confidence",),
    "pixel_size": ("pixel_size",),
    "scan_angle": ("scan_angle",),
    "view_zenith": ("view_zenith"),
    "day_night": ("day_night",),
    "satellite": ("satellite", "sat"),
}
present, absent = [], []
for logical, cands in wanted_detection_cols.items():
    (present if clusters_cols & set(cands) else absent).append(logical)

print()
print(f"  detection columns found: {len(clusters_cols)}")
print(f"    present: {present}")
print(f"    absent : {absent}")

gaz_keys = set(report.get("external", {}).get("gazetteer", {}))
print(f"  gazetteer schemas sampled: {len(gaz_keys)}")
print(f"  event table columns: {len(event_cols)}")

# Group-level availability, ledger-driven.
availability = {}
for group in fl.FEATURE_GROUPS:
    names = [
        n for n in fl.FEATURE_GROUPS[group] if n not in fl._PLACEHOLDERS
    ]
    if group == fl.GROUP_INDUSTRIAL:
        state = "ready" if gaz_keys else "needs gazetteer"
    elif group in (fl.GROUP_THERMAL, fl.GROUP_TEMPORAL, fl.GROUP_SPREAD,
                   fl.GROUP_QUALITY):
        state = "ready" if present else f"missing {absent}"
    elif group == fl.GROUP_TERRAIN:
        state = "cell 263 (GEE)"
    else:
        state = "needs GEE / external"
    availability[group] = {"n": len(names), "state": state}
    print(f"  {group:24s} {len(names):>3d}  {state}")

report["availability"] = availability
report["detection_columns_present"] = present
report["detection_columns_absent"] = absent
report["cluster_columns"] = sorted(clusters_cols)
report["event_columns"] = sorted(event_cols)

# ===========================================================================
# 6. WRITE THE REPORT
# ===========================================================================

print()
print("=" * 92)
print(f"elapsed: {time.time() - t_start:.1f}s")
print("=" * 92)

qc_dir = PROJECT_ROOT / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out = qc_dir / "project_recon.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"Wrote {out}  ({out.stat().st_size / 1024:.1f} KB)")

print()
print("This is the reference for every path and column name from here on.")
print("If a cell hardcodes something this report contradicts, the report wins.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# COLAB CELL: project reconnaissance -- what actually exists, and cheaply
#
# Purpose
# -------
# One consolidated report of every input the 141 features need: real folder
# names, real column names, real dtypes, real geometry types. Stops all the
# guessing that has been producing wrong paths and wrong column names.
#
# Cost control -- this does NOT scan the data
# -------------------------------------------
#   * folder listings      : name-only, no file reads
#   * parquet schemas      : ParquetFile(...).schema_arrow -- FOOTER METADATA
#                            ONLY, no row group is decoded
#   * data samples         : exactly ONE file per *distinct schema signature*,
#                            read with a single row group
#   * cluster files        : schema only; dates come from the 5,337 filenames
#                            rather than from opening them
# So the whole report costs a few thousand file-header reads, not 15M rows.
#
# Output: a printed report plus qc/project_recon.json
# ============================================================================

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu

RECON_VERSION = "1.0"
SAMPLE_ROWS = 5
SAMPLE_MAX_FILES = 40          # cap on distinct schemas we open data for

print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
report = {
    "recon_version": RECON_VERSION,
    "project_root": str(PROJECT_ROOT),
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
}
t_start = time.time()


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} TB"


def schema_of(path):
    """Column names + types from the parquet footer. No data is decoded."""
    try:
        pf = pq.ParquetFile(path)
        # pf.schema_arrow IS a pyarrow.Schema and is itself iterable over
        # fields. `pf.schema_arrow.schema` does not exist -- using it made every
        # schema lookup fail, which silently zeroed out the whole availability
        # verdict in the first run of this script.
        arrow_schema = pf.schema_arrow
        return {
            "columns": list(arrow_schema.names),
            "types": {f.name: str(f.type) for f in arrow_schema},
            "n_rows": pf.metadata.num_rows,
            "n_row_groups": pf.metadata.num_row_groups,
        }
    except Exception as exc:                        # noqa: BLE001
        return {"error": f"{exc.__class__.__name__}: {exc}"}


def signature(info):
    """Stable identity for 'files that look alike', used to sample once."""
    return tuple(sorted(info.get("columns", [])))


# ===========================================================================
# 1. FOLDER INVENTORY -- names only
# ===========================================================================

print()
print("=" * 92)
print("1. FOLDER INVENTORY (name listing only, no file reads)")
print("=" * 92)

folders = [
    ("events", cu.find_artefact(PROJECT_ROOT, "events")[0]),
    ("clusters", cu.find_artefact(PROJECT_ROOT, "clusters")[0]),
    ("edges", cu.find_artefact(PROJECT_ROOT, "edges")[0]),
    ("mappings", cu.find_artefact(PROJECT_ROOT, "mappings")[0]),
    ("external", cu.find_artefact(PROJECT_ROOT, "external")[0]),
    ("feature_audit", cu.find_artefact(PROJECT_ROOT, "feature_audit")[0]),
    ("qc", cu.path_ok(PROJECT_ROOT / "qc") and PROJECT_ROOT / "qc" or None),
    ("qc_canon", cu.path_ok(_CANON / "qc") and _CANON / "qc" or None),
]
report["folders"] = {}
print(f"{'label':16s} {'files':>7s} {'parquet':>8s} {'csv':>6s} "
      f"{'json':>6s} {'other':>6s}  path")
for label, path in folders:
    if path is None:
        print(f"{label:16s} {'-':>7s}  (not found)")
        report["folders"][label] = None
        continue
    try:
        names = [p.name for p in path.iterdir()]
    except OSError as exc:
        print(f"{label:16s}  !! {exc}")
        continue
    ext = Counter(Path(n).suffix.lower() for n in names)
    dirs = [n for n in names if (path / n).is_dir()]
    files = [n for n in names if not (path / n).is_dir()]
    fext = Counter(Path(n).suffix.lower() for n in files)
    other = len(files) - fext.get(".parquet", 0) - fext.get(".csv", 0) - fext.get(
        ".json", 0
    )
    print(f"{label:16s} {len(files):>7d} {fext.get('.parquet',0):>8d} "
          f"{fext.get('.csv',0):>6d} {fext.get('.json',0):>6d} {other:>6d}  "
          f"{path.name}/")
    if dirs:
        print(f"{'':16s} {'':7s} subdirs: {sorted(dirs)[:8]}")
    report["folders"][label] = {
        "path": str(path),
        "n_files": len(files),
        "n_dirs": len(dirs),
        "extensions": dict(fext),
        "subdirs": sorted(dirs)[:20],
        "file_samples": sorted(files)[:8],
    }

# ===========================================================================
# 2. EVENT BASE -- the frozen 7,265,849-row table
# ===========================================================================

print()
print("=" * 92)
print("2. EVENT BASE")
print("=" * 92)

events_dir = cu.find_artefact(PROJECT_ROOT, "events")[0]
if events_dir is None:
    raise FileNotFoundError("events/ not found under the project root")

report["event_tables"] = {}
schema_errors = []
event_files = sorted(p for p in events_dir.rglob("*.parquet") if p.is_file())
for path in event_files:
    info = schema_of(path)
    if "error" in info:
        schema_errors.append((path, info["error"]))
        print(f"  {str(path.relative_to(events_dir)):46s} !! {info['error']}")
        continue
    print()
    print(f"  {path.relative_to(events_dir)}   rows={info['n_rows']:>10,}  "
          f"row_groups={info['n_row_groups']:>4d}  cols={len(info['columns'])}")
    for col, typ in info["types"].items():
        print(f"    {col:34s} {typ}")
    report["event_tables"][str(path.relative_to(events_dir))] = info

# A schema probe that fails is an ERROR, not an empty schema. The first run of
# this script reported "0 detection columns" purely because of that bug, which
# would have read as "the inputs are missing" and sent us re-downloading data
# that was present all along. Never let it be silent again.
if schema_errors:
    print()
    print("=" * 92)
    print(f"SCHEMA PROBE FAILED FOR {len(schema_errors)} FILE(S)")
    print("=" * 92)
    for path, err in schema_errors[:10]:
        print(f"  {path.name}: {err}")
    print()
    print("The availability verdict below would be MEANINGLESS. Fix the probe")
    print("before reading anything into it.")
    raise RuntimeError(
        f"{len(schema_errors)} schema probe(s) failed -- availability verdict "
        "is not trustworthy"
    )

# Verify the frozen row count and inspect the coordinate magnitudes once.
for name in ("canonical_event_base.parquet", "event_base.parquet"):
    candidate = events_dir / name
    if cu.path_ok(candidate):
        info = schema_of(candidate)
        if info.get("n_rows") == 7_265_849:
            print()
            print(f"  FROZEN ROW COUNT CONFIRMED: {name} = "
                  f"{info['n_rows']:,}")
            report["event_base_rows"] = info["n_rows"]
            probe_cols = [
                c
                for c in (
                    "event_centroid_x", "event_centroid_y",
                    "event_centroid_lon", "event_centroid_lat",
                    "event_id", "start_time", "end_time",
                )
                if c in info["columns"]
            ]
            head = pd.read_parquet(candidate, columns=probe_cols).head(2000)
            for col in probe_cols:
                s = pd.to_numeric(head[col], errors="coerce")
                if s.notna().any():
                    print(f"    {col:26s} min={s.min():>16.3f} "
                          f"max={s.max():>16.3f}")
            print()
            print("  SAMPLE ROWS")
            for row in head.head(3).to_dict("records"):
                print("   ", {k: row[k] for k in list(row)[:6]})
            report["event_base_probe"] = {
                c: {
                    "min": float(pd.to_numeric(head[c], errors="coerce").min()),
                    "max": float(pd.to_numeric(head[c], errors="coerce").max()),
                }
                for c in probe_cols
                if pd.to_numeric(head[c], errors="coerce").notna().any()
            }
        break

# Membership: the event -> local cluster bridge.
print()
print("membership table:")
for path in sorted(events_dir.glob("*membership*.parquet")):
    info = schema_of(path)
    print(f"  {path.name}: rows={info.get('n_rows', 0):,} "
          f"cols={info.get('columns')}")
    report["membership"] = {"file": path.name, **info}
    head = pd.read_parquet(path).head(2000)
    for col in head.columns:
        print(f"    {col:32s} sample={list(head[col].unique()[:3])}")

# ===========================================================================
# 3. CLUSTERS / EDGES / MAPPINGS -- schema of ONE file, dates from filenames
# ===========================================================================

print()
print("=" * 92)
print("3. CLUSTERS / EDGES / MAPPINGS")
print("=" * 92)

import lineage_features as lf

report["daily_sets"] = {}
for label, leaf, prefix in (
    ("clusters", "clusters", "clusters"),
    ("edges", "edges", "edges"),
    ("mappings", "mappings", None),
):
    path = cu.find_artefact(PROJECT_ROOT, leaf)[0]
    if path is None:
        print(f"  {label}: NOT FOUND")
        continue
    index, stats = lf.index_daily_files(path, prefix=prefix)
    dates = sorted(index)
    print()
    print(f"  {label}/  {len(index):,} dated files  {index[dates[0]].name if dates else ''}"
          f" .. {index[dates[-1]].name if dates else ''}")
    print(f"    stats: {stats}")
    entry = {"path": str(path), "n_dated_files": len(index),
             "first": dates[0] if dates else None,
             "last": dates[-1] if dates else None,
             "index_stats": stats}
    if dates:
        info = schema_of(index[dates[0]])
        print(f"    schema of {index[dates[0]].name}: "
              f"rows={info.get('n_rows', 0):,}")
        for col, typ in info.get("types", {}).items():
            print(f"      {col:30s} {typ}")
        entry["schema"] = info
    report["daily_sets"][label] = entry

# ===========================================================================
# 4. GAZETTEER -- one file per distinct schema, geometry + CRS + categories
# ===========================================================================

print()
print("=" * 92)
print("4. EXTERNAL / GAZETTEER")
print("=" * 92)

external = cu.find_artefact(PROJECT_ROOT, "external")[0]
if external is None:
    print("  external/ NOT FOUND")
else:
    report["external"] = {}
    ext_subfolders = {}
    for child in sorted(external.iterdir()):
        if child.is_dir():
            try:
                ext_subfolders[child.name] = sum(
                    1 for _ in child.rglob("*") if _.is_file()
                )
            except OSError:
                ext_subfolders[child.name] = -1

    # Subfolder census: names + counts, no reads.
    print("  external/ subfolders:")
    for name, count in sorted(ext_subfolders.items()):
        print(f"    {name:42s} {count:>6d} files")
    report["external"]["subfolders"] = ext_subfolders

    # Gazetteer: dedupe by name (same as P3a) then sample ONE per schema.
    fac_dir = external / "osm_facilities"
    if cu.is_dir(fac_dir):
        seen = {}
        for p in sorted(fac_dir.rglob("*.geoparquet")):
            penalty = 1_000 * sum(
                1
                for part in p.relative_to(fac_dir).parts[:-1]
                if re.search(r"\s*\(\d+\)$|\b_v\d+\b", part)
            )
            if p.name not in seen or penalty < seen[p.name][0]:
                seen[p.name] = (penalty, p)
        gaz = [v[1] for v in seen.values()]
        print()
        print(f"  osm_facilities: {len(gaz)} unique files "
              f"(duplicates collapsed)")

        by_sig = {}
        for p in gaz:
            info = schema_of(p)
            if "error" in info:
                continue
            by_sig.setdefault(signature(info), []).append((p, info))

        gaz_report = {}
        for i, (sig, group) in enumerate(sorted(by_sig.items(),
                                                key=lambda kv: -len(kv[1]))):
            if i >= SAMPLE_MAX_FILES:
                break
            sample_path, info = group[0]
            print()
            print(f"    [{len(group)} file(s) share this schema] "
                  f"{sample_path.name}")
            for col, typ in info["types"].items():
                print(f"      {col:30s} {typ}")
            entry = {
                "n_files_sharing_schema": len(group),
                "sample": str(sample_path.relative_to(fac_dir)),
                "columns": info["columns"],
                "types": info["types"],
                "rows": info["n_rows"],
                "files": [str(q.relative_to(fac_dir)) for q, _ in group][:10],
            }
            # One data sample: geometry types, CRS, category values.
            try:
                head = pd.read_parquet(sample_path).head(SAMPLE_ROWS * 200)
                geom_col = next(
                    (c for c in ("geometry", "geom", "wkb_geometry")
                     if c in head.columns),
                    None,
                )
                if geom_col:
                    import shapely
                    from shapely import from_wkb

                    geoms = from_wkb(
                        np.asarray(head[geom_col].to_numpy(zero_copy_only=False))
                    )
                    entry["geometry_types"] = dict(
                        Counter(g.geom_type for g in geoms if g is not None)
                    )
                    entry["crs"] = (
                        head.attrs.get("crs")
                        or str(getattr(sample_path, "crs", ""))[:60]
                    )
                    print(f"      geometry types: {entry['geometry_types']}")
                for cand in ("category", "facility_type", "type", "class"):
                    if cand in head.columns:
                        vals = head[cand].dropna().astype(str)
                        entry["category_column"] = cand
                        entry["category_values"] = sorted(vals.unique())[:25]
                        print(f"      {cand}: "
                              f"{entry['category_values'][:12]}")
                        break
            except Exception as exc:                 # noqa: BLE001
                entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
            gaz_report[str(sample_path.relative_to(fac_dir))] = entry

        report["external"]["gazetteer"] = gaz_report

# ===========================================================================
# 5. FEATURE INPUT AVAILABILITY -- the point of the whole exercise
# ===========================================================================

print()
print("=" * 92)
print("5. FEATURE INPUT AVAILABILITY (141 features)")
print("=" * 92)

import feature_ledger as fl

clusters_cols = set()
if "clusters" in report.get("daily_sets", {}):
    clusters_cols = set(
        report["daily_sets"]["clusters"].get("schema", {}).get("columns", [])
    )
event_cols = set()
for info in report.get("event_tables", {}).values():
    event_cols |= set(info.get("columns", []))

# Which lineage inputs are genuinely present in the detection columns?
wanted_detection_cols = {
    "frp": ("frp", "FRP"),
    "i4": ("i4", "brightness_i4"),
    "i5": ("i5", "brightness_i5"),
    "longitude": ("longitude", "lon"),
    "latitude": ("latitude", "lat"),
    "subpixel_temp": ("subpixel_temp",),
    "source_area": ("source_area",),
    "confidence": ("confidence",),
    "pixel_size": ("pixel_size",),
    "scan_angle": ("scan_angle",),
    "view_zenith": ("view_zenith"),
    "day_night": ("day_night",),
    "satellite": ("satellite", "sat"),
}
present, absent = [], []
for logical, cands in wanted_detection_cols.items():
    (present if clusters_cols & set(cands) else absent).append(logical)

print()
print(f"  detection columns found: {len(clusters_cols)}")
print(f"    present: {present}")
print(f"    absent : {absent}")

gaz_keys = set(report.get("external", {}).get("gazetteer", {}))
print(f"  gazetteer schemas sampled: {len(gaz_keys)}")
print(f"  event table columns: {len(event_cols)}")

# Group-level availability, ledger-driven.
availability = {}
for group in fl.FEATURE_GROUPS:
    names = [
        n for n in fl.FEATURE_GROUPS[group] if n not in fl._PLACEHOLDERS
    ]
    if group == fl.GROUP_INDUSTRIAL:
        state = "ready" if gaz_keys else "needs gazetteer"
    elif group in (fl.GROUP_THERMAL, fl.GROUP_TEMPORAL, fl.GROUP_SPREAD,
                   fl.GROUP_QUALITY):
        state = "ready" if present else f"missing {absent}"
    elif group == fl.GROUP_TERRAIN:
        state = "cell 263 (GEE)"
    else:
        state = "needs GEE / external"
    availability[group] = {"n": len(names), "state": state}
    print(f"  {group:24s} {len(names):>3d}  {state}")

report["availability"] = availability
report["detection_columns_present"] = present
report["detection_columns_absent"] = absent
report["cluster_columns"] = sorted(clusters_cols)
report["event_columns"] = sorted(event_cols)

# ===========================================================================
# 6. WRITE THE REPORT
# ===========================================================================

print()
print("=" * 92)
print(f"elapsed: {time.time() - t_start:.1f}s")
print("=" * 92)

qc_dir = PROJECT_ROOT / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out = qc_dir / "project_recon.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"Wrote {out}  ({out.stat().st_size / 1024:.1f} KB)")

print()
print("This is the reference for every path and column name from here on.")
print("If a cell hardcodes something this report contradicts, the report wins.")

In [ ]:
# ============================================================================
# COLAB CELL: project reconnaissance -- what actually exists, and cheaply
#
# Purpose
# -------
# One consolidated report of every input the 141 features need: real folder
# names, real column names, real dtypes, real geometry types. Stops all the
# guessing that has been producing wrong paths and wrong column names.
#
# Cost control -- this does NOT scan the data
# -------------------------------------------
#   * folder listings      : name-only, no file reads
#   * parquet schemas      : ParquetFile(...).schema_arrow -- FOOTER METADATA
#                            ONLY, no row group is decoded
#   * data samples         : exactly ONE file per *distinct schema signature*,
#                            read with a single row group
#   * cluster files        : schema only; dates come from the 5,337 filenames
#                            rather than from opening them
# So the whole report costs a few thousand file-header reads, not 15M rows.
#
# Output: a printed report plus qc/project_recon.json
# ============================================================================

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu

RECON_VERSION = "1.0"
SAMPLE_ROWS = 5
SAMPLE_MAX_FILES = 40          # cap on distinct schemas we open data for

print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
report = {
    "recon_version": RECON_VERSION,
    "project_root": str(PROJECT_ROOT),
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
}
t_start = time.time()


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} TB"


def schema_of(path):
    """Column names + types from the parquet footer. No data is decoded."""
    try:
        pf = pq.ParquetFile(path)
        # pf.schema_arrow IS a pyarrow.Schema and is itself iterable over
        # fields. `pf.schema_arrow.schema` does not exist -- using it made every
        # schema lookup fail, which silently zeroed out the whole availability
        # verdict in the first run of this script.
        arrow_schema = pf.schema_arrow
        return {
            "columns": list(arrow_schema.names),
            "types": {f.name: str(f.type) for f in arrow_schema},
            "n_rows": pf.metadata.num_rows,
            "n_row_groups": pf.metadata.num_row_groups,
        }
    except Exception as exc:                        # noqa: BLE001
        return {"error": f"{exc.__class__.__name__}: {exc}"}


def signature(info):
    """Stable identity for 'files that look alike', used to sample once."""
    return tuple(sorted(info.get("columns", [])))


# ===========================================================================
# 1. FOLDER INVENTORY -- names only
# ===========================================================================

print()
print("=" * 92)
print("1. FOLDER INVENTORY (name listing only, no file reads)")
print("=" * 92)

folders = [
    ("events", cu.find_artefact(PROJECT_ROOT, "events")[0]),
    ("clusters", cu.find_artefact(PROJECT_ROOT, "clusters")[0]),
    ("edges", cu.find_artefact(PROJECT_ROOT, "edges")[0]),
    ("mappings", cu.find_artefact(PROJECT_ROOT, "mappings")[0]),
    ("external", cu.find_artefact(PROJECT_ROOT, "external")[0]),
    ("feature_audit", cu.find_artefact(PROJECT_ROOT, "feature_audit")[0]),
    ("qc", cu.path_ok(PROJECT_ROOT / "qc") and PROJECT_ROOT / "qc" or None),
    ("qc_canon", cu.path_ok(_CANON / "qc") and _CANON / "qc" or None),
]
report["folders"] = {}
print(f"{'label':16s} {'files':>7s} {'parquet':>8s} {'csv':>6s} "
      f"{'json':>6s} {'other':>6s}  path")
for label, path in folders:
    if path is None:
        print(f"{label:16s} {'-':>7s}  (not found)")
        report["folders"][label] = None
        continue
    try:
        names = [p.name for p in path.iterdir()]
    except OSError as exc:
        print(f"{label:16s}  !! {exc}")
        continue
    ext = Counter(Path(n).suffix.lower() for n in names)
    dirs = [n for n in names if (path / n).is_dir()]
    files = [n for n in names if not (path / n).is_dir()]
    fext = Counter(Path(n).suffix.lower() for n in files)
    other = len(files) - fext.get(".parquet", 0) - fext.get(".csv", 0) - fext.get(
        ".json", 0
    )
    print(f"{label:16s} {len(files):>7d} {fext.get('.parquet',0):>8d} "
          f"{fext.get('.csv',0):>6d} {fext.get('.json',0):>6d} {other:>6d}  "
          f"{path.name}/")
    if dirs:
        print(f"{'':16s} {'':7s} subdirs: {sorted(dirs)[:8]}")
    report["folders"][label] = {
        "path": str(path),
        "n_files": len(files),
        "n_dirs": len(dirs),
        "extensions": dict(fext),
        "subdirs": sorted(dirs)[:20],
        "file_samples": sorted(files)[:8],
    }

# ===========================================================================
# 2. EVENT BASE -- the frozen 7,265,849-row table
# ===========================================================================

print()
print("=" * 92)
print("2. EVENT BASE")
print("=" * 92)

events_dir = cu.find_artefact(PROJECT_ROOT, "events")[0]
if events_dir is None:
    raise FileNotFoundError("events/ not found under the project root")

report["event_tables"] = {}
schema_errors = []
event_files = sorted(p for p in events_dir.rglob("*.parquet") if p.is_file())
for path in event_files:
    info = schema_of(path)
    if "error" in info:
        schema_errors.append((path, info["error"]))
        print(f"  {str(path.relative_to(events_dir)):46s} !! {info['error']}")
        continue
    print()
    print(f"  {path.relative_to(events_dir)}   rows={info['n_rows']:>10,}  "
          f"row_groups={info['n_row_groups']:>4d}  cols={len(info['columns'])}")
    for col, typ in info["types"].items():
        print(f"    {col:34s} {typ}")
    report["event_tables"][str(path.relative_to(events_dir))] = info

# A schema probe that fails is an ERROR, not an empty schema. The first run of
# this script reported "0 detection columns" purely because of that bug, which
# would have read as "the inputs are missing" and sent us re-downloading data
# that was present all along. Never let it be silent again.
if schema_errors:
    print()
    print("=" * 92)
    print(f"SCHEMA PROBE FAILED FOR {len(schema_errors)} FILE(S)")
    print("=" * 92)
    for path, err in schema_errors[:10]:
        print(f"  {path.name}: {err}")
    print()
    print("The availability verdict below would be MEANINGLESS. Fix the probe")
    print("before reading anything into it.")
    raise RuntimeError(
        f"{len(schema_errors)} schema probe(s) failed -- availability verdict "
        "is not trustworthy"
    )

# Verify the frozen row count and inspect the coordinate magnitudes once.
for name in ("canonical_event_base.parquet", "event_base.parquet"):
    candidate = events_dir / name
    if cu.path_ok(candidate):
        info = schema_of(candidate)
        if info.get("n_rows") == 7_265_849:
            print()
            print(f"  FROZEN ROW COUNT CONFIRMED: {name} = "
                  f"{info['n_rows']:,}")
            report["event_base_rows"] = info["n_rows"]
            probe_cols = [
                c
                for c in (
                    "event_centroid_x", "event_centroid_y",
                    "event_centroid_lon", "event_centroid_lat",
                    "event_id", "start_time", "end_time",
                )
                if c in info["columns"]
            ]
            head = pd.read_parquet(candidate, columns=probe_cols).head(2000)
            for col in probe_cols:
                s = pd.to_numeric(head[col], errors="coerce")
                if s.notna().any():
                    print(f"    {col:26s} min={s.min():>16.3f} "
                          f"max={s.max():>16.3f}")
            print()
            print("  SAMPLE ROWS")
            for row in head.head(3).to_dict("records"):
                print("   ", {k: row[k] for k in list(row)[:6]})
            report["event_base_probe"] = {
                c: {
                    "min": float(pd.to_numeric(head[c], errors="coerce").min()),
                    "max": float(pd.to_numeric(head[c], errors="coerce").max()),
                }
                for c in probe_cols
                if pd.to_numeric(head[c], errors="coerce").notna().any()
            }
        break

# Membership: the event -> local cluster bridge.
print()
print("membership table:")
for path in sorted(events_dir.glob("*membership*.parquet")):
    info = schema_of(path)
    print(f"  {path.name}: rows={info.get('n_rows', 0):,} "
          f"cols={info.get('columns')}")
    report["membership"] = {"file": path.name, **info}
    head = pd.read_parquet(path).head(2000)
    for col in head.columns:
        print(f"    {col:32s} sample={list(head[col].unique()[:3])}")

# ===========================================================================
# 3. CLUSTERS / EDGES / MAPPINGS -- schema of ONE file, dates from filenames
# ===========================================================================

print()
print("=" * 92)
print("3. CLUSTERS / EDGES / MAPPINGS")
print("=" * 92)

import lineage_features as lf

report["daily_sets"] = {}
for label, leaf, prefix in (
    ("clusters", "clusters", "clusters"),
    ("edges", "edges", "edges"),
    ("mappings", "mappings", None),
):
    path = cu.find_artefact(PROJECT_ROOT, leaf)[0]
    if path is None:
        print(f"  {label}: NOT FOUND")
        continue
    index, stats = lf.index_daily_files(path, prefix=prefix)
    dates = sorted(index)
    print()
    print(f"  {label}/  {len(index):,} dated files  {index[dates[0]].name if dates else ''}"
          f" .. {index[dates[-1]].name if dates else ''}")
    print(f"    stats: {stats}")
    entry = {"path": str(path), "n_dated_files": len(index),
             "first": dates[0] if dates else None,
             "last": dates[-1] if dates else None,
             "index_stats": stats}
    if dates:
        info = schema_of(index[dates[0]])
        print(f"    schema of {index[dates[0]].name}: "
              f"rows={info.get('n_rows', 0):,}")
        for col, typ in info.get("types", {}).items():
            print(f"      {col:30s} {typ}")
        entry["schema"] = info
    report["daily_sets"][label] = entry

# ===========================================================================
# 4. GAZETTEER -- one file per distinct schema, geometry + CRS + categories
# ===========================================================================

print()
print("=" * 92)
print("4. EXTERNAL / GAZETTEER")
print("=" * 92)

external = cu.find_artefact(PROJECT_ROOT, "external")[0]
if external is None:
    print("  external/ NOT FOUND")
else:
    report["external"] = {}
    ext_subfolders = {}
    for child in sorted(external.iterdir()):
        if child.is_dir():
            try:
                ext_subfolders[child.name] = sum(
                    1 for _ in child.rglob("*") if _.is_file()
                )
            except OSError:
                ext_subfolders[child.name] = -1

    # Subfolder census: names + counts, no reads.
    print("  external/ subfolders:")
    for name, count in sorted(ext_subfolders.items()):
        print(f"    {name:42s} {count:>6d} files")
    report["external"]["subfolders"] = ext_subfolders

    # Gazetteer: dedupe by name (same as P3a) then sample ONE per schema.
    fac_dir = external / "osm_facilities"
    if cu.is_dir(fac_dir):
        seen = {}
        for p in sorted(fac_dir.rglob("*.geoparquet")):
            penalty = 1_000 * sum(
                1
                for part in p.relative_to(fac_dir).parts[:-1]
                if re.search(r"\s*\(\d+\)$|\b_v\d+\b", part)
            )
            if p.name not in seen or penalty < seen[p.name][0]:
                seen[p.name] = (penalty, p)
        gaz = [v[1] for v in seen.values()]
        print()
        print(f"  osm_facilities: {len(gaz)} unique files "
              f"(duplicates collapsed)")

        by_sig = {}
        for p in gaz:
            info = schema_of(p)
            if "error" in info:
                continue
            by_sig.setdefault(signature(info), []).append((p, info))

        gaz_report = {}
        for i, (sig, group) in enumerate(sorted(by_sig.items(),
                                                key=lambda kv: -len(kv[1]))):
            if i >= SAMPLE_MAX_FILES:
                break
            sample_path, info = group[0]
            print()
            print(f"    [{len(group)} file(s) share this schema] "
                  f"{sample_path.name}")
            for col, typ in info["types"].items():
                print(f"      {col:30s} {typ}")
            entry = {
                "n_files_sharing_schema": len(group),
                "sample": str(sample_path.relative_to(fac_dir)),
                "columns": info["columns"],
                "types": info["types"],
                "rows": info["n_rows"],
                "files": [str(q.relative_to(fac_dir)) for q, _ in group][:10],
            }
            # One data sample: geometry types, CRS, category values.
            try:
                head = pd.read_parquet(sample_path).head(SAMPLE_ROWS * 200)
                geom_col = next(
                    (c for c in ("geometry", "geom", "wkb_geometry")
                     if c in head.columns),
                    None,
                )
                if geom_col:
                    import shapely
                    from shapely import from_wkb

                    geoms = from_wkb(
                        np.asarray(head[geom_col].to_numpy(zero_copy_only=False))
                    )
                    entry["geometry_types"] = dict(
                        Counter(g.geom_type for g in geoms if g is not None)
                    )
                    entry["crs"] = (
                        head.attrs.get("crs")
                        or str(getattr(sample_path, "crs", ""))[:60]
                    )
                    print(f"      geometry types: {entry['geometry_types']}")
                for cand in ("category", "facility_type", "type", "class"):
                    if cand in head.columns:
                        vals = head[cand].dropna().astype(str)
                        entry["category_column"] = cand
                        entry["category_values"] = sorted(vals.unique())[:25]
                        print(f"      {cand}: "
                              f"{entry['category_values'][:12]}")
                        break
            except Exception as exc:                 # noqa: BLE001
                entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
            gaz_report[str(sample_path.relative_to(fac_dir))] = entry

        report["external"]["gazetteer"] = gaz_report

# ===========================================================================
# 5. FEATURE INPUT AVAILABILITY -- the point of the whole exercise
# ===========================================================================

print()
print("=" * 92)
print("5. FEATURE INPUT AVAILABILITY (141 features)")
print("=" * 92)

import feature_ledger as fl

clusters_cols = set()
if "clusters" in report.get("daily_sets", {}):
    clusters_cols = set(
        report["daily_sets"]["clusters"].get("schema", {}).get("columns", [])
    )
event_cols = set()
for info in report.get("event_tables", {}).values():
    event_cols |= set(info.get("columns", []))

# Which lineage inputs are genuinely present in the detection columns?
wanted_detection_cols = {
    "frp": ("frp", "FRP"),
    "i4": ("i4", "brightness_i4"),
    "i5": ("i5", "brightness_i5"),
    "longitude": ("longitude", "lon"),
    "latitude": ("latitude", "lat"),
    "subpixel_temp": ("subpixel_temp",),
    "source_area": ("source_area",),
    "confidence": ("confidence",),
    "pixel_size": ("pixel_size",),
    "scan_angle": ("scan_angle",),
    "view_zenith": ("view_zenith"),
    "day_night": ("day_night",),
    "satellite": ("satellite", "sat"),
}
present, absent = [], []
for logical, cands in wanted_detection_cols.items():
    (present if clusters_cols & set(cands) else absent).append(logical)

print()
print(f"  detection columns found: {len(clusters_cols)}")
print(f"    present: {present}")
print(f"    absent : {absent}")

gaz_keys = set(report.get("external", {}).get("gazetteer", {}))
print(f"  gazetteer schemas sampled: {len(gaz_keys)}")
print(f"  event table columns: {len(event_cols)}")

# Group-level availability, ledger-driven.
availability = {}
for group in fl.FEATURE_GROUPS:
    names = [
        n for n in fl.FEATURE_GROUPS[group] if n not in fl._PLACEHOLDERS
    ]
    if group == fl.GROUP_INDUSTRIAL:
        state = "ready" if gaz_keys else "needs gazetteer"
    elif group in (fl.GROUP_THERMAL, fl.GROUP_TEMPORAL, fl.GROUP_SPREAD,
                   fl.GROUP_QUALITY):
        state = "ready" if present else f"missing {absent}"
    elif group == fl.GROUP_TERRAIN:
        state = "cell 263 (GEE)"
    else:
        state = "needs GEE / external"
    availability[group] = {"n": len(names), "state": state}
    print(f"  {group:24s} {len(names):>3d}  {state}")

report["availability"] = availability
report["detection_columns_present"] = present
report["detection_columns_absent"] = absent
report["cluster_columns"] = sorted(clusters_cols)
report["event_columns"] = sorted(event_cols)

# ===========================================================================
# 6. WRITE THE REPORT
# ===========================================================================

print()
print("=" * 92)
print(f"elapsed: {time.time() - t_start:.1f}s")
print("=" * 92)

qc_dir = PROJECT_ROOT / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out = qc_dir / "project_recon.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"Wrote {out}  ({out.stat().st_size / 1024:.1f} KB)")

print()
print("This is the reference for every path and column name from here on.")
print("If a cell hardcodes something this report contradicts, the report wins.")

In [ ]:
# ============================================================================
# COLAB CELL: project reconnaissance -- what actually exists, and cheaply
#
# Purpose
# -------
# One consolidated report of every input the 141 features need: real folder
# names, real column names, real dtypes, real geometry types. Stops all the
# guessing that has been producing wrong paths and wrong column names.
#
# Cost control -- this does NOT scan the data
# -------------------------------------------
#   * folder listings      : name-only, no file reads
#   * parquet schemas      : ParquetFile(...).schema_arrow -- FOOTER METADATA
#                            ONLY, no row group is decoded
#   * data samples         : exactly ONE file per *distinct schema signature*,
#                            read with a single row group
#   * cluster files        : schema only; dates come from the 5,337 filenames
#                            rather than from opening them
# So the whole report costs a few thousand file-header reads, not 15M rows.
#
# Output: a printed report plus qc/project_recon.json
# ============================================================================

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu

RECON_VERSION = "1.0"
SAMPLE_ROWS = 5
SAMPLE_MAX_FILES = 40          # cap on distinct schemas we open data for

print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
report = {
    "recon_version": RECON_VERSION,
    "project_root": str(PROJECT_ROOT),
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
}
t_start = time.time()


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} TB"


def schema_of(path):
    """Column names + types from the parquet footer. No data is decoded."""
    try:
        pf = pq.ParquetFile(path)
        # pf.schema_arrow IS a pyarrow.Schema and is itself iterable over
        # fields. `pf.schema_arrow.schema` does not exist -- using it made every
        # schema lookup fail, which silently zeroed out the whole availability
        # verdict in the first run of this script.
        arrow_schema = pf.schema_arrow
        return {
            "columns": list(arrow_schema.names),
            "types": {f.name: str(f.type) for f in arrow_schema},
            "n_rows": pf.metadata.num_rows,
            "n_row_groups": pf.metadata.num_row_groups,
        }
    except Exception as exc:                        # noqa: BLE001
        return {"error": f"{exc.__class__.__name__}: {exc}"}


def signature(info):
    """Stable identity for 'files that look alike', used to sample once."""
    return tuple(sorted(info.get("columns", [])))


# ===========================================================================
# 1. FOLDER INVENTORY -- names only
# ===========================================================================

print()
print("=" * 92)
print("1. FOLDER INVENTORY (name listing only, no file reads)")
print("=" * 92)

folders = [
    ("events", cu.find_artefact(PROJECT_ROOT, "events")[0]),
    ("clusters", cu.find_artefact(PROJECT_ROOT, "clusters")[0]),
    ("edges", cu.find_artefact(PROJECT_ROOT, "edges")[0]),
    ("mappings", cu.find_artefact(PROJECT_ROOT, "mappings")[0]),
    ("external", cu.find_artefact(PROJECT_ROOT, "external")[0]),
    ("feature_audit", cu.find_artefact(PROJECT_ROOT, "feature_audit")[0]),
    ("qc", cu.path_ok(PROJECT_ROOT / "qc") and PROJECT_ROOT / "qc" or None),
    ("qc_canon", cu.path_ok(_CANON / "qc") and _CANON / "qc" or None),
]
report["folders"] = {}
print(f"{'label':16s} {'files':>7s} {'parquet':>8s} {'csv':>6s} "
      f"{'json':>6s} {'other':>6s}  path")
for label, path in folders:
    if path is None:
        print(f"{label:16s} {'-':>7s}  (not found)")
        report["folders"][label] = None
        continue
    try:
        names = [p.name for p in path.iterdir()]
    except OSError as exc:
        print(f"{label:16s}  !! {exc}")
        continue
    ext = Counter(Path(n).suffix.lower() for n in names)
    dirs = [n for n in names if (path / n).is_dir()]
    files = [n for n in names if not (path / n).is_dir()]
    fext = Counter(Path(n).suffix.lower() for n in files)
    other = len(files) - fext.get(".parquet", 0) - fext.get(".csv", 0) - fext.get(
        ".json", 0
    )
    print(f"{label:16s} {len(files):>7d} {fext.get('.parquet',0):>8d} "
          f"{fext.get('.csv',0):>6d} {fext.get('.json',0):>6d} {other:>6d}  "
          f"{path.name}/")
    if dirs:
        print(f"{'':16s} {'':7s} subdirs: {sorted(dirs)[:8]}")
    report["folders"][label] = {
        "path": str(path),
        "n_files": len(files),
        "n_dirs": len(dirs),
        "extensions": dict(fext),
        "subdirs": sorted(dirs)[:20],
        "file_samples": sorted(files)[:8],
    }

# ===========================================================================
# 2. EVENT BASE -- the frozen 7,265,849-row table
# ===========================================================================

print()
print("=" * 92)
print("2. EVENT BASE")
print("=" * 92)

events_dir = cu.find_artefact(PROJECT_ROOT, "events")[0]
if events_dir is None:
    raise FileNotFoundError("events/ not found under the project root")

report["event_tables"] = {}
schema_errors = []
event_files = sorted(p for p in events_dir.rglob("*.parquet") if p.is_file())
for path in event_files:
    info = schema_of(path)
    if "error" in info:
        schema_errors.append((path, info["error"]))
        print(f"  {str(path.relative_to(events_dir)):46s} !! {info['error']}")
        continue
    print()
    print(f"  {path.relative_to(events_dir)}   rows={info['n_rows']:>10,}  "
          f"row_groups={info['n_row_groups']:>4d}  cols={len(info['columns'])}")
    for col, typ in info["types"].items():
        print(f"    {col:34s} {typ}")
    report["event_tables"][str(path.relative_to(events_dir))] = info

# A schema probe that fails is an ERROR, not an empty schema. The first run of
# this script reported "0 detection columns" purely because of that bug, which
# would have read as "the inputs are missing" and sent us re-downloading data
# that was present all along. Never let it be silent again.
if schema_errors:
    print()
    print("=" * 92)
    print(f"SCHEMA PROBE FAILED FOR {len(schema_errors)} FILE(S)")
    print("=" * 92)
    for path, err in schema_errors[:10]:
        print(f"  {path.name}: {err}")
    print()
    print("The availability verdict below would be MEANINGLESS. Fix the probe")
    print("before reading anything into it.")
    raise RuntimeError(
        f"{len(schema_errors)} schema probe(s) failed -- availability verdict "
        "is not trustworthy"
    )

# Verify the frozen row count and inspect the coordinate magnitudes once.
for name in ("canonical_event_base.parquet", "event_base.parquet"):
    candidate = events_dir / name
    if cu.path_ok(candidate):
        info = schema_of(candidate)
        if info.get("n_rows") == 7_265_849:
            print()
            print(f"  FROZEN ROW COUNT CONFIRMED: {name} = "
                  f"{info['n_rows']:,}")
            report["event_base_rows"] = info["n_rows"]
            probe_cols = [
                c
                for c in (
                    "event_centroid_x", "event_centroid_y",
                    "event_centroid_lon", "event_centroid_lat",
                    "event_id", "start_time", "end_time",
                )
                if c in info["columns"]
            ]
            head = pd.read_parquet(candidate, columns=probe_cols).head(2000)
            for col in probe_cols:
                s = pd.to_numeric(head[col], errors="coerce")
                if s.notna().any():
                    print(f"    {col:26s} min={s.min():>16.3f} "
                          f"max={s.max():>16.3f}")
            print()
            print("  SAMPLE ROWS")
            for row in head.head(3).to_dict("records"):
                print("   ", {k: row[k] for k in list(row)[:6]})
            report["event_base_probe"] = {
                c: {
                    "min": float(pd.to_numeric(head[c], errors="coerce").min()),
                    "max": float(pd.to_numeric(head[c], errors="coerce").max()),
                }
                for c in probe_cols
                if pd.to_numeric(head[c], errors="coerce").notna().any()
            }
        break

# Membership: the event -> local cluster bridge.
print()
print("membership table:")
for path in sorted(events_dir.glob("*membership*.parquet")):
    info = schema_of(path)
    print(f"  {path.name}: rows={info.get('n_rows', 0):,} "
          f"cols={info.get('columns')}")
    report["membership"] = {"file": path.name, **info}
    head = pd.read_parquet(path).head(2000)
    for col in head.columns:
        print(f"    {col:32s} sample={list(head[col].unique()[:3])}")

# ===========================================================================
# 3. CLUSTERS / EDGES / MAPPINGS -- schema of ONE file, dates from filenames
# ===========================================================================

print()
print("=" * 92)
print("3. CLUSTERS / EDGES / MAPPINGS")
print("=" * 92)

import lineage_features as lf

report["daily_sets"] = {}
for label, leaf, prefix in (
    ("clusters", "clusters", "clusters"),
    ("edges", "edges", "edges"),
    ("mappings", "mappings", None),
):
    path = cu.find_artefact(PROJECT_ROOT, leaf)[0]
    if path is None:
        print(f"  {label}: NOT FOUND")
        continue
    index, stats = lf.index_daily_files(path, prefix=prefix)
    dates = sorted(index)
    print()
    print(f"  {label}/  {len(index):,} dated files  {index[dates[0]].name if dates else ''}"
          f" .. {index[dates[-1]].name if dates else ''}")
    print(f"    stats: {stats}")
    entry = {"path": str(path), "n_dated_files": len(index),
             "first": dates[0] if dates else None,
             "last": dates[-1] if dates else None,
             "index_stats": stats}
    if dates:
        info = schema_of(index[dates[0]])
        print(f"    schema of {index[dates[0]].name}: "
              f"rows={info.get('n_rows', 0):,}")
        for col, typ in info.get("types", {}).items():
            print(f"      {col:30s} {typ}")
        entry["schema"] = info
    report["daily_sets"][label] = entry

# ===========================================================================
# 4. GAZETTEER -- one file per distinct schema, geometry + CRS + categories
# ===========================================================================

print()
print("=" * 92)
print("4. EXTERNAL / GAZETTEER")
print("=" * 92)

external = cu.find_artefact(PROJECT_ROOT, "external")[0]
if external is None:
    print("  external/ NOT FOUND")
else:
    report["external"] = {}
    ext_subfolders = {}
    for child in sorted(external.iterdir()):
        if child.is_dir():
            try:
                ext_subfolders[child.name] = sum(
                    1 for _ in child.rglob("*") if _.is_file()
                )
            except OSError:
                ext_subfolders[child.name] = -1

    # Subfolder census: names + counts, no reads.
    print("  external/ subfolders:")
    for name, count in sorted(ext_subfolders.items()):
        print(f"    {name:42s} {count:>6d} files")
    report["external"]["subfolders"] = ext_subfolders

    # Gazetteer: dedupe by name (same as P3a) then sample ONE per schema.
    fac_dir = external / "osm_facilities"
    if cu.is_dir(fac_dir):
        seen = {}
        for p in sorted(fac_dir.rglob("*.geoparquet")):
            penalty = 1_000 * sum(
                1
                for part in p.relative_to(fac_dir).parts[:-1]
                if re.search(r"\s*\(\d+\)$|\b_v\d+\b", part)
            )
            if p.name not in seen or penalty < seen[p.name][0]:
                seen[p.name] = (penalty, p)
        gaz = [v[1] for v in seen.values()]
        print()
        print(f"  osm_facilities: {len(gaz)} unique files "
              f"(duplicates collapsed)")

        by_sig = {}
        for p in gaz:
            info = schema_of(p)
            if "error" in info:
                continue
            by_sig.setdefault(signature(info), []).append((p, info))

        gaz_report = {}
        for i, (sig, group) in enumerate(sorted(by_sig.items(),
                                                key=lambda kv: -len(kv[1]))):
            if i >= SAMPLE_MAX_FILES:
                break
            sample_path, info = group[0]
            print()
            print(f"    [{len(group)} file(s) share this schema] "
                  f"{sample_path.name}")
            for col, typ in info["types"].items():
                print(f"      {col:30s} {typ}")
            entry = {
                "n_files_sharing_schema": len(group),
                "sample": str(sample_path.relative_to(fac_dir)),
                "columns": info["columns"],
                "types": info["types"],
                "rows": info["n_rows"],
                "files": [str(q.relative_to(fac_dir)) for q, _ in group][:10],
            }
            # One data sample: geometry types, CRS, category values.
            try:
                head = pd.read_parquet(sample_path).head(SAMPLE_ROWS * 200)
                geom_col = next(
                    (c for c in ("geometry", "geom", "wkb_geometry")
                     if c in head.columns),
                    None,
                )
                if geom_col:
                    import shapely
                    from shapely import from_wkb

                    geoms = from_wkb(
                        np.asarray(head[geom_col].to_numpy(zero_copy_only=False))
                    )
                    entry["geometry_types"] = dict(
                        Counter(g.geom_type for g in geoms if g is not None)
                    )
                    entry["crs"] = (
                        head.attrs.get("crs")
                        or str(getattr(sample_path, "crs", ""))[:60]
                    )
                    print(f"      geometry types: {entry['geometry_types']}")
                for cand in ("category", "facility_type", "type", "class"):
                    if cand in head.columns:
                        vals = head[cand].dropna().astype(str)
                        entry["category_column"] = cand
                        entry["category_values"] = sorted(vals.unique())[:25]
                        print(f"      {cand}: "
                              f"{entry['category_values'][:12]}")
                        break
            except Exception as exc:                 # noqa: BLE001
                entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
            gaz_report[str(sample_path.relative_to(fac_dir))] = entry

        report["external"]["gazetteer"] = gaz_report

# ===========================================================================
# 5. FEATURE INPUT AVAILABILITY -- the point of the whole exercise
# ===========================================================================

print()
print("=" * 92)
print("5. FEATURE INPUT AVAILABILITY (141 features)")
print("=" * 92)

import feature_ledger as fl

clusters_cols = set()
if "clusters" in report.get("daily_sets", {}):
    clusters_cols = set(
        report["daily_sets"]["clusters"].get("schema", {}).get("columns", [])
    )
event_cols = set()
for info in report.get("event_tables", {}).values():
    event_cols |= set(info.get("columns", []))

# Which lineage inputs are genuinely present in the detection columns?
wanted_detection_cols = {
    "frp": ("frp", "FRP"),
    "i4": ("i4", "brightness_i4"),
    "i5": ("i5", "brightness_i5"),
    "longitude": ("longitude", "lon"),
    "latitude": ("latitude", "lat"),
    "subpixel_temp": ("subpixel_temp",),
    "source_area": ("source_area",),
    "confidence": ("confidence",),
    "pixel_size": ("pixel_size",),
    "scan_angle": ("scan_angle",),
    "view_zenith": ("view_zenith"),
    "day_night": ("day_night",),
    "satellite": ("satellite", "sat"),
}
present, absent = [], []
for logical, cands in wanted_detection_cols.items():
    (present if clusters_cols & set(cands) else absent).append(logical)

print()
print(f"  detection columns found: {len(clusters_cols)}")
print(f"    present: {present}")
print(f"    absent : {absent}")

gaz_keys = set(report.get("external", {}).get("gazetteer", {}))
print(f"  gazetteer schemas sampled: {len(gaz_keys)}")
print(f"  event table columns: {len(event_cols)}")

# Group-level availability, ledger-driven.
availability = {}
for group in fl.FEATURE_GROUPS:
    names = [
        n for n in fl.FEATURE_GROUPS[group] if n not in fl._PLACEHOLDERS
    ]
    if group == fl.GROUP_INDUSTRIAL:
        state = "ready" if gaz_keys else "needs gazetteer"
    elif group in (fl.GROUP_THERMAL, fl.GROUP_TEMPORAL, fl.GROUP_SPREAD,
                   fl.GROUP_QUALITY):
        state = "ready" if present else f"missing {absent}"
    elif group == fl.GROUP_TERRAIN:
        state = "cell 263 (GEE)"
    else:
        state = "needs GEE / external"
    availability[group] = {"n": len(names), "state": state}
    print(f"  {group:24s} {len(names):>3d}  {state}")

report["availability"] = availability
report["detection_columns_present"] = present
report["detection_columns_absent"] = absent
report["cluster_columns"] = sorted(clusters_cols)
report["event_columns"] = sorted(event_cols)

# ===========================================================================
# 6. WRITE THE REPORT
# ===========================================================================

print()
print("=" * 92)
print(f"elapsed: {time.time() - t_start:.1f}s")
print("=" * 92)

qc_dir = PROJECT_ROOT / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out = qc_dir / "project_recon.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"Wrote {out}  ({out.stat().st_size / 1024:.1f} KB)")

print()
print("This is the reference for every path and column name from here on.")
print("If a cell hardcodes something this report contradicts, the report wins.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# COLAB CELL: project reconnaissance -- what actually exists, and cheaply
#
# Purpose
# -------
# One consolidated report of every input the 141 features need: real folder
# names, real column names, real dtypes, real geometry types. Stops all the
# guessing that has been producing wrong paths and wrong column names.
#
# Cost control -- this does NOT scan the data
# -------------------------------------------
#   * folder listings      : name-only, no file reads
#   * parquet schemas      : ParquetFile(...).schema_arrow -- FOOTER METADATA
#                            ONLY, no row group is decoded
#   * data samples         : exactly ONE file per *distinct schema signature*,
#                            read with a single row group
#   * cluster files        : schema only; dates come from the 5,337 filenames
#                            rather than from opening them
# So the whole report costs a few thousand file-header reads, not 15M rows.
#
# Output: a printed report plus qc/project_recon.json
# ============================================================================

import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu

RECON_VERSION = "1.0"
SAMPLE_ROWS = 5
SAMPLE_MAX_FILES = 40          # cap on distinct schemas we open data for

print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
report = {
    "recon_version": RECON_VERSION,
    "project_root": str(PROJECT_ROOT),
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
}
t_start = time.time()


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} TB"


def schema_of(path):
    """Column names + types from the parquet footer. No data is decoded."""
    try:
        pf = pq.ParquetFile(path)
        # pf.schema_arrow IS a pyarrow.Schema and is itself iterable over
        # fields. `pf.schema_arrow.schema` does not exist -- using it made every
        # schema lookup fail, which silently zeroed out the whole availability
        # verdict in the first run of this script.
        arrow_schema = pf.schema_arrow
        return {
            "columns": list(arrow_schema.names),
            "types": {f.name: str(f.type) for f in arrow_schema},
            "n_rows": pf.metadata.num_rows,
            "n_row_groups": pf.metadata.num_row_groups,
        }
    except Exception as exc:                        # noqa: BLE001
        return {"error": f"{exc.__class__.__name__}: {exc}"}


def signature(info):
    """Stable identity for 'files that look alike', used to sample once."""
    return tuple(sorted(info.get("columns", [])))


# ===========================================================================
# 1. FOLDER INVENTORY -- names only
# ===========================================================================

print()
print("=" * 92)
print("1. FOLDER INVENTORY (name listing only, no file reads)")
print("=" * 92)

folders = [
    ("events", cu.find_artefact(PROJECT_ROOT, "events")[0]),
    ("clusters", cu.find_artefact(PROJECT_ROOT, "clusters")[0]),
    ("edges", cu.find_artefact(PROJECT_ROOT, "edges")[0]),
    ("mappings", cu.find_artefact(PROJECT_ROOT, "mappings")[0]),
    ("external", cu.find_artefact(PROJECT_ROOT, "external")[0]),
    ("feature_audit", cu.find_artefact(PROJECT_ROOT, "feature_audit")[0]),
    ("india_firms_root", cu.path_ok(PROJECT_ROOT / "India_Firms.parquet")
     and PROJECT_ROOT or None),
    ("qc", cu.path_ok(PROJECT_ROOT / "qc") and PROJECT_ROOT / "qc" or None),
    ("qc_canon", cu.path_ok(_CANON / "qc") and _CANON / "qc" or None),
]
report["folders"] = {}
print(f"{'label':16s} {'files':>7s} {'parquet':>8s} {'csv':>6s} "
      f"{'json':>6s} {'other':>6s}  path")
for label, path in folders:
    if path is None:
        print(f"{label:16s} {'-':>7s}  (not found)")
        report["folders"][label] = None
        continue
    try:
        names = [p.name for p in path.iterdir()]
    except OSError as exc:
        print(f"{label:16s}  !! {exc}")
        continue
    ext = Counter(Path(n).suffix.lower() for n in names)
    dirs = [n for n in names if (path / n).is_dir()]
    files = [n for n in names if not (path / n).is_dir()]
    fext = Counter(Path(n).suffix.lower() for n in files)
    other = len(files) - fext.get(".parquet", 0) - fext.get(".csv", 0) - fext.get(
        ".json", 0
    )
    print(f"{label:16s} {len(files):>7d} {fext.get('.parquet',0):>8d} "
          f"{fext.get('.csv',0):>6d} {fext.get('.json',0):>6d} {other:>6d}  "
          f"{path.name}/")
    if dirs:
        print(f"{'':16s} {'':7s} subdirs: {sorted(dirs)[:8]}")
    report["folders"][label] = {
        "path": str(path),
        "n_files": len(files),
        "n_dirs": len(dirs),
        "extensions": dict(fext),
        "subdirs": sorted(dirs)[:20],
        "file_samples": sorted(files)[:8],
    }

# ===========================================================================
# 2. EVENT BASE -- the frozen 7,265,849-row table
# ===========================================================================

print()
print("=" * 92)
print("2. EVENT BASE")
print("=" * 92)

events_dir = cu.find_artefact(PROJECT_ROOT, "events")[0]
if events_dir is None:
    raise FileNotFoundError("events/ not found under the project root")

report["event_tables"] = {}
schema_errors = []
event_files = sorted(p for p in events_dir.rglob("*.parquet") if p.is_file())
for path in event_files:
    info = schema_of(path)
    if "error" in info:
        schema_errors.append((path, info["error"]))
        print(f"  {str(path.relative_to(events_dir)):46s} !! {info['error']}")
        continue
    print()
    print(f"  {path.relative_to(events_dir)}   rows={info['n_rows']:>10,}  "
          f"row_groups={info['n_row_groups']:>4d}  cols={len(info['columns'])}")
    for col, typ in info["types"].items():
        print(f"    {col:34s} {typ}")
    report["event_tables"][str(path.relative_to(events_dir))] = info

# A schema probe that fails is an ERROR, not an empty schema. The first run of
# this script reported "0 detection columns" purely because of that bug, which
# would have read as "the inputs are missing" and sent us re-downloading data
# that was present all along. Never let it be silent again.
if schema_errors:
    print()
    print("=" * 92)
    print(f"SCHEMA PROBE FAILED FOR {len(schema_errors)} FILE(S)")
    print("=" * 92)
    for path, err in schema_errors[:10]:
        print(f"  {path.name}: {err}")
    print()
    print("The availability verdict below would be MEANINGLESS. Fix the probe")
    print("before reading anything into it.")
    raise RuntimeError(
        f"{len(schema_errors)} schema probe(s) failed -- availability verdict "
        "is not trustworthy"
    )

# Verify the frozen row count and inspect the coordinate magnitudes once.
for name in ("canonical_event_base.parquet", "event_base.parquet"):
    candidate = events_dir / name
    if cu.path_ok(candidate):
        info = schema_of(candidate)
        if info.get("n_rows") == 7_265_849:
            print()
            print(f"  FROZEN ROW COUNT CONFIRMED: {name} = "
                  f"{info['n_rows']:,}")
            report["event_base_rows"] = info["n_rows"]
            probe_cols = [
                c
                for c in (
                    "event_centroid_x", "event_centroid_y",
                    "event_centroid_lon", "event_centroid_lat",
                    "event_id", "start_time", "end_time",
                )
                if c in info["columns"]
            ]
            head = pd.read_parquet(candidate, columns=probe_cols).head(2000)
            for col in probe_cols:
                s = pd.to_numeric(head[col], errors="coerce")
                if s.notna().any():
                    print(f"    {col:26s} min={s.min():>16.3f} "
                          f"max={s.max():>16.3f}")
            print()
            print("  SAMPLE ROWS")
            for row in head.head(3).to_dict("records"):
                print("   ", {k: row[k] for k in list(row)[:6]})
            report["event_base_probe"] = {
                c: {
                    "min": float(pd.to_numeric(head[c], errors="coerce").min()),
                    "max": float(pd.to_numeric(head[c], errors="coerce").max()),
                }
                for c in probe_cols
                if pd.to_numeric(head[c], errors="coerce").notna().any()
            }
        break

# Membership: the event -> local cluster bridge.
print()
print("membership table:")
for path in sorted(events_dir.glob("*membership*.parquet")):
    info = schema_of(path)
    print(f"  {path.name}: rows={info.get('n_rows', 0):,} "
          f"cols={info.get('columns')}")
    report["membership"] = {"file": path.name, **info}
    head = pd.read_parquet(path).head(2000)
    for col in head.columns:
        print(f"    {col:32s} sample={list(head[col].unique()[:3])}")

# ===========================================================================
# 2b. INDIA_FIRMS -- the DETECTION-level table
#
# This is the only table with per-detection radiometry. clusters/ holds
# cluster-level aggregates (n_detections, total_frp, max_frp, ...) and has NO
# frp/i4/i5/confidence columns, so the lineage pass cannot be fed from it.
# ===========================================================================

print()
print("=" * 92)
print("2b. INDIA_FIRMS (detection level)")
print("=" * 92)

firms_candidates = []
_direct = PROJECT_ROOT / "India_Firms.parquet"
if cu.path_ok(_direct):
    firms_candidates.append(_direct)
for leaf in ("India_Firms", "india_firms"):
    hits = cu.find_all_artefact(PROJECT_ROOT, leaf)
    firms_candidates.extend(h for h, _ in hits if h.is_file())

report["india_firms"] = None
if not firms_candidates:
    print("  India_Firms.parquet NOT FOUND -- searching the whole project root")
else:
    path = firms_candidates[0]
    info = schema_of(path)
    print()
    print(f"  {path}")
    if "error" in info:
        print(f"  !! {info['error']}")
    else:
        print(f"  rows={info['n_rows']:,}  row_groups={info['n_row_groups']}  "
              f"cols={len(info['columns'])}")
        for col, typ in info["types"].items():
            print(f"    {col:34s} {typ}")
        entry = dict(info)
        entry["path"] = str(path)
        # ONE data sample: the id formats and value ranges that decide how the
        # mapping bridge joins.
        try:
            head = pd.read_parquet(path).head(2000)
            print()
            print("  sample values:")
            for col in head.columns:
                s = head[col]
                if pd.api.types.is_numeric_dtype(s):
                    print(f"    {col:32s} min={s.min():>16.3f} max={s.max():>16.3f}")
                else:
                    print(f"    {col:32s} {list(s.dropna().unique()[:3])}")
            entry["sample"] = {
                c: (list(map(str, head[c].dropna().unique()[:3]))
                    if not pd.api.types.is_numeric_dtype(head[c])
                    else {"min": float(head[c].min()), "max": float(head[c].max())})
                for c in head.columns
            }
        except Exception as exc:                 # noqa: BLE001
            entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
        report["india_firms"] = entry

# ===========================================================================
# 3. CLUSTERS / EDGES / MAPPINGS -- schema of ONE file, dates from filenames
# ===========================================================================

print()
print("=" * 92)
print("3. CLUSTERS / EDGES / MAPPINGS")
print("=" * 92)

import lineage_features as lf

report["daily_sets"] = {}
for label, leaf, prefix in (
    ("clusters", "clusters", "clusters"),
    ("edges", "edges", "edges"),
    ("mappings", "mappings", None),
):
    path = cu.find_artefact(PROJECT_ROOT, leaf)[0]
    if path is None:
        print(f"  {label}: NOT FOUND")
        continue
    index, stats = lf.index_daily_files(path, prefix=prefix)
    dates = sorted(index)
    print()
    print(f"  {label}/  {len(index):,} dated files  {index[dates[0]].name if dates else ''}"
          f" .. {index[dates[-1]].name if dates else ''}")
    print(f"    stats: {stats}")
    entry = {"path": str(path), "n_dated_files": len(index),
             "first": dates[0] if dates else None,
             "last": dates[-1] if dates else None,
             "index_stats": stats}
    if dates:
        info = schema_of(index[dates[0]])
        print(f"    schema of {index[dates[0]].name}: "
              f"rows={info.get('n_rows', 0):,}")
        for col, typ in info.get("types", {}).items():
            print(f"      {col:30s} {typ}")
        entry["schema"] = info
    report["daily_sets"][label] = entry

# ===========================================================================
# 4. GAZETTEER -- one file per distinct schema, geometry + CRS + categories
# ===========================================================================

print()
print("=" * 92)
print("4. EXTERNAL / GAZETTEER")
print("=" * 92)

external = cu.find_artefact(PROJECT_ROOT, "external")[0]
if external is None:
    print("  external/ NOT FOUND")
else:
    report["external"] = {}
    ext_subfolders = {}
    for child in sorted(external.iterdir()):
        if child.is_dir():
            try:
                ext_subfolders[child.name] = sum(
                    1 for _ in child.rglob("*") if _.is_file()
                )
            except OSError:
                ext_subfolders[child.name] = -1

    # Subfolder census: names + counts, no reads.
    print("  external/ subfolders:")
    for name, count in sorted(ext_subfolders.items()):
        print(f"    {name:42s} {count:>6d} files")
    report["external"]["subfolders"] = ext_subfolders

    # Gazetteer: dedupe by name (same as P3a) then sample ONE per schema.
    fac_dir = external / "osm_facilities"
    if cu.is_dir(fac_dir):
        seen = {}
        for p in sorted(fac_dir.rglob("*.geoparquet")):
            penalty = 1_000 * sum(
                1
                for part in p.relative_to(fac_dir).parts[:-1]
                if re.search(r"\s*\(\d+\)$|\b_v\d+\b", part)
            )
            if p.name not in seen or penalty < seen[p.name][0]:
                seen[p.name] = (penalty, p)
        gaz = [v[1] for v in seen.values()]
        print()
        print(f"  osm_facilities: {len(gaz)} unique files "
              f"(duplicates collapsed)")

        by_sig = {}
        for p in gaz:
            info = schema_of(p)
            if "error" in info:
                continue
            by_sig.setdefault(signature(info), []).append((p, info))

        gaz_report = {}
        for i, (sig, group) in enumerate(sorted(by_sig.items(),
                                                key=lambda kv: -len(kv[1]))):
            if i >= SAMPLE_MAX_FILES:
                break
            sample_path, info = group[0]
            print()
            print(f"    [{len(group)} file(s) share this schema] "
                  f"{sample_path.name}")
            for col, typ in info["types"].items():
                print(f"      {col:30s} {typ}")
            entry = {
                "n_files_sharing_schema": len(group),
                "sample": str(sample_path.relative_to(fac_dir)),
                "columns": info["columns"],
                "types": info["types"],
                "rows": info["n_rows"],
                "files": [str(q.relative_to(fac_dir)) for q, _ in group][:10],
            }
            # One data sample: geometry types, CRS, category values.
            try:
                head = pd.read_parquet(sample_path).head(SAMPLE_ROWS * 200)
                geom_col = next(
                    (c for c in ("geometry", "geom", "wkb_geometry")
                     if c in head.columns),
                    None,
                )
                if geom_col:
                    import shapely
                    from shapely import from_wkb

                    geoms = from_wkb(
                        np.asarray(head[geom_col].to_numpy(zero_copy_only=False))
                    )
                    entry["geometry_types"] = dict(
                        Counter(g.geom_type for g in geoms if g is not None)
                    )
                    entry["crs"] = (
                        head.attrs.get("crs")
                        or str(getattr(sample_path, "crs", ""))[:60]
                    )
                    print(f"      geometry types: {entry['geometry_types']}")
                for cand in ("category", "facility_type", "type", "class"):
                    if cand in head.columns:
                        vals = head[cand].dropna().astype(str)
                        entry["category_column"] = cand
                        entry["category_values"] = sorted(vals.unique())[:25]
                        print(f"      {cand}: "
                              f"{entry['category_values'][:12]}")
                        break
            except Exception as exc:                 # noqa: BLE001
                entry["sample_error"] = f"{exc.__class__.__name__}: {exc}"
            gaz_report[str(sample_path.relative_to(fac_dir))] = entry

        report["external"]["gazetteer"] = gaz_report

# ===========================================================================
# 5. FEATURE INPUT AVAILABILITY -- the point of the whole exercise
# ===========================================================================

print()
print("=" * 92)
print("5. FEATURE INPUT AVAILABILITY (141 features)")
print("=" * 92)

import feature_ledger as fl

# The detection columns live in India_Firms, NOT in clusters/. clusters/ holds
# cluster-level aggregates. Checking the wrong table made the first two recon
# runs report the lineage inputs as "missing" when they were simply being looked
# for in the wrong place.
cluster_cols = set()
if "clusters" in report.get("daily_sets", {}):
    cluster_cols = set(
        report["daily_sets"]["clusters"].get("schema", {}).get("columns", [])
    )
firms_cols = set()
if report.get("india_firms"):
    firms_cols = set(report["india_firms"].get("columns", []))
clusters_cols = firms_cols | cluster_cols   # union: lineage can draw on both
event_cols = set()
for info in report.get("event_tables", {}).values():
    event_cols |= set(info.get("columns", []))

# Which lineage inputs are genuinely present in the detection columns?
wanted_detection_cols = {
    "frp": ("frp", "FRP"),
    "i4": ("i4", "brightness_i4"),
    "i5": ("i5", "brightness_i5"),
    "longitude": ("longitude", "lon"),
    "latitude": ("latitude", "lat"),
    "subpixel_temp": ("subpixel_temp",),
    "source_area": ("source_area",),
    "confidence": ("confidence",),
    "pixel_size": ("pixel_size",),
    "scan_angle": ("scan_angle",),
    "view_zenith": ("view_zenith"),
    "day_night": ("day_night",),
    "satellite": ("satellite", "sat"),
}
present, absent = [], []
for logical, cands in wanted_detection_cols.items():
    (present if clusters_cols & set(cands) else absent).append(logical)

print()
print(f"  India_Firms columns : {len(firms_cols)}")
print(f"  clusters/ columns   : {len(cluster_cols)}")
print(f"  union considered    : {len(clusters_cols)}")
print(f"    present: {present}")
print(f"    absent : {absent}")
print()
print("  NOTE: clusters/ holds cluster aggregates only. Per-detection")
print("  radiometry (i4/i5/confidence/scan angle) is available solely if")
print("  India_Firms carries it -- check section 2b above before assuming.")

gaz_keys = set(report.get("external", {}).get("gazetteer", {}))
print(f"  gazetteer schemas sampled: {len(gaz_keys)}")
print(f"  event table columns: {len(event_cols)}")

# Group-level availability, ledger-driven.
availability = {}
for group in fl.FEATURE_GROUPS:
    names = [
        n for n in fl.FEATURE_GROUPS[group] if n not in fl._PLACEHOLDERS
    ]
    if group == fl.GROUP_INDUSTRIAL:
        state = "ready" if gaz_keys else "needs gazetteer"
    elif group in (fl.GROUP_THERMAL, fl.GROUP_TEMPORAL, fl.GROUP_SPREAD,
                   fl.GROUP_QUALITY):
        state = "ready" if present else f"missing {absent}"
    elif group == fl.GROUP_TERRAIN:
        state = "cell 263 (GEE)"
    else:
        state = "needs GEE / external"
    availability[group] = {"n": len(names), "state": state}
    print(f"  {group:24s} {len(names):>3d}  {state}")

report["availability"] = availability
report["detection_columns_present"] = present
report["detection_columns_absent"] = absent
report["cluster_columns"] = sorted(clusters_cols)
report["event_columns"] = sorted(event_cols)

# ===========================================================================
# 6. WRITE THE REPORT
# ===========================================================================

print()
print("=" * 92)
print(f"elapsed: {time.time() - t_start:.1f}s")
print("=" * 92)

qc_dir = PROJECT_ROOT / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
out = qc_dir / "project_recon.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"Wrote {out}  ({out.stat().st_size / 1024:.1f} KB)")

print()
print("This is the reference for every path and column name from here on.")
print("If a cell hardcodes something this report contradicts, the report wins.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = True
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("Set INSPECT_ONLY = False to run the full pass.")
    raise SystemExit(0)

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

print()
print("=" * 74)
print("P4a TIER 1 -- FULL PASS")
print("=" * 74)

OUT_DIR.mkdir(parents=True, exist_ok=True)
DATE_BATCH = max(1, int(DATE_BATCH))

null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
rows_out = 0
chunk_files = []
events_seen = set()
unmatched_clusters = 0
clusters_read = 0

for batch_no, start in enumerate(range(0, len(ordered_dates), DATE_BATCH)):
    window = ordered_dates[start:start + DATE_BATCH]

    tables = []
    for day in window:
        path = CLUSTER_INDEX[day]
        if not cu.path_ok(path):
            print(f"  !! unreadable {path.name}")
            continue
        tables.append(pq.read_table(path))
    if not tables:
        continue
    det = pa.concat_tables(tables).to_pandas()
    del tables

    clusters_read += len(det)
    det[sample_cluster_col] = det[sample_cluster_col].astype(str)

    # Keep only clusters the membership table knows about, and say how many
    # were dropped: an unexpected number here means the join keys disagree.
    known = det[sample_cluster_col].isin(member_index.index)
    n_unknown = int((~known).sum())
    if n_unknown:
        unmatched_clusters += n_unknown
        if batch_no == 0:
            print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                  f"in membership (expected only if this is the first day)")
    det = det[known]
    if det.empty:
        continue

    # The membership table carries its own `date`, which collides with the
    # cluster file's `date` and makes the join raise outright. Bring across
    # only what is actually needed.
    bring = [MEM_EVENT_COL]
    if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
        bring.append(MEM_DATE_COL)
    elif MEM_DATE_COL:
        # Same name on both sides: keep the cluster file's own value and skip
        # the duplicate rather than failing on an overlapping column.
        print("  note: membership date column shares a name with the cluster "
              "file; using the cluster file's value")

    joined = det.join(
        member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
    )
    if joined.empty:
        print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
        break

    computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
    # One row per event: an event spans many daily batches, so later batches
    # would otherwise re-emit events already written.
    fresh = computed[~computed.index.isin(events_seen)]
    if fresh.empty:
        continue
    events_seen.update(fresh.index.tolist())

    for name in ct.CLUSTER_TIER_FEATURES:
        null_counts[name] += int(fresh[name].isna().sum())

    out = fresh.reset_index()
    path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
    out.to_parquet(path_out, index=False, compression="snappy")
    chunk_files.append({"file": path_out.name, "rows": len(out)})
    rows_out += len(out)

    if batch_no % 5 == 0:
        print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
              f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

# ---------------------------------------------------------------------------
# 7. QC
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("P4a TIER 1 -- QC")
print("=" * 74)
print(f"cluster rows read      : {clusters_read:,}")
print(f"clusters not in member : {unmatched_clusters:,}")
print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
      f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
    ct.CLUSTER_TIER_FEATURES
print(f"feature columns        : {len(features_cols)}")

missing = n_events_total - rows_out
if missing > 0:
    print()
    print(f"  NOTE: {missing:,} membership events produced no features. That is")
    print("  expected only if those events have no clusters on disk. Events with")
    print("  a single cluster still produce FRP/count/duration values; only the")
    print("  trend and shape features are NaN for them.")

print()
print(f"{'feature':32s} {'NaN %':>8s}")
summary = {}
for name in ct.CLUSTER_TIER_FEATURES:
    frac = null_counts[name] / rows_out if rows_out else float("nan")
    summary[name] = frac
    flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
    print(f"  {name:30s} {frac:>7.2%}{flag}")

manifest = {
    "clusters_dir": str(CLUSTERS_DIR),
    "membership": str(MEMBERSHIP),
    "membership_rows": int(len(membership)),
    "events_in_membership": int(n_events_total),
    "events_written": int(rows_out),
    "cluster_rows_read": int(clusters_read),
    "clusters_not_in_membership": int(unmatched_clusters),
    "date_batch": DATE_BATCH,
    "tier": 1,
    "feature_order": ct.CLUSTER_TIER_FEATURES,
    "nan_fraction": summary,
    "deferred_features": list(ct.DEFERRED_REASONS),
    "chunks": chunk_files,
}
(OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
print()
print("Wrote", OUT_DIR / "manifest_tier1.json")

print()
print("NEXT")
print("  1. Read the NaN table. A high NaN share on a trend feature usually")
print("     means single-cluster events, which is correct. A high NaN share on")
print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
print("  3. Join tier1 + industrial -> 52 features, then label and train.")
print("  4. The 20 deferred features are NOT here by accident -- see")
print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")

In [ ]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = True
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("Set INSPECT_ONLY = False and re-run this cell for the full pass.")
    _STOP = True
else:
    _STOP = False

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

print()
print("=" * 74)
print("P4a TIER 1 -- FULL PASS")
print("=" * 74)

def run_full_pass():
    """Execute the whole pass. Called only when INSPECT_ONLY is False."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    DATE_BATCH = max(1, int(DATE_BATCH))

    null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
    rows_out = 0
    chunk_files = []
    events_seen = set()
    unmatched_clusters = 0
    clusters_read = 0

    for batch_no, start in enumerate(range(0, len(ordered_dates), DATE_BATCH)):
        window = ordered_dates[start:start + DATE_BATCH]

        tables = []
        for day in window:
            path = CLUSTER_INDEX[day]
            if not cu.path_ok(path):
                print(f"  !! unreadable {path.name}")
                continue
            tables.append(pq.read_table(path))
        if not tables:
            continue
        det = pa.concat_tables(tables).to_pandas()
        del tables

        clusters_read += len(det)
        det[sample_cluster_col] = det[sample_cluster_col].astype(str)

        # Keep only clusters the membership table knows about, and say how many
        # were dropped: an unexpected number here means the join keys disagree.
        known = det[sample_cluster_col].isin(member_index.index)
        n_unknown = int((~known).sum())
        if n_unknown:
            unmatched_clusters += n_unknown
            if batch_no == 0:
                print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                      f"in membership (expected only if this is the first day)")
        det = det[known]
        if det.empty:
            continue

        # The membership table carries its own `date`, which collides with the
        # cluster file's `date` and makes the join raise outright. Bring across
        # only what is actually needed.
        bring = [MEM_EVENT_COL]
        if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
            bring.append(MEM_DATE_COL)
        elif MEM_DATE_COL:
            # Same name on both sides: keep the cluster file's own value and skip
            # the duplicate rather than failing on an overlapping column.
            print("  note: membership date column shares a name with the cluster "
                  "file; using the cluster file's value")

        joined = det.join(
            member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
        )
        if joined.empty:
            print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
            break

        computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
        # One row per event: an event spans many daily batches, so later batches
        # would otherwise re-emit events already written.
        fresh = computed[~computed.index.isin(events_seen)]
        if fresh.empty:
            continue
        events_seen.update(fresh.index.tolist())

        for name in ct.CLUSTER_TIER_FEATURES:
            null_counts[name] += int(fresh[name].isna().sum())

        out = fresh.reset_index()
        path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path_out, index=False, compression="snappy")
        chunk_files.append({"file": path_out.name, "rows": len(out)})
        rows_out += len(out)

        if batch_no % 5 == 0:
            print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
                  f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

    # ---------------------------------------------------------------------------
    # 7. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 74)
    print("P4a TIER 1 -- QC")
    print("=" * 74)
    print(f"cluster rows read      : {clusters_read:,}")
    print(f"clusters not in member : {unmatched_clusters:,}")
    print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
          f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

    assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
        ct.CLUSTER_TIER_FEATURES
    print(f"feature columns        : {len(features_cols)}")

    missing = n_events_total - rows_out
    if missing > 0:
        print()
        print(f"  NOTE: {missing:,} membership events produced no features. That is")
        print("  expected only if those events have no clusters on disk. Events with")
        print("  a single cluster still produce FRP/count/duration values; only the")
        print("  trend and shape features are NaN for them.")

    print()
    print(f"{'feature':32s} {'NaN %':>8s}")
    summary = {}
    for name in ct.CLUSTER_TIER_FEATURES:
        frac = null_counts[name] / rows_out if rows_out else float("nan")
        summary[name] = frac
        flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
        print(f"  {name:30s} {frac:>7.2%}{flag}")

    manifest = {
        "clusters_dir": str(CLUSTERS_DIR),
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "events_in_membership": int(n_events_total),
        "events_written": int(rows_out),
        "cluster_rows_read": int(clusters_read),
        "clusters_not_in_membership": int(unmatched_clusters),
        "date_batch": DATE_BATCH,
        "tier": 1,
        "feature_order": ct.CLUSTER_TIER_FEATURES,
        "nan_fraction": summary,
        "deferred_features": list(ct.DEFERRED_REASONS),
        "chunks": chunk_files,
    }
    (OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
    print()
    print("Wrote", OUT_DIR / "manifest_tier1.json")

    print()
    print("NEXT")
    print("  1. Read the NaN table. A high NaN share on a trend feature usually")
    print("     means single-cluster events, which is correct. A high NaN share on")
    print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
    print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
    print("  3. Join tier1 + industrial -> 52 features, then label and train.")
    print("  4. The 20 deferred features are NOT here by accident -- see")
    print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = True          # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("3. Then set INSPECT_ONLY = False and re-run the whole cell.")
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")
    raise SystemExit(0)

# ===========================================================================
# 5. COMPUTE  (only reached once INSPECT_ONLY = False)
# ===========================================================================

import shapely
from shapely import STRtree
from shapely.geometry import LineString, MultiLineString

print()
print("=" * 78)
print("P3a STEP 3 -- PROJECTION AND TREES")
print("=" * 78)

from pyproj import Transformer

to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


def project_points(lon, lat):
    """Vectorised lon/lat -> Equal Earth metres."""
    x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                               np.asarray(lat, dtype="float64"))
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    # Infinite transform results mean the point fell outside the projection.
    bad = ~np.isfinite(x) | ~np.isfinite(y)
    if bad.any():
        print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
        x[bad] = np.nan
        y[bad] = np.nan
    return x, y


# --- per-category point trees ----------------------------------------------
category_trees: dict = {}
for cat in ind.CATEGORIES:
    pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
    if pts:
        arr = np.array(pts, dtype="float64")
        catx, caty = project_points(arr[:, 0], arr[:, 1])
        good = np.isfinite(catx) & np.isfinite(caty)
        category_trees[cat] = np.column_stack([catx[good], caty[good]])
    else:
        category_trees[cat] = None
    n = 0 if category_trees[cat] is None else len(category_trees[cat])
    print(f"  {cat:14s} point tree: {n:>8,d}")

# --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
boundary_xy = None
if boundary_chunks:
    raw = np.concatenate(boundary_chunks, axis=0)
    bx, by = project_points(raw[:, 0], raw[:, 1])
    good = np.isfinite(bx) & np.isfinite(by)
    boundary_xy = np.column_stack([bx[good], by[good]])
    print(f"  boundary samples: {len(boundary_xy):,d} "
          f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
else:
    print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

# --- polygons for INSIDE_FACILITY -------------------------------------------
# The original geometries are reused here, not the densified ring samples: the
# point-in-polygon answer must come from valid closed rings.
polys = [rec["geom"] for rec in records if rec["geom"] is not None]
polys = [p for p in polys if p.is_valid and not p.is_empty]
inside_tree = STRtree(polys) if polys else None
print(f"  STRtree polygons: {len(polys)}")
if not polys:
    print("  no usable polygons -> INSIDE_FACILITY will be NaN")

print()
print("=" * 78)
print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
print("=" * 78)

OUT_DIR.mkdir(parents=True, exist_ok=True)
read_cols = [X_COL, Y_COL]
read_cols += list(present_flare.values())
read_cols += [c for c in ("event_id",) if c in event_columns]
read_cols = list(dict.fromkeys(read_cols))
print("reading columns:", read_cols)

need_cols = set(read_cols)
null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
rows_written = 0
chunk_files = []
events_seen = 0

batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
for batch_no, batch in enumerate(batch_iter):
    frame = batch.to_pandas()
    if frame.empty:
        continue
    if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
        break

    col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
    col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
    if COORD_KIND == "projected metres":
        # Already Equal Earth metres. Projecting again would rescale every
        # distance, so this branch deliberately does nothing.
        x, y = col_x, col_y
    else:
        x, y = project_points(col_x, col_y)
    pts = np.column_stack([x, y])
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        continue
    if not finite.all():
        # Unprojectable centroids still need their 18 keys emitted as NaN.
        pts = np.where(finite[:, None], pts, np.nan)

    inside = None
    if inside_tree is not None:
        geoms = shapely.points(x, y)
        hits = inside_tree.query(geoms, predicate="intersects")
        inside = np.zeros(len(pts), dtype="float64")
        inside[np.unique(hits[0])] = 1.0
        del geoms

    boundary = None
    if boundary_xy is not None:
        boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
        full = np.full(len(pts), np.nan)
        full[finite] = boundary
        boundary = full

    flare_site = None
    if inside_tree is not None:
        # A centroid inside any polygon is treated as flare-capable only when
        # the polygon belongs to a flare-capable category; the STRtree here is
        # over all polygons, so fall back to the union distance plus the
        # category trees for the per-category test.
        flare_site = np.zeros(len(pts), dtype=bool)
        flare_pts = []
        for cat in ind.FLARE_SITE_CATEGORIES:
            tree = category_trees.get(cat)
            if tree is None or len(tree) == 0:
                continue
            d = ind.nearest_distance(pts[finite], tree)
            sub = np.zeros(int(finite.sum()), dtype=bool)
            sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
            flare_pts.append(sub)
        if flare_pts:
            flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
        else:
            flare_site = None

    support = {}
    for key, col in present_flare.items():
        support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

    features = ind.compute_industrial_features(
        pts,
        category_trees,
        crs=TARGET_CRS,
        inside_flags=inside,
        boundary_distance_m=boundary,
        category_is_flare_site=flare_site,
        **support,
    )

    out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
    if "event_id" in frame.columns:
        out.insert(0, "event_id", frame["event_id"].to_numpy())

    for name in ind.INDUSTRIAL_FEATURES:
        null_counts[name] += int(out[name].isna().sum())

    path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
    out.to_parquet(path, index=False, compression="snappy")
    chunk_files.append({"file": path.name, "rows": len(out)})
    rows_written += len(out)
    events_seen += len(out)

    if batch_no % 5 == 0:
        print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
              f"total={rows_written:>10,d}")

print()
print(f"rows written: {rows_written:,}")

# ---------------------------------------------------------------------------
# 6. QC
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a QC")
print("=" * 78)
print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
summary = {}
for name in ind.INDUSTRIAL_FEATURES:
    nulls = null_counts[name]
    frac = 100.0 * nulls / rows_written if rows_written else float("nan")
    summary[name] = {"nan_fraction": frac, "nan_count": nulls}
    print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "event_base": str(EVENT_BASE),
    "events_total_in_table": event_pf.metadata.num_rows,
    "rows_written": rows_written,
    "target_crs": TARGET_CRS,
    "chunk_rows": CHUNK_ROWS,
    "max_events": MAX_EVENTS,
    "boundary_densify_m": BOUNDARY_DENSIFY_M,
    "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
    "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
    "feature_order": ind.INDUSTRIAL_FEATURES,
    "null_fraction": summary,
    "chunks": chunk_files,
}
with open(OUT_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print("Wrote", OUT_DIR / "manifest.json")

print()
print("Status to record in feature_ledger.py once you have inspected the")
print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
print("runtime_verified stays False until P2/P4 actually execute them.")

In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print(f"MODE: {MODE}")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = False         # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")

# ===========================================================================
def run_full_pass():
    """Run the 18 industrial features over every event."""
    # 5. COMPUTE  (only reached once INSPECT_ONLY = False)
    # ===========================================================================

    import shapely
    from shapely import STRtree
    from shapely.geometry import LineString, MultiLineString

    print()
    print("=" * 78)
    print("P3a STEP 3 -- PROJECTION AND TREES")
    print("=" * 78)

    from pyproj import Transformer

    to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


    def project_points(lon, lat):
        """Vectorised lon/lat -> Equal Earth metres."""
        x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                                   np.asarray(lat, dtype="float64"))
        x = np.asarray(x, dtype="float64")
        y = np.asarray(y, dtype="float64")
        # Infinite transform results mean the point fell outside the projection.
        bad = ~np.isfinite(x) | ~np.isfinite(y)
        if bad.any():
            print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
            x[bad] = np.nan
            y[bad] = np.nan
        return x, y


    # --- per-category point trees ----------------------------------------------
    category_trees: dict = {}
    for cat in ind.CATEGORIES:
        pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
        if pts:
            arr = np.array(pts, dtype="float64")
            catx, caty = project_points(arr[:, 0], arr[:, 1])
            good = np.isfinite(catx) & np.isfinite(caty)
            category_trees[cat] = np.column_stack([catx[good], caty[good]])
        else:
            category_trees[cat] = None
        n = 0 if category_trees[cat] is None else len(category_trees[cat])
        print(f"  {cat:14s} point tree: {n:>8,d}")

    # --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
    boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
    boundary_xy = None
    if boundary_chunks:
        raw = np.concatenate(boundary_chunks, axis=0)
        bx, by = project_points(raw[:, 0], raw[:, 1])
        good = np.isfinite(bx) & np.isfinite(by)
        boundary_xy = np.column_stack([bx[good], by[good]])
        print(f"  boundary samples: {len(boundary_xy):,d} "
              f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
    else:
        print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

    # --- polygons for INSIDE_FACILITY -------------------------------------------
    # The original geometries are reused here, not the densified ring samples: the
    # point-in-polygon answer must come from valid closed rings.
    polys = [rec["geom"] for rec in records if rec["geom"] is not None]
    polys = [p for p in polys if p.is_valid and not p.is_empty]
    inside_tree = STRtree(polys) if polys else None
    print(f"  STRtree polygons: {len(polys)}")
    if not polys:
        print("  no usable polygons -> INSIDE_FACILITY will be NaN")

    print()
    print("=" * 78)
    print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
    print("=" * 78)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    read_cols = [X_COL, Y_COL]
    read_cols += list(present_flare.values())
    read_cols += [c for c in ("event_id",) if c in event_columns]
    read_cols = list(dict.fromkeys(read_cols))
    print("reading columns:", read_cols)

    need_cols = set(read_cols)
    null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
    rows_written = 0
    chunk_files = []
    events_seen = 0

    batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
    for batch_no, batch in enumerate(batch_iter):
        frame = batch.to_pandas()
        if frame.empty:
            continue
        if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
            break

        col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
        col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
        if COORD_KIND == "projected metres":
            # Already Equal Earth metres. Projecting again would rescale every
            # distance, so this branch deliberately does nothing.
            x, y = col_x, col_y
        else:
            x, y = project_points(col_x, col_y)
        pts = np.column_stack([x, y])
        finite = np.isfinite(x) & np.isfinite(y)
        if not finite.any():
            continue
        if not finite.all():
            # Unprojectable centroids still need their 18 keys emitted as NaN.
            pts = np.where(finite[:, None], pts, np.nan)

        inside = None
        if inside_tree is not None:
            geoms = shapely.points(x, y)
            hits = inside_tree.query(geoms, predicate="intersects")
            inside = np.zeros(len(pts), dtype="float64")
            inside[np.unique(hits[0])] = 1.0
            del geoms

        boundary = None
        if boundary_xy is not None:
            boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
            full = np.full(len(pts), np.nan)
            full[finite] = boundary
            boundary = full

        flare_site = None
        if inside_tree is not None:
            # A centroid inside any polygon is treated as flare-capable only when
            # the polygon belongs to a flare-capable category; the STRtree here is
            # over all polygons, so fall back to the union distance plus the
            # category trees for the per-category test.
            flare_site = np.zeros(len(pts), dtype=bool)
            flare_pts = []
            for cat in ind.FLARE_SITE_CATEGORIES:
                tree = category_trees.get(cat)
                if tree is None or len(tree) == 0:
                    continue
                d = ind.nearest_distance(pts[finite], tree)
                sub = np.zeros(int(finite.sum()), dtype=bool)
                sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
                flare_pts.append(sub)
            if flare_pts:
                flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
            else:
                flare_site = None

        support = {}
        for key, col in present_flare.items():
            support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

        features = ind.compute_industrial_features(
            pts,
            category_trees,
            crs=TARGET_CRS,
            inside_flags=inside,
            boundary_distance_m=boundary,
            category_is_flare_site=flare_site,
            **support,
        )

        out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
        if "event_id" in frame.columns:
            out.insert(0, "event_id", frame["event_id"].to_numpy())

        for name in ind.INDUSTRIAL_FEATURES:
            null_counts[name] += int(out[name].isna().sum())

        path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path, index=False, compression="snappy")
        chunk_files.append({"file": path.name, "rows": len(out)})
        rows_written += len(out)
        events_seen += len(out)

        if batch_no % 5 == 0:
            print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
                  f"total={rows_written:>10,d}")

    print()
    print(f"rows written: {rows_written:,}")

    # ---------------------------------------------------------------------------
    # 6. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 78)
    print("P3a QC")
    print("=" * 78)
    print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
    summary = {}
    for name in ind.INDUSTRIAL_FEATURES:
        nulls = null_counts[name]
        frac = 100.0 * nulls / rows_written if rows_written else float("nan")
        summary[name] = {"nan_fraction": frac, "nan_count": nulls}
        print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "event_base": str(EVENT_BASE),
        "events_total_in_table": event_pf.metadata.num_rows,
        "rows_written": rows_written,
        "target_crs": TARGET_CRS,
        "chunk_rows": CHUNK_ROWS,
        "max_events": MAX_EVENTS,
        "boundary_densify_m": BOUNDARY_DENSIFY_M,
        "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
        "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
        "feature_order": ind.INDUSTRIAL_FEATURES,
        "null_fraction": summary,
        "chunks": chunk_files,
    }
    with open(OUT_DIR / "manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", OUT_DIR / "manifest.json")

    print()
    print("Status to record in feature_ledger.py once you have inspected the")
    print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
    print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
    print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
    print("runtime_verified stays False until P2/P4 actually execute them.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = False
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print(f"MODE: {MODE}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

def run_full_pass():
    """Execute the whole pass. Called only when INSPECT_ONLY is False."""
    print()
    print("=" * 74)
    print("P4a TIER 1 -- FULL PASS  (writing to features/lineage/)")
    print("=" * 74)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    DATE_BATCH = max(1, int(DATE_BATCH))

    null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
    rows_out = 0
    chunk_files = []
    events_seen = set()
    unmatched_clusters = 0
    clusters_read = 0

    for batch_no, start in enumerate(range(0, len(ordered_dates), DATE_BATCH)):
        window = ordered_dates[start:start + DATE_BATCH]

        tables = []
        for day in window:
            path = CLUSTER_INDEX[day]
            if not cu.path_ok(path):
                print(f"  !! unreadable {path.name}")
                continue
            tables.append(pq.read_table(path))
        if not tables:
            continue
        det = pa.concat_tables(tables).to_pandas()
        del tables

        clusters_read += len(det)
        det[sample_cluster_col] = det[sample_cluster_col].astype(str)

        # Keep only clusters the membership table knows about, and say how many
        # were dropped: an unexpected number here means the join keys disagree.
        known = det[sample_cluster_col].isin(member_index.index)
        n_unknown = int((~known).sum())
        if n_unknown:
            unmatched_clusters += n_unknown
            if batch_no == 0:
                print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                      f"in membership (expected only if this is the first day)")
        det = det[known]
        if det.empty:
            continue

        # The membership table carries its own `date`, which collides with the
        # cluster file's `date` and makes the join raise outright. Bring across
        # only what is actually needed.
        bring = [MEM_EVENT_COL]
        if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
            bring.append(MEM_DATE_COL)
        elif MEM_DATE_COL:
            # Same name on both sides: keep the cluster file's own value and skip
            # the duplicate rather than failing on an overlapping column.
            print("  note: membership date column shares a name with the cluster "
                  "file; using the cluster file's value")

        joined = det.join(
            member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
        )
        if joined.empty:
            print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
            break

        computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
        # One row per event: an event spans many daily batches, so later batches
        # would otherwise re-emit events already written.
        fresh = computed[~computed.index.isin(events_seen)]
        if fresh.empty:
            continue
        events_seen.update(fresh.index.tolist())

        for name in ct.CLUSTER_TIER_FEATURES:
            null_counts[name] += int(fresh[name].isna().sum())

        out = fresh.reset_index()
        path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path_out, index=False, compression="snappy")
        chunk_files.append({"file": path_out.name, "rows": len(out)})
        rows_out += len(out)

        if batch_no % 5 == 0:
            print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
                  f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

    # ---------------------------------------------------------------------------
    # 7. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 74)
    print("P4a TIER 1 -- QC")
    print("=" * 74)
    print(f"cluster rows read      : {clusters_read:,}")
    print(f"clusters not in member : {unmatched_clusters:,}")
    print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
          f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

    assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
        ct.CLUSTER_TIER_FEATURES
    print(f"feature columns        : {len(features_cols)}")

    missing = n_events_total - rows_out
    if missing > 0:
        print()
        print(f"  NOTE: {missing:,} membership events produced no features. That is")
        print("  expected only if those events have no clusters on disk. Events with")
        print("  a single cluster still produce FRP/count/duration values; only the")
        print("  trend and shape features are NaN for them.")

    print()
    print(f"{'feature':32s} {'NaN %':>8s}")
    summary = {}
    for name in ct.CLUSTER_TIER_FEATURES:
        frac = null_counts[name] / rows_out if rows_out else float("nan")
        summary[name] = frac
        flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
        print(f"  {name:30s} {frac:>7.2%}{flag}")

    manifest = {
        "clusters_dir": str(CLUSTERS_DIR),
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "events_in_membership": int(n_events_total),
        "events_written": int(rows_out),
        "cluster_rows_read": int(clusters_read),
        "clusters_not_in_membership": int(unmatched_clusters),
        "date_batch": DATE_BATCH,
        "tier": 1,
        "feature_order": ct.CLUSTER_TIER_FEATURES,
        "nan_fraction": summary,
        "deferred_features": list(ct.DEFERRED_REASONS),
        "chunks": chunk_files,
    }
    (OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
    print()
    print("Wrote", OUT_DIR / "manifest_tier1.json")

    print()
    print("NEXT")
    print("  1. Read the NaN table. A high NaN share on a trend feature usually")
    print("     means single-cluster events, which is correct. A high NaN share on")
    print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
    print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
    print("  3. Join tier1 + industrial -> 52 features, then label and train.")
    print("  4. The 20 deferred features are NOT here by accident -- see")
    print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = False
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print(f"MODE: {MODE}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

def run_full_pass():
    """Execute the whole pass. Called only when INSPECT_ONLY is False."""
    print()
    print("=" * 74)
    print("P4a TIER 1 -- FULL PASS  (writing to features/lineage/)")
    print("=" * 74)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    batch_size = max(1, int(DATE_BATCH))

    null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
    rows_out = 0
    chunk_files = []
    events_seen = set()
    unmatched_clusters = 0
    clusters_read = 0

    for batch_no, start in enumerate(range(0, len(ordered_dates), batch_size)):
        window = ordered_dates[start:start + batch_size]

        tables = []
        for day in window:
            path = CLUSTER_INDEX[day]
            if not cu.path_ok(path):
                print(f"  !! unreadable {path.name}")
                continue
            tables.append(pq.read_table(path))
        if not tables:
            continue
        det = pa.concat_tables(tables).to_pandas()
        del tables

        clusters_read += len(det)
        det[sample_cluster_col] = det[sample_cluster_col].astype(str)

        # Keep only clusters the membership table knows about, and say how many
        # were dropped: an unexpected number here means the join keys disagree.
        known = det[sample_cluster_col].isin(member_index.index)
        n_unknown = int((~known).sum())
        if n_unknown:
            unmatched_clusters += n_unknown
            if batch_no == 0:
                print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                      f"in membership (expected only if this is the first day)")
        det = det[known]
        if det.empty:
            continue

        # The membership table carries its own `date`, which collides with the
        # cluster file's `date` and makes the join raise outright. Bring across
        # only what is actually needed.
        bring = [MEM_EVENT_COL]
        if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
            bring.append(MEM_DATE_COL)
        elif MEM_DATE_COL:
            # Same name on both sides: keep the cluster file's own value and skip
            # the duplicate rather than failing on an overlapping column.
            print("  note: membership date column shares a name with the cluster "
                  "file; using the cluster file's value")

        joined = det.join(
            member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
        )
        if joined.empty:
            print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
            break

        computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
        # One row per event: an event spans many daily batches, so later batches
        # would otherwise re-emit events already written.
        fresh = computed[~computed.index.isin(events_seen)]
        if fresh.empty:
            continue
        events_seen.update(fresh.index.tolist())

        for name in ct.CLUSTER_TIER_FEATURES:
            null_counts[name] += int(fresh[name].isna().sum())

        out = fresh.reset_index()
        path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path_out, index=False, compression="snappy")
        chunk_files.append({"file": path_out.name, "rows": len(out)})
        rows_out += len(out)

        if batch_no % 5 == 0:
            print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
                  f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

    # ---------------------------------------------------------------------------
    # 7. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 74)
    print("P4a TIER 1 -- QC")
    print("=" * 74)
    print(f"cluster rows read      : {clusters_read:,}")
    print(f"clusters not in member : {unmatched_clusters:,}")
    print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
          f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

    assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
        ct.CLUSTER_TIER_FEATURES
    print(f"feature columns        : {len(features_cols)}")

    missing = n_events_total - rows_out
    if missing > 0:
        print()
        print(f"  NOTE: {missing:,} membership events produced no features. That is")
        print("  expected only if those events have no clusters on disk. Events with")
        print("  a single cluster still produce FRP/count/duration values; only the")
        print("  trend and shape features are NaN for them.")

    print()
    print(f"{'feature':32s} {'NaN %':>8s}")
    summary = {}
    for name in ct.CLUSTER_TIER_FEATURES:
        frac = null_counts[name] / rows_out if rows_out else float("nan")
        summary[name] = frac
        flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
        print(f"  {name:30s} {frac:>7.2%}{flag}")

    manifest = {
        "clusters_dir": str(CLUSTERS_DIR),
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "events_in_membership": int(n_events_total),
        "events_written": int(rows_out),
        "cluster_rows_read": int(clusters_read),
        "clusters_not_in_membership": int(unmatched_clusters),
        "date_batch": batch_size,
        "tier": 1,
        "feature_order": ct.CLUSTER_TIER_FEATURES,
        "nan_fraction": summary,
        "deferred_features": list(ct.DEFERRED_REASONS),
        "chunks": chunk_files,
    }
    (OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
    print()
    print("Wrote", OUT_DIR / "manifest_tier1.json")

    print()
    print("NEXT")
    print("  1. Read the NaN table. A high NaN share on a trend feature usually")
    print("     means single-cluster events, which is correct. A high NaN share on")
    print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
    print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
    print("  3. Join tier1 + industrial -> 52 features, then label and train.")
    print("  4. The 20 deferred features are NOT here by accident -- see")
    print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print(f"MODE: {MODE}")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = False         # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")

# ===========================================================================
def run_full_pass():
    """Run the 18 industrial features over every event."""
    # 5. COMPUTE  (only reached once INSPECT_ONLY = False)
    # ===========================================================================

    import shapely
    from shapely import STRtree
    from shapely.geometry import LineString, MultiLineString

    print()
    print("=" * 78)
    print("P3a STEP 3 -- PROJECTION AND TREES")
    print("=" * 78)

    from pyproj import Transformer

    to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


    def project_points(lon, lat):
        """Vectorised lon/lat -> Equal Earth metres."""
        x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                                   np.asarray(lat, dtype="float64"))
        x = np.asarray(x, dtype="float64")
        y = np.asarray(y, dtype="float64")
        # Infinite transform results mean the point fell outside the projection.
        bad = ~np.isfinite(x) | ~np.isfinite(y)
        if bad.any():
            print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
            x[bad] = np.nan
            y[bad] = np.nan
        return x, y


    # --- per-category point trees ----------------------------------------------
    category_trees: dict = {}
    for cat in ind.CATEGORIES:
        pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
        if pts:
            arr = np.array(pts, dtype="float64")
            catx, caty = project_points(arr[:, 0], arr[:, 1])
            good = np.isfinite(catx) & np.isfinite(caty)
            category_trees[cat] = np.column_stack([catx[good], caty[good]])
        else:
            category_trees[cat] = None
        n = 0 if category_trees[cat] is None else len(category_trees[cat])
        print(f"  {cat:14s} point tree: {n:>8,d}")

    # --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
    boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
    boundary_xy = None
    if boundary_chunks:
        raw = np.concatenate(boundary_chunks, axis=0)
        bx, by = project_points(raw[:, 0], raw[:, 1])
        good = np.isfinite(bx) & np.isfinite(by)
        boundary_xy = np.column_stack([bx[good], by[good]])
        print(f"  boundary samples: {len(boundary_xy):,d} "
              f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
    else:
        print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

    # --- polygons for INSIDE_FACILITY -------------------------------------------
    # The original geometries are reused here, not the densified ring samples: the
    # point-in-polygon answer must come from valid closed rings.
    polys = [rec["geom"] for rec in records if rec["geom"] is not None]
    polys = [p for p in polys if p.is_valid and not p.is_empty]
    inside_tree = STRtree(polys) if polys else None
    print(f"  STRtree polygons: {len(polys)}")
    if not polys:
        print("  no usable polygons -> INSIDE_FACILITY will be NaN")

    print()
    print("=" * 78)
    print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
    print("=" * 78)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    read_cols = [X_COL, Y_COL]
    read_cols += list(present_flare.values())
    read_cols += [c for c in ("event_id",) if c in event_columns]
    read_cols = list(dict.fromkeys(read_cols))
    print("reading columns:", read_cols)

    need_cols = set(read_cols)
    null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
    rows_written = 0
    chunk_files = []
    events_seen = 0

    batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
    for batch_no, batch in enumerate(batch_iter):
        frame = batch.to_pandas()
        if frame.empty:
            continue
        if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
            break

        col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
        col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
        if COORD_KIND == "projected metres":
            # Already Equal Earth metres. Projecting again would rescale every
            # distance, so this branch deliberately does nothing.
            x, y = col_x, col_y
        else:
            x, y = project_points(col_x, col_y)
        pts = np.column_stack([x, y])
        finite = np.isfinite(x) & np.isfinite(y)
        if not finite.any():
            continue
        if not finite.all():
            # Unprojectable centroids still need their 18 keys emitted as NaN.
            pts = np.where(finite[:, None], pts, np.nan)

        inside = None
        if inside_tree is not None:
            geoms = shapely.points(x, y)
            hits = inside_tree.query(geoms, predicate="intersects")
            inside = np.zeros(len(pts), dtype="float64")
            inside[np.unique(hits[0])] = 1.0
            del geoms

        boundary = None
        if boundary_xy is not None:
            boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
            full = np.full(len(pts), np.nan)
            full[finite] = boundary
            boundary = full

        flare_site = None
        if inside_tree is not None:
            # A centroid inside any polygon is treated as flare-capable only when
            # the polygon belongs to a flare-capable category; the STRtree here is
            # over all polygons, so fall back to the union distance plus the
            # category trees for the per-category test.
            flare_site = np.zeros(len(pts), dtype=bool)
            flare_pts = []
            for cat in ind.FLARE_SITE_CATEGORIES:
                tree = category_trees.get(cat)
                if tree is None or len(tree) == 0:
                    continue
                d = ind.nearest_distance(pts[finite], tree)
                sub = np.zeros(int(finite.sum()), dtype=bool)
                sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
                flare_pts.append(sub)
            if flare_pts:
                flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
            else:
                flare_site = None

        support = {}
        for key, col in present_flare.items():
            support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

        features = ind.compute_industrial_features(
            pts,
            category_trees,
            crs=TARGET_CRS,
            inside_flags=inside,
            boundary_distance_m=boundary,
            category_is_flare_site=flare_site,
            **support,
        )

        out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
        if "event_id" in frame.columns:
            out.insert(0, "event_id", frame["event_id"].to_numpy())

        for name in ind.INDUSTRIAL_FEATURES:
            null_counts[name] += int(out[name].isna().sum())

        path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path, index=False, compression="snappy")
        chunk_files.append({"file": path.name, "rows": len(out)})
        rows_written += len(out)
        events_seen += len(out)

        if batch_no % 5 == 0:
            print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
                  f"total={rows_written:>10,d}")

    print()
    print(f"rows written: {rows_written:,}")

    # ---------------------------------------------------------------------------
    # 6. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 78)
    print("P3a QC")
    print("=" * 78)
    print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
    summary = {}
    for name in ind.INDUSTRIAL_FEATURES:
        nulls = null_counts[name]
        frac = 100.0 * nulls / rows_written if rows_written else float("nan")
        summary[name] = {"nan_fraction": frac, "nan_count": nulls}
        print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "event_base": str(EVENT_BASE),
        "events_total_in_table": event_pf.metadata.num_rows,
        "rows_written": rows_written,
        "target_crs": TARGET_CRS,
        "chunk_rows": CHUNK_ROWS,
        "max_events": MAX_EVENTS,
        "boundary_densify_m": BOUNDARY_DENSIFY_M,
        "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
        "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
        "feature_order": ind.INDUSTRIAL_FEATURES,
        "null_fraction": summary,
        "chunks": chunk_files,
    }
    with open(OUT_DIR / "manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", OUT_DIR / "manifest.json")

    print()
    print("Status to record in feature_ledger.py once you have inspected the")
    print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
    print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
    print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
    print("runtime_verified stays False until P2/P4 actually execute them.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = False
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print(f"MODE: {MODE}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

def run_full_pass():
    """Execute the whole pass. Called only when INSPECT_ONLY is False."""
    print()
    print("=" * 74)
    print("P4a TIER 1 -- FULL PASS  (writing to features/lineage/)")
    print("=" * 74)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    batch_size = max(1, int(DATE_BATCH))

    null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
    rows_out = 0
    chunk_files = []
    events_seen = set()
    unmatched_clusters = 0
    clusters_read = 0

    for batch_no, start in enumerate(range(0, len(ordered_dates), batch_size)):
        window = ordered_dates[start:start + batch_size]

        tables = []
        for day in window:
            path = CLUSTER_INDEX[day]
            if not cu.path_ok(path):
                print(f"  !! unreadable {path.name}")
                continue
            tables.append(pq.read_table(path))
        if not tables:
            continue
        det = pa.concat_tables(tables).to_pandas()
        del tables

        clusters_read += len(det)
        det[sample_cluster_col] = det[sample_cluster_col].astype(str)

        # Keep only clusters the membership table knows about, and say how many
        # were dropped: an unexpected number here means the join keys disagree.
        known = det[sample_cluster_col].isin(member_index.index)
        n_unknown = int((~known).sum())
        if n_unknown:
            unmatched_clusters += n_unknown
            if batch_no == 0:
                print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                      f"in membership (expected only if this is the first day)")
        det = det[known]
        if det.empty:
            continue

        # The membership table carries its own `date`, which collides with the
        # cluster file's `date` and makes the join raise outright. Bring across
        # only what is actually needed.
        bring = [MEM_EVENT_COL]
        if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
            bring.append(MEM_DATE_COL)
        elif MEM_DATE_COL:
            # Same name on both sides: keep the cluster file's own value and skip
            # the duplicate rather than failing on an overlapping column.
            print("  note: membership date column shares a name with the cluster "
                  "file; using the cluster file's value")

        joined = det.join(
            member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
        )
        if joined.empty:
            print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
            break

        computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
        # One row per event: an event spans many daily batches, so later batches
        # would otherwise re-emit events already written.
        fresh = computed[~computed.index.isin(events_seen)]
        if fresh.empty:
            continue
        events_seen.update(fresh.index.tolist())

        for name in ct.CLUSTER_TIER_FEATURES:
            null_counts[name] += int(fresh[name].isna().sum())

        out = fresh.reset_index()
        path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path_out, index=False, compression="snappy")
        chunk_files.append({"file": path_out.name, "rows": len(out)})
        rows_out += len(out)

        if batch_no % 5 == 0:
            print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
                  f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

    # ---------------------------------------------------------------------------
    # 7. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 74)
    print("P4a TIER 1 -- QC")
    print("=" * 74)
    print(f"cluster rows read      : {clusters_read:,}")
    print(f"clusters not in member : {unmatched_clusters:,}")
    print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
          f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

    assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
        ct.CLUSTER_TIER_FEATURES
    print(f"feature columns        : {len(features_cols)}")

    missing = n_events_total - rows_out
    if missing > 0:
        print()
        print(f"  NOTE: {missing:,} membership events produced no features. That is")
        print("  expected only if those events have no clusters on disk. Events with")
        print("  a single cluster still produce FRP/count/duration values; only the")
        print("  trend and shape features are NaN for them.")

    print()
    print(f"{'feature':32s} {'NaN %':>8s}")
    summary = {}
    for name in ct.CLUSTER_TIER_FEATURES:
        frac = null_counts[name] / rows_out if rows_out else float("nan")
        summary[name] = frac
        flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
        print(f"  {name:30s} {frac:>7.2%}{flag}")

    manifest = {
        "clusters_dir": str(CLUSTERS_DIR),
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "events_in_membership": int(n_events_total),
        "events_written": int(rows_out),
        "cluster_rows_read": int(clusters_read),
        "clusters_not_in_membership": int(unmatched_clusters),
        "date_batch": batch_size,
        "tier": 1,
        "feature_order": ct.CLUSTER_TIER_FEATURES,
        "nan_fraction": summary,
        "deferred_features": list(ct.DEFERRED_REASONS),
        "chunks": chunk_files,
    }
    (OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
    print()
    print("Wrote", OUT_DIR / "manifest_tier1.json")

    print()
    print("NEXT")
    print("  1. Read the NaN table. A high NaN share on a trend feature usually")
    print("     means single-cluster events, which is correct. A high NaN share on")
    print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
    print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
    print("  3. Join tier1 + industrial -> 52 features, then label and train.")
    print("  4. The 20 deferred features are NOT here by accident -- see")
    print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print(f"MODE: {MODE}")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = False         # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")

# ===========================================================================
def run_full_pass():
    """Run the 18 industrial features over every event."""
    # 5. COMPUTE  (only reached once INSPECT_ONLY = False)
    # ===========================================================================

    import shapely
    from shapely import STRtree
    from shapely.geometry import LineString, MultiLineString

    print()
    print("=" * 78)
    print("P3a STEP 3 -- PROJECTION AND TREES")
    print("=" * 78)

    from pyproj import Transformer

    to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


    def project_points(lon, lat):
        """Vectorised lon/lat -> Equal Earth metres."""
        x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                                   np.asarray(lat, dtype="float64"))
        x = np.asarray(x, dtype="float64")
        y = np.asarray(y, dtype="float64")
        # Infinite transform results mean the point fell outside the projection.
        bad = ~np.isfinite(x) | ~np.isfinite(y)
        if bad.any():
            print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
            x[bad] = np.nan
            y[bad] = np.nan
        return x, y


    # --- per-category point trees ----------------------------------------------
    category_trees: dict = {}
    for cat in ind.CATEGORIES:
        pts = [(r["x"], r["y"]) for r in records if r["category"] == cat and r["boundary"] is None]
        if pts:
            arr = np.array(pts, dtype="float64")
            catx, caty = project_points(arr[:, 0], arr[:, 1])
            good = np.isfinite(catx) & np.isfinite(caty)
            category_trees[cat] = np.column_stack([catx[good], caty[good]])
        else:
            category_trees[cat] = None
        n = 0 if category_trees[cat] is None else len(category_trees[cat])
        print(f"  {cat:14s} point tree: {n:>8,d}")

    # --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
    boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
    boundary_xy = None
    if boundary_chunks:
        raw = np.concatenate(boundary_chunks, axis=0)
        bx, by = project_points(raw[:, 0], raw[:, 1])
        good = np.isfinite(bx) & np.isfinite(by)
        boundary_xy = np.column_stack([bx[good], by[good]])
        print(f"  boundary samples: {len(boundary_xy):,d} "
              f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
    else:
        print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

    # --- polygons for INSIDE_FACILITY -------------------------------------------
    # The original geometries are reused here, not the densified ring samples: the
    # point-in-polygon answer must come from valid closed rings.
    polys = [rec["geom"] for rec in records if rec["geom"] is not None]
    polys = [p for p in polys if p.is_valid and not p.is_empty]
    inside_tree = STRtree(polys) if polys else None
    print(f"  STRtree polygons: {len(polys)}")
    if not polys:
        print("  no usable polygons -> INSIDE_FACILITY will be NaN")

    print()
    print("=" * 78)
    print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
    print("=" * 78)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    read_cols = [X_COL, Y_COL]
    read_cols += list(present_flare.values())
    read_cols += [c for c in ("event_id",) if c in event_columns]
    read_cols = list(dict.fromkeys(read_cols))
    print("reading columns:", read_cols)

    need_cols = set(read_cols)
    null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
    rows_written = 0
    chunk_files = []
    events_seen = 0

    batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
    for batch_no, batch in enumerate(batch_iter):
        frame = batch.to_pandas()
        if frame.empty:
            continue
        if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
            break

        col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
        col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
        if COORD_KIND == "projected metres":
            # Already Equal Earth metres. Projecting again would rescale every
            # distance, so this branch deliberately does nothing.
            x, y = col_x, col_y
        else:
            x, y = project_points(col_x, col_y)
        pts = np.column_stack([x, y])
        finite = np.isfinite(x) & np.isfinite(y)
        if not finite.any():
            continue
        if not finite.all():
            # Unprojectable centroids still need their 18 keys emitted as NaN.
            pts = np.where(finite[:, None], pts, np.nan)

        inside = None
        if inside_tree is not None:
            geoms = shapely.points(x, y)
            hits = inside_tree.query(geoms, predicate="intersects")
            inside = np.zeros(len(pts), dtype="float64")
            inside[np.unique(hits[0])] = 1.0
            del geoms

        boundary = None
        if boundary_xy is not None:
            boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
            full = np.full(len(pts), np.nan)
            full[finite] = boundary
            boundary = full

        flare_site = None
        if inside_tree is not None:
            # A centroid inside any polygon is treated as flare-capable only when
            # the polygon belongs to a flare-capable category; the STRtree here is
            # over all polygons, so fall back to the union distance plus the
            # category trees for the per-category test.
            flare_site = np.zeros(len(pts), dtype=bool)
            flare_pts = []
            for cat in ind.FLARE_SITE_CATEGORIES:
                tree = category_trees.get(cat)
                if tree is None or len(tree) == 0:
                    continue
                d = ind.nearest_distance(pts[finite], tree)
                sub = np.zeros(int(finite.sum()), dtype=bool)
                sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
                flare_pts.append(sub)
            if flare_pts:
                flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
            else:
                flare_site = None

        support = {}
        for key, col in present_flare.items():
            support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

        features = ind.compute_industrial_features(
            pts,
            category_trees,
            crs=TARGET_CRS,
            inside_flags=inside,
            boundary_distance_m=boundary,
            category_is_flare_site=flare_site,
            **support,
        )

        out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
        if "event_id" in frame.columns:
            out.insert(0, "event_id", frame["event_id"].to_numpy())

        for name in ind.INDUSTRIAL_FEATURES:
            null_counts[name] += int(out[name].isna().sum())

        path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path, index=False, compression="snappy")
        chunk_files.append({"file": path.name, "rows": len(out)})
        rows_written += len(out)
        events_seen += len(out)

        if batch_no % 5 == 0:
            print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
                  f"total={rows_written:>10,d}")

    print()
    print(f"rows written: {rows_written:,}")

    # ---------------------------------------------------------------------------
    # 6. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 78)
    print("P3a QC")
    print("=" * 78)
    print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
    summary = {}
    for name in ind.INDUSTRIAL_FEATURES:
        nulls = null_counts[name]
        frac = 100.0 * nulls / rows_written if rows_written else float("nan")
        summary[name] = {"nan_fraction": frac, "nan_count": nulls}
        print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "event_base": str(EVENT_BASE),
        "events_total_in_table": event_pf.metadata.num_rows,
        "rows_written": rows_written,
        "target_crs": TARGET_CRS,
        "chunk_rows": CHUNK_ROWS,
        "max_events": MAX_EVENTS,
        "boundary_densify_m": BOUNDARY_DENSIFY_M,
        "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
        "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
        "feature_order": ind.INDUSTRIAL_FEATURES,
        "null_fraction": summary,
        "chunks": chunk_files,
    }
    with open(OUT_DIR / "manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", OUT_DIR / "manifest.json")

    print()
    print("Status to record in feature_ledger.py once you have inspected the")
    print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
    print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
    print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
    print("runtime_verified stays False until P2/P4 actually execute them.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# P3a -- COLAB CELL: industrial context features for all global fire events
#
# Run this AFTER the P1 cell, and after P1b has confirmed that
#   external/osm_facilities/ is populated (37 files, 0.28 GB -- verified OK).
#
# What it does
#   1. Reads every geoparquet under external/osm_facilities/, normalises the
#      facility taxonomy onto the 10 categories the ledger needs, and prints
#      the resolved mapping so you can confirm it before any compute happens.
#   2. Projects facility coordinates and event centroids to EPSG:6933 metres.
#   3. Streams canonical_event_base.parquet in batches -- never loading all
#      7,265,849 events into memory -- and emits the 18 INDUSTRIAL features
#      per event using cKDTree + STRtree.
#   4. Writes chunked parquet + a QC report.
#
# Cost: no Earth Engine, no API quota. Pure CPU geometry.
#
# THE CELL IS SAFE BY DEFAULT. It stops after step 1 and prints what it found.
# Nothing is written until you set INSPECT_ONLY = False and confirm the printed
# CATEGORY_MAP is right, because a silently wrong category map would poison
# both the model features and the labels.
# ============================================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import feature_ledger as fl
import industrial_features as ind
import colab_utils as cu

print("Ledger version   :", fl.LEDGER_VERSION)
print("Industrial module: OK")
MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print(f"MODE: {MODE}")
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 0. CONFIGURATION
# ---------------------------------------------------------------------------

INSPECT_ONLY = False         # flip to False once CATEGORY_MAP below is verified
CHUNK_ROWS = 200_000         # events per batch; raise if RAM allows
MAX_EVENTS = None            # set an int to smoke-test on a subset, e.g. 50_000
TARGET_CRS = ind.REQUIRED_CRS  # EPSG:6933
BOUNDARY_DENSIFY_M = 50.0    # boundary sampling step for the boundary distance

# ---- Resolve Drive locations ------------------------------------------------
# Everything derived from PROJECT_ROOT is computed HERE, after resolution.
# Deriving paths in the config block above ran before resolution and raised
# `TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'`.
cu.require_drive("the facility gazetteer")

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}.\n"
        "Run colab_survey_tree.py and paste the output."
    )

_CANON = PROJECT_ROOT / "full_2012_2026"
EXTERNAL_ROOT, _ext_table = cu.pick_by_file_count(PROJECT_ROOT, "external")
if EXTERNAL_ROOT is None:
    EXTERNAL_ROOT = _CANON / "external"
EVENTS_DIR, _ev_table = cu.pick_by_file_count(PROJECT_ROOT, "events")
if EVENTS_DIR is None:
    EVENTS_DIR = _CANON / "events"

FACILITY_DIR = EXTERNAL_ROOT / "osm_facilities"
OUT_DIR = _CANON / "features" / "industrial"
QC_DIR = PROJECT_ROOT / "qc"

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("project root", PROJECT_ROOT),
    ("external", EXTERNAL_ROOT),
    ("facilities", FACILITY_DIR),
    ("events", EVENTS_DIR),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "MISSING"
    print(f"  [{flag}] {label:14s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 1. GAZETTEER TAXONOMY
#
# The 37 files are named after OSM/GEM categories, not after the 10 ledger
# categories, and several files overlap (e.g. `industrial.geoparquet` almost
# certainly contains the members of `cement`, `steel`, `lng`, ... as well).
# Mapping them by hand here is the single highest-leverage thing in this cell:
# a wrong mapping silently corrupts both the features and the labels.
#
# Each entry is (regex over the filename, target category or None to skip).
# Order matters -- first match wins.
# ---------------------------------------------------------------------------

CATEGORY_MAP: "list[tuple[str, str | None]]" = [
    (r"refiner", "refinery"),
    (r"petrochem|petro_chem", "petrochemical"),
    (r"power[_ ]?plant|powerstation|thermal_power", "powerplant"),
    (r"steel|metal|blast_furnace|coking", "steel"),
    (r"cement|lime|clinker", "cement"),
    (r"(^|_)lng(_|$)|liquef|gas_terminal|gas_process|gas_plant", "lng"),
    (r"mine|mining|quarry|aggregate|open_pit|coal_", "mine"),
    (r"landfill|waste|incinerat|slag", "landfill"),
    (r"pipeline|pipe_?line|petroduct", "pipeline"),
    (r"well[_ ]?pad|wellhead|drill_?site|oil_?pad|onshore", "wellpad"),
    # Supersets: keep only as a fallback when no specific file exists.
    (r"all_industrial|^industrial\.", None),   # skipped: duplicates the above
    (r"facilit|facility", None),               # skipped: ditto
]

# Event centroid columns in canonical_event_base.parquet. Verified names are
# centroid_lon / centroid_lat (see HANDOFF 4, cell 260 event_geometry); the
# cell prints the real schema so this can be corrected in place if they differ.
# Discovered automatically in STEP 2 from the real event-table schema; these
# are only fallbacks if discovery fails.
EVENT_LON_COL = "event_centroid_lon"
EVENT_LAT_COL = "event_centroid_lat"
X_COL = None
Y_COL = None
COORD_KIND = None

# Optional event columns feeding KNOWN_FLARE. Missing ones degrade KNOWN_FLARE
# to NaN rather than to 0, so the absence is visible in the QC report.
FLARE_SUPPORT_COLUMNS = {
    "active_days": "active_days",
    "detection_count": "detection_count",
    "centroid_displacement_m": "centroid_displacement_m",
    "frp_cv": "frp_cv",
}

# ---------------------------------------------------------------------------
# 2. INSPECT THE GAZETTEER  (always runs)
# ---------------------------------------------------------------------------


def resolve_category(stem: str):
    """Map a geoparquet stem onto one of the 10 ledger categories, or None."""
    name = stem.lower()
    for pattern, category in CATEGORY_MAP:
        if re.search(pattern, name):
            return category
    return None


def _dir_penalty(path):
    """Ranking key for a directory name: unsuffixed wins over " (n)" copies.

    The gazetteer is duplicated across directories (`facility_categories/` and
    a macOS-style `facility_categories (1)/`), so the same filename appears
    twice with different row counts. Loading both double-counts every facility
    in the duplicated folder, which biases the FACILITY_COUNT_* features and
    the DIST_* trees.
    """
    penalty = 0
    for part in path.relative_to(FACILITY_DIR).parts[:-1]:
        m = re.search(r"\s*\((\d+)\)$", part)
        if m:
            penalty += 1_000 + int(m.group(1))
    return penalty


def list_facility_files():
    if not cu.path_ok(FACILITY_DIR):
        raise FileNotFoundError(
            f"{FACILITY_DIR} not found. Run colab_p1b_external_inventory.py "
            "first -- osm_facilities must be confirmed present before P3a."
        )
    files = sorted(
        [p for p in FACILITY_DIR.rglob("*.geoparquet")]
        + [p for p in FACILITY_DIR.rglob("*.parquet") if "geoparquet" not in p.suffix]
    )
    if not files:
        raise FileNotFoundError(f"No geoparquet files under {FACILITY_DIR}")

    # Collapse same-named duplicates: keep the one in the least-suffixed
    # directory, and report every drop so it is visible rather than silent.
    best = {}
    for path in files:
        key = path.name
        rank = (_dir_penalty(path), str(path))
        if key not in best or rank < best[key][0]:
            best[key] = (rank, path)

    kept = sorted((v[1] for v in best.values()), key=lambda p: p.name)
    dropped = [p for p in files if p not in set(kept)]

    print()
    print("gazetteer duplicate handling")
    print("-" * 74)
    print(f"  files found   : {len(files)}")
    print(f"  unique names  : {len(kept)}")
    print(f"  dropped as dup: {len(dropped)}")
    if dropped:
        print("  dropped copies (a directory was duplicated on Drive):")
        for path in sorted(dropped, key=lambda p: str(p))[:12]:
            kept_twin = next(
                (q for q in kept if q.name == path.name), None
            )
            rel = path.relative_to(FACILITY_DIR)
            note = ""
            if kept_twin is not None:
                note = f"  kept {kept_twin.relative_to(FACILITY_DIR)}"
            print(f"    {rel}{note}")
    print("-" * 74)
    return kept


def read_gazetteer():
    """Read every facility file, keeping points, polygons and category labels.

    Returns (records, per-file report). `records` entries are
    {"category", "x", "y", "boundary": [np.ndarray | None]}.
    """
    files = list_facility_files()
    print("=" * 78)
    print("P3a STEP 1 -- GAZETTEER INSPECTION")
    print("=" * 78)
    print(f"files: {len(files)}")

    import shapely
    from shapely import wkb as shapely_wkb

    records = []
    report = []

    for path in files:
        category = resolve_category(path.stem)
        try:
            table = pq.read_table(path)
        except Exception as exc:                       # noqa: BLE001
            report.append({"file": path.name, "category": category, "error": str(exc)})
            print(f"  SKIP {path.name}: {exc}")
            continue

        cols = {c.lower(): c for c in table.column_names}
        geom_col = next(
            (cols[k] for k in ("geometry", "geom", "wkb_geometry") if k in cols), None
        )
        if geom_col is None:
            report.append(
                {"file": path.name, "category": category, "error": "no geometry column"}
            )
            print(f"  SKIP {path.name}: columns {table.column_names}")
            continue

        geoms = shapely.from_wkb(np.asarray(table.column(geom_col).to_numpy(zero_copy_only=False)))
        geoms = np.asarray(geoms, dtype="object")

        # Category label column, used to refine the filename mapping when the
        # file is a superset such as `industrial`.
        label_col = next(
            (cols[k] for k in ("category", "facility_type", "type", "class", "tags")
             if k in cols),
            None,
        )
        labels = (
            table.column(label_col).to_numpy(zero_copy_only=False)
            if label_col
            else None
        )

        point_count = 0
        polygon_count = 0
        unresolved: dict = {}

        for i, geom in enumerate(geoms):
            if geom is None or getattr(geom, "is_empty", True):
                continue
            cat = category
            if cat is None and labels is not None:
                cat = refine_category(labels[i])
            if cat is None:
                # Unmapped and unlabelled: keep the coordinates out of every
                # category tree rather than guessing. Counted for the report.
                key = "UNMAPPED"
                unresolved[key] = unresolved.get(key, 0) + 1
                continue

            if geom.geom_type == "Point":
                records.append(
                    {
                        "category": cat,
                        "x": geom.x,
                        "y": geom.y,
                        "boundary": None,
                        "geom": None,
                        # Representative point for the distance trees. For a
                        # Point this is the point itself.
                        "rep_x": geom.x,
                        "rep_y": geom.y,
                        "is_point": True,
                    }
                )
                point_count += 1
            else:
                rings = densify_boundary(geom)
                if rings is not None:
                    # A polygon still has to appear in the DIST_* trees, or it
                    # is invisible to the model. Its centroid is the
                    # representative point. Without this, `mine` contributed
                    # 425 points while 10,886 polygons were dropped, and
                    # `petrochemical` had no points at all -- making
                    # DIST_PETROCHEMICAL_LOG 100% NaN and DIST_MINE wrong for
                    # every event near a polygonal mine.
                    try:
                        centroid = geom.centroid
                        rep_x, rep_y = float(centroid.x), float(centroid.y)
                        if not np.isfinite(rep_x) or not np.isfinite(rep_y):
                            rep_x = rep_y = None
                    except Exception:              # noqa: BLE001
                        rep_x = rep_y = None
                    records.append(
                        {
                            "category": cat,
                            "x": None,
                            "y": None,
                            "boundary": rings,
                            # The original geometry, kept because a point-in-
                            # polygon test needs real rings. Rebuilding a polygon
                            # from the densified open path would be invalid.
                            "geom": geom,
                            "rep_x": rep_x,
                            "rep_y": rep_y,
                            "is_point": False,
                        }
                    )
                    polygon_count += 1

        report.append(
            {
                "file": path.name,
                "category": category,
                "rows": len(geoms),
                "points": point_count,
                "polygons": polygon_count,
                "unmapped": unresolved,
                "columns": list(table.column_names),
            }
        )
        flag = "skip" if category is None else f"-> {category}"
        print(
            f"  {path.name:52s} rows={len(geoms):>7d} "
            f"pts={point_count:>7d} poly={polygon_count:>6d}  {flag}"
        )

    return records, report


def refine_category(raw_label) -> "str | None":
    """Map an in-file category label onto the 10 ledger categories."""
    if raw_label is None:
        return None
    text = raw_label.decode() if isinstance(raw_label, bytes) else str(raw_label)
    return resolve_category(text)


def densify_boundary(geom, step_m=BOUNDARY_DENSIFY_M):
    """Sample a polygon's rings at `step_m` so a boundary distance can be
    approximated with a cKDTree.

    Exact for points and for polygons far from any vertex; inside the polygon
    the distance is 0 regardless, so the sampling only affects the outside
    case, where the error is bounded by step_m / 2. Documented approximation,
    not an exact boundary-to-boundary distance.
    """
    import shapely
    from shapely.geometry import LineString, MultiLineString, MultiPolygon

    parts = [geom] if geom.geom_type != "MultiPolygon" else list(geom.geoms)
    lines = []
    for part in parts:
        if not hasattr(part, "exterior"):
            continue
        rings = [part.exterior] + list(getattr(part, "interiors", []))
        for ring in rings:
            ring_coords = np.asarray(ring.coords, dtype="float64")[:-1]
            if len(ring_coords) >= 2:
                lines.append(LineString(ring_coords))
    if not lines:
        return None
    line = lines[0] if len(lines) == 1 else MultiLineString(lines)
    try:
        sampled = shapely.segmentize(line, max_segment_length=float(step_m))
    except AttributeError:                             # older shapely
        sampled = line
    coords = np.asarray(shapely.get_coordinates(sampled), dtype="float64")
    return coords if len(coords) else None


records, gazetteer_report = read_gazetteer()

print()
print("-" * 78)
print("RESOLVED CATEGORY MAP -- CONFIRM THIS BEFORE COMPUTING")
print("-" * 78)
per_category_points: dict = {}
per_category_polys: dict = {}
for rec in records:
    bucket_points = per_category_points.setdefault(rec["category"], 0)
    bucket_polys = per_category_polys.setdefault(rec["category"], 0)
    if rec["boundary"] is None:
        per_category_points[rec["category"]] = bucket_points + 1
    else:
        per_category_polys[rec["category"]] = bucket_polys + 1

for cat in ind.CATEGORIES:
    np_pts = per_category_points.get(cat, 0)
    np_poly = per_category_polys.get(cat, 0)
    status = "ok" if (np_pts or np_poly) else "MISSING -> feature will be NaN"
    print(
        f"  {cat:14s} points={np_pts:>8d} polygons={np_poly:>6d}   {status}"
    )
stray = per_category_points.get("UNMAPPED", 0) + per_category_polys.get("UNMAPPED", 0)
print(f"  unmapped records dropped: {stray}")
print()
print("Categories that end up MISSING are not an error: the corresponding")
print("DIST_* feature is reported as NaN, which is distinguishable from a")
print("measured zero. Add a mapping above if a category should have been found.")

# ---------------------------------------------------------------------------
# 3. INSPECT THE EVENT TABLE
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("P3a STEP 2 -- EVENT TABLE SCHEMA")
print("=" * 78)

EVENT_BASE = next(
    (
        p
        for p in (
            EVENTS_DIR / "canonical_event_base.parquet",
            EVENTS_DIR / "events.parquet",
            EVENTS_DIR / "global_event_base.parquet",
        )
        if cu.path_ok(p)
    ),
    None,
)
if EVENT_BASE is None:
    print()
    print("parquet files under", EVENTS_DIR)
    for p_ in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p_.name)
    raise FileNotFoundError(
        "No canonical event base found under "
        f"{EVENTS_DIR}. It is a frozen artefact: restore it from Drive, never "
        "rebuild it."
    )

event_pf = pq.ParquetFile(EVENT_BASE)
event_columns = event_pf.schema_arrow.names
print(f"rows      : {event_pf.metadata.num_rows:,}")
print(f"row groups: {event_pf.metadata.num_row_groups:,}")
print(f"columns   : {event_columns}")

# --- automatic coordinate discovery ---------------------------------------
# The handoff names `centroid_lon`/`centroid_lat`, but the real table uses
# `event_centroid_lon`/`event_centroid_lat` AND carries pre-projected
# `event_centroid_x`/`event_centroid_y`. Preferring the projected pair removes
# a projection step entirely and eliminates a class of unit error, so it wins
# when both are present -- but only after the magnitudes confirm metres.
_printable = {c.lower(): c for c in event_columns}
_pairs = (
    ("event_centroid_x", "event_centroid_y"),
    ("centroid_x", "centroid_y"),
    ("event_centroid_lon", "event_centroid_lat"),
    ("centroid_lon", "centroid_lat"),
    ("lon", "lat"),
    ("longitude", "latitude"),
)


def _looks_like_metres(df, col):
    """True when `col` holds projected metres rather than degrees."""
    try:
        head = pd.to_numeric(
            df[col].head(2000), errors="coerce"
        ).abs().max()
    except Exception:                                # noqa: BLE001
        return False
    if not np.isfinite(head):
        return False
    # Degrees never exceed 180; Equal Earth metres are in the millions.
    return head > 1000.0


_probe_cols = [c for pair in _pairs for c in pair if c in event_columns]
_probe = (
    pd.read_parquet(EVENT_BASE, columns=_probe_cols).head(2000)
    if _probe_cols
    else pd.DataFrame()
)
COORD_SOURCE = None
for cx, cy in _pairs:
    if cx in event_columns and cy in event_columns:
        if _looks_like_metres(_probe, cx) and _looks_like_metres(_probe, cy):
            X_COL, Y_COL, COORD_KIND = cx, cy, "projected metres"
            COORD_SOURCE = f"{cx}/{cy} (already in {TARGET_CRS})"
            break
        EVENT_LON_COL, EVENT_LAT_COL = cx, cy
        COORD_KIND = "lon/lat degrees -> must project"
        COORD_SOURCE = f"{cx}/{cy}"
        break

if COORD_SOURCE is None:
    print()
    print("!! No coordinate pair found in the event table.")
    print(f"   columns: {event_columns}")
    print("   Set X_COL / Y_COL by hand and re-run. Do not guess: a wrong")
    print("   centroid silently shifts every industrial feature.")
    raise KeyError("no coordinate columns")

print()
print(f"coordinate columns : {COORD_SOURCE}")
print(f"coordinate kind    : {COORD_KIND}")
if COORD_KIND != "projected metres":
    print("  -> these are degrees; the cell will project to "
          f"{TARGET_CRS} before any distance maths")

# --- pre-existing feature columns -----------------------------------------
# The event base already carries cell-325-style aggregates. Note them so they
# can be compared against the P4a lineage output rather than recomputed blind.
PREEXISTING = [
    c
    for c in event_columns
    if c.startswith("geometry_")
    or c in ("n_detections", "total_frp", "max_frp", "duration_hours",
             "n_local_clusters")
]
if PREEXISTING:
    print()
    print(f"PRE-EXISTING aggregate columns ({len(PREEXISTING)}): "
          f"{PREEXISTING}")
    print("  These look like cell-325 outputs. The P4a lineage pass computes")
    print("  an independent version of several of them; the two should be")
    print("  compared, not assumed equal.")

# Columns that appear twice with a _1 suffix indicate a bad merge at some
# point. Harmless here (only the un-suffixed names are read) but worth naming.
_suffixed = [c for c in event_columns if re.fullmatch(r".+_1", c)]
if _suffixed:
    print()
    print(f"WARNING: {len(_suffixed)} column(s) look like merge duplicates: "
          f"{_suffixed[:8]}")
    print("  This cell reads only the un-suffixed originals, so no action is")
    print("  needed here -- but a previous merge left debris in the table.")

present_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v in event_columns}
missing_flare = {k: v for k, v in FLARE_SUPPORT_COLUMNS.items() if v not in event_columns}
print()
print("KNOWN_FLARE support columns present:", present_flare or "none")
print("KNOWN_FLARE support columns missing:", missing_flare or "none")
if missing_flare:
    print(
        "  KNOWN_FLARE will be NaN for every event. That is the honest result: "
        "it needs an all-events pass (P3g) or the cell-325 outputs."
    )

# ---------------------------------------------------------------------------
# 4. STOP HERE UNLESS CONFIRMED
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    print()
    print("=" * 78)
    print("INSPECTION COMPLETE -- NOTHING WRITTEN")
    print("=" * 78)
    print("1. Check the CATEGORY MAP above. Any MISSING category that should")
    print("   exist? Add a regex to CATEGORY_MAP and re-run this cell.")
    print("2. If the centroid columns were wrong, fix EVENT_LON_COL/ _LAT_COL.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False
    print()
    print("QC_DIR.mkdir(parents=True, exist_ok=True)")
    with open(QC_DIR / "p3a_gazetteer_report.json", "w") as fh:
        json.dump(
            {
                "files": gazetteer_report,
                "points_per_category": per_category_points,
                "polygons_per_category": per_category_polys,
                "event_columns": event_columns,
                "flare_support_present": list(present_flare),
                "flare_support_missing": list(missing_flare),
            },
            fh,
            indent=2,
        )
    print("Wrote", QC_DIR / "p3a_gazetteer_report.json")

# ===========================================================================
def run_full_pass():
    """Run the 18 industrial features over every event."""
    # 5. COMPUTE  (only reached once INSPECT_ONLY = False)
    # ===========================================================================

    import shapely
    from shapely import STRtree
    from shapely.geometry import LineString, MultiLineString

    print()
    print("=" * 78)
    print("P3a STEP 3 -- PROJECTION AND TREES")
    print("=" * 78)

    from pyproj import Transformer

    to_target = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)


    def project_points(lon, lat):
        """Vectorised lon/lat -> Equal Earth metres."""
        x, y = to_target.transform(np.asarray(lon, dtype="float64"),
                                   np.asarray(lat, dtype="float64"))
        x = np.asarray(x, dtype="float64")
        y = np.asarray(y, dtype="float64")
        # Infinite transform results mean the point fell outside the projection.
        bad = ~np.isfinite(x) | ~np.isfinite(y)
        if bad.any():
            print(f"  !! {int(bad.sum())} event centroids failed to project -> NaN")
            x[bad] = np.nan
            y[bad] = np.nan
        return x, y


    # --- per-category point trees ----------------------------------------------
    category_trees: dict = {}
    for cat in ind.CATEGORIES:
        # Every facility with a usable representative point, polygons included.
        # Filtering on `boundary is None` here would silently drop every polygon
        # from the distance features.
        pts = [
            (r["rep_x"], r["rep_y"])
            for r in records
            if r["category"] == cat
            and r.get("rep_x") is not None
            and r.get("rep_y") is not None
        ]
        if pts:
            arr = np.array(pts, dtype="float64")
            catx, caty = project_points(arr[:, 0], arr[:, 1])
            good = np.isfinite(catx) & np.isfinite(caty)
            category_trees[cat] = np.column_stack([catx[good], caty[good]])
        else:
            category_trees[cat] = None
        n = 0 if category_trees[cat] is None else len(category_trees[cat])
        print(f"  {cat:14s} point tree: {n:>8,d}")

    # --- boundary point cloud for DIST_FACILITY_BOUNDARY_LOG ---------------------
    boundary_chunks = [r["boundary"] for r in records if r["boundary"] is not None]
    boundary_xy = None
    if boundary_chunks:
        raw = np.concatenate(boundary_chunks, axis=0)
        bx, by = project_points(raw[:, 0], raw[:, 1])
        good = np.isfinite(bx) & np.isfinite(by)
        boundary_xy = np.column_stack([bx[good], by[good]])
        print(f"  boundary samples: {len(boundary_xy):,d} "
              f"(step {BOUNDARY_DENSIFY_M:.0f} m, error <= {BOUNDARY_DENSIFY_M / 2:.0f} m outside)")
    else:
        print("  no polygon boundaries -> DIST_FACILITY_BOUNDARY_LOG will be NaN")

    # --- polygons for INSIDE_FACILITY -------------------------------------------
    # The original geometries are reused here, not the densified ring samples: the
    # point-in-polygon answer must come from valid closed rings.
    polys = [rec["geom"] for rec in records if rec["geom"] is not None]
    polys = [p for p in polys if p.is_valid and not p.is_empty]
    inside_tree = STRtree(polys) if polys else None
    print(f"  STRtree polygons: {len(polys)}")
    if not polys:
        print("  no usable polygons -> INSIDE_FACILITY will be NaN")

    print()
    print("=" * 78)
    print("P3a STEP 4 -- STREAMING THE 7,265,849 EVENTS")
    print("=" * 78)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    read_cols = [X_COL, Y_COL]
    read_cols += list(present_flare.values())
    read_cols += [c for c in ("event_id",) if c in event_columns]
    read_cols = list(dict.fromkeys(read_cols))
    print("reading columns:", read_cols)

    need_cols = set(read_cols)
    null_counts = {name: 0 for name in ind.INDUSTRIAL_FEATURES}
    rows_written = 0
    chunk_files = []
    events_seen = 0

    batch_iter = event_pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols)
    for batch_no, batch in enumerate(batch_iter):
        frame = batch.to_pandas()
        if frame.empty:
            continue
        if MAX_EVENTS is not None and events_seen >= MAX_EVENTS:
            break

        col_x = pd.to_numeric(frame[X_COL], errors="coerce").to_numpy("float64")
        col_y = pd.to_numeric(frame[Y_COL], errors="coerce").to_numpy("float64")
        if COORD_KIND == "projected metres":
            # Already Equal Earth metres. Projecting again would rescale every
            # distance, so this branch deliberately does nothing.
            x, y = col_x, col_y
        else:
            x, y = project_points(col_x, col_y)
        pts = np.column_stack([x, y])
        finite = np.isfinite(x) & np.isfinite(y)
        if not finite.any():
            continue
        if not finite.all():
            # Unprojectable centroids still need their 18 keys emitted as NaN.
            pts = np.where(finite[:, None], pts, np.nan)

        inside = None
        if inside_tree is not None:
            geoms = shapely.points(x, y)
            hits = inside_tree.query(geoms, predicate="intersects")
            inside = np.zeros(len(pts), dtype="float64")
            inside[np.unique(hits[0])] = 1.0
            del geoms

        boundary = None
        if boundary_xy is not None:
            boundary = ind.nearest_distance(pts[np.isfinite(x) & np.isfinite(y)], boundary_xy)
            full = np.full(len(pts), np.nan)
            full[finite] = boundary
            boundary = full

        flare_site = None
        if inside_tree is not None:
            # A centroid inside any polygon is treated as flare-capable only when
            # the polygon belongs to a flare-capable category; the STRtree here is
            # over all polygons, so fall back to the union distance plus the
            # category trees for the per-category test.
            flare_site = np.zeros(len(pts), dtype=bool)
            flare_pts = []
            for cat in ind.FLARE_SITE_CATEGORIES:
                tree = category_trees.get(cat)
                if tree is None or len(tree) == 0:
                    continue
                d = ind.nearest_distance(pts[finite], tree)
                sub = np.zeros(int(finite.sum()), dtype=bool)
                sub[np.nan_to_num(d, nan=np.inf) <= 10_000.0] = True
                flare_pts.append(sub)
            if flare_pts:
                flare_site[finite] = np.any(np.vstack(flare_pts), axis=0)
            else:
                flare_site = None

        support = {}
        for key, col in present_flare.items():
            support[key] = pd.to_numeric(frame[col], errors="coerce").to_numpy("float64")

        features = ind.compute_industrial_features(
            pts,
            category_trees,
            crs=TARGET_CRS,
            inside_flags=inside,
            boundary_distance_m=boundary,
            category_is_flare_site=flare_site,
            **support,
        )

        out = pd.DataFrame(features, columns=ind.INDUSTRIAL_FEATURES)
        if "event_id" in frame.columns:
            out.insert(0, "event_id", frame["event_id"].to_numpy())

        for name in ind.INDUSTRIAL_FEATURES:
            null_counts[name] += int(out[name].isna().sum())

        path = OUT_DIR / f"industrial_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path, index=False, compression="snappy")
        chunk_files.append({"file": path.name, "rows": len(out)})
        rows_written += len(out)
        events_seen += len(out)

        if batch_no % 5 == 0:
            print(f"  chunk {batch_no:>4d}  rows={len(out):>7,d}  "
                  f"total={rows_written:>10,d}")

    print()
    print(f"rows written: {rows_written:,}")

    # ---------------------------------------------------------------------------
    # 6. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 78)
    print("P3a QC")
    print("=" * 78)
    print(f"{'feature':30s} {'NaN %':>8s} {'min':>12s} {'max':>12s}")
    summary = {}
    for name in ind.INDUSTRIAL_FEATURES:
        nulls = null_counts[name]
        frac = 100.0 * nulls / rows_written if rows_written else float("nan")
        summary[name] = {"nan_fraction": frac, "nan_count": nulls}
        print(f"{name:30s} {frac:>7.2f}% {'-':>12s} {'-':>12s}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "event_base": str(EVENT_BASE),
        "events_total_in_table": event_pf.metadata.num_rows,
        "rows_written": rows_written,
        "target_crs": TARGET_CRS,
        "chunk_rows": CHUNK_ROWS,
        "max_events": MAX_EVENTS,
        "boundary_densify_m": BOUNDARY_DENSIFY_M,
        "category_points": {k: 0 if v is None else len(v) for k, v in category_trees.items()},
        "boundary_samples": 0 if boundary_xy is None else len(boundary_xy),
        "feature_order": ind.INDUSTRIAL_FEATURES,
        "null_fraction": summary,
        "chunks": chunk_files,
    }
    with open(OUT_DIR / "manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", OUT_DIR / "manifest.json")

    print()
    print("Status to record in feature_ledger.py once you have inspected the")
    print("output and a 10-event spot check passes: all 18 INDUSTRIAL features")
    print("move from MISSING_IMPLEMENTATION to NEW_IMPLEMENTATION. They are NOT")
    print("COMPUTED_HISTORICAL, because no historical implementation exists, and")
    print("runtime_verified stays False until P2/P4 actually execute them.")


if not _STOP:
    run_full_pass()


In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import training_design as td

INSPECT_ONLY = True
CHUNK_EVENTS = 500_000
SITE_DEG = db.SITE_DEG

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

FEATURE_COLUMNS = list(ct.CLUSTER_TIER_FEATURES) + list(ind.INDUSTRIAL_FEATURES)

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Join, derive, label, and write X / y / meta in chunks."""
    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)

    # --- read the event base once, for coordinates and time only -----------
    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")
    base = pq.ParquetFile(base_path)
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time", "end_time")
                 if c in base.schema_arrow.names]
    print(f"  columns read: {base_cols}")

    # --- stream the feature chunks, joining on event_id --------------------
    tier1 = pd.concat(
        [pd.read_parquet(p) for p in TIER1_FILES], ignore_index=True
    )
    industrial = pd.concat(
        [pd.read_parquet(p) for p in IND_FILES], ignore_index=True
    )
    print(f"  tier1 rows      : {len(tier1):,}")
    print(f"  industrial rows : {len(industrial):,}")
    for name, frame in (("tier1", tier1), ("industrial", industrial)):
        missing = [c for c in frame.columns if c not in FEATURE_COLUMNS + ["event_id"]]
        if missing:
            raise KeyError(f"{name} chunks have unexpected columns {missing}")
        absent = [c for c in FEATURE_COLUMNS if c not in frame.columns]
        if absent:
            raise KeyError(f"{name} chunks are missing {absent}")

    # Deduplicate defensively: an event must contribute one row.
    tier1 = tier1.drop_duplicates(subset=["event_id"], keep="first")
    industrial = industrial.drop_duplicates(subset=["event_id"], keep="first")
    print(f"  after dedup     : tier1 {len(tier1):,}  "
          f"industrial {len(industrial):,}")

    # --- assemble in chunks to bound memory -------------------------------
    base_ids = None
    x_chunks, y_chunks, meta_chunks = [], [], []
    written = 0

    reader = pq.ParquetFile(base_path)
    for batch_no, batch in enumerate(reader.iter_batches(
            batch_size=CHUNK_EVENTS, columns=base_cols)):
        meta = batch.to_pandas()
        if meta.empty:
            continue
        if base_ids is None:
            base_ids = meta["event_id"]

        joined = (
            meta
            .merge(tier1, on="event_id", how="inner")
            .merge(industrial, on="event_id", how="inner")
        )
        if joined.empty:
            print(f"  batch {batch_no}: join produced 0 rows")
            continue

        # --- derived features -------------------------------------------
        flare_inputs = db.derive_flare_inputs(joined)
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(
            joined["FRP_ANOMALY_VS_SITE"]
        )
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # --- labels -----------------------------------------------------
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )

        # --- split keys -------------------------------------------------
        joined = td.add_split_keys(joined)

        # --- split X / y / meta ------------------------------------------
        x, dropped = db.strip_coordinate_columns(
            joined, extra=[c for c in joined.columns
                           if c in ALL_FEATURES and c not in FEATURE_COLUMNS
                           and c not in DERIVED_FEATURE_COLUMNS]
        )
        x_cols = [c for c in ALL_FEATURES if c in x]
        if x_cols != [c for c in ALL_FEATURES if c in x]:
            raise AssertionError("feature order drifted")

        x_chunks.append(joined[x_cols].copy())
        y_chunks.append(
            joined[["event_id", "label", "label_rule", "label_confidence",
                    "sample_weight"]].copy()
        )
        meta_chunks.append(
            joined[["event_id", "event_centroid_lon", "event_centroid_lat",
                    "start_time", "start_year", "spatial_block", "site_id"]]
            .copy()
        )
        written += len(joined)
        print(f"  batch {batch_no:>3d}  rows={len(joined):>8,}  "
              f"total={written:>10,}  {time.time() - t0:>7.1f}s")

    print()
    print(f"assembled {written:,} events")

    # --- write ---------------------------------------------------------
    X = pd.concat(x_chunks, ignore_index=True)
    y = pd.concat(y_chunks, ignore_index=True)
    meta_all = pd.concat(meta_chunks, ignore_index=True)

    db.assert_no_location_features(list(X.columns))

    X.to_parquet(OUT_DIR / "X.parquet", index=False, compression="snappy")
    y.to_parquet(OUT_DIR / "y.parquet", index=False, compression="snappy")
    meta_all.to_parquet(OUT_DIR / "meta.parquet", index=False,
                        compression="snappy")
    print(f"Wrote {OUT_DIR}/X.parquet    {len(X):,} x {len(X.columns)}")
    print(f"Wrote {OUT_DIR}/y.parquet    {len(y):,}")
    print(f"Wrote {OUT_DIR}/meta.parquet {len(meta_all):,}")

    # --- QC ------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    report = lab.label_report(y["label"], y["label_rule"])
    print(f"  events            : {report['n_events']:,}")
    for cls in lab.LABEL_CLASSES:
        n = report["counts"][cls]
        print(f"  {cls:20s} {n:>10,}  {report['fractions'][cls]:>7.2%}")
    print(f"  supervised        : {report['n_supervised']:,} "
          f"({report['supervised_fraction']:.2%})")
    print(f"  UNKNOWN kept      : {report['n_unknown']:,}")

    plan = td.training_plan(list(X.columns), report["counts"])
    print()
    print(f"  model A features  : {plan['model_a']['n']}")
    print(f"  model B features  : {plan['model_b']['n']}  (non-leak)")
    print(f"  label-derived in B: {plan['leakage_report']['label_derived_in_b']}")
    print(f"  headline split    : {plan['headline_split']}")
    print(f"  review threshold  : {db.REVIEW_CONFIDENCE}")

    dense = [
        c for c in X.columns if X[c].notna().mean() > 0.5
    ]
    print()
    print(f"  features >50% populated: {len(dense)} of {len(X.columns)}")
    print(f"  MISSING_FEATURE_FRACTION mean: "
          f"{float(X['MISSING_FEATURE_FRACTION'].mean()):.3f}")

    manifest = {
        "n_events": int(len(X)),
        "features": list(X.columns),
        "label_counts": report["counts"],
        "label_rules": report["rules"],
        "model_a_n": plan["model_a"]["n"],
        "model_b_n": plan["model_b"]["n"],
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "nan_fraction": {c: float(X[c].isna().mean()) for c in X.columns},
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/manifest.json")
    print()
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
CHUNK_EVENTS = 500_000
SITE_DEG = db.SITE_DEG

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Join, derive, label, and write X / y / meta in chunks."""
    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)

    # --- read the event base once, for coordinates and time only -----------
    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")
    base = pq.ParquetFile(base_path)
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time", "end_time")
                 if c in base.schema_arrow.names]
    print(f"  columns read: {base_cols}")

    # --- stream the feature chunks, joining on event_id --------------------
    tier1 = pd.concat(
        [pd.read_parquet(p) for p in TIER1_FILES], ignore_index=True
    )
    industrial = pd.concat(
        [pd.read_parquet(p) for p in IND_FILES], ignore_index=True
    )
    print(f"  tier1 rows      : {len(tier1):,}")
    print(f"  industrial rows : {len(industrial):,}")
    # Validate each block against ITS OWN expected columns, never the union:
    # the tier-1 file legitimately does not contain the industrial features.
    try:
        db.validate_feature_blocks(
            tier1.columns, industrial.columns,
            expected_tier1=TIER1_EXPECTED,
            expected_industrial=INDUSTRIAL_EXPECTED,
        )
    except ValueError as exc:
        raise KeyError(str(exc)) from exc
    print("  feature blocks validated against their own schemas: PASS")

    # Deduplicate defensively: an event must contribute one row.
    tier1 = tier1.drop_duplicates(subset=["event_id"], keep="first")
    industrial = industrial.drop_duplicates(subset=["event_id"], keep="first")
    print(f"  after dedup     : tier1 {len(tier1):,}  "
          f"industrial {len(industrial):,}")

    # --- assemble in chunks to bound memory -------------------------------
    base_ids = None
    x_chunks, y_chunks, meta_chunks = [], [], []
    written = 0

    reader = pq.ParquetFile(base_path)
    for batch_no, batch in enumerate(reader.iter_batches(
            batch_size=CHUNK_EVENTS, columns=base_cols)):
        meta = batch.to_pandas()
        if meta.empty:
            continue
        if base_ids is None:
            base_ids = meta["event_id"]

        joined = (
            meta
            .merge(tier1, on="event_id", how="inner")
            .merge(industrial, on="event_id", how="inner")
        )
        if joined.empty:
            print(f"  batch {batch_no}: join produced 0 rows")
            continue

        # --- derived features -------------------------------------------
        flare_inputs = db.derive_flare_inputs(joined)
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(
            joined["FRP_ANOMALY_VS_SITE"]
        )
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # --- labels -----------------------------------------------------
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )

        # --- split keys -------------------------------------------------
        joined = td.add_split_keys(joined)

        # --- split X / y / meta ------------------------------------------
        x, dropped = db.strip_coordinate_columns(
            joined, extra=[c for c in joined.columns
                           if c in ALL_FEATURES and c not in FEATURE_COLUMNS
                           and c not in DERIVED_FEATURE_COLUMNS]
        )
        x_cols = [c for c in ALL_FEATURES if c in x]
        if x_cols != [c for c in ALL_FEATURES if c in x]:
            raise AssertionError("feature order drifted")

        x_chunks.append(joined[x_cols].copy())
        y_chunks.append(
            joined[["event_id", "label", "label_rule", "label_confidence",
                    "sample_weight"]].copy()
        )
        meta_chunks.append(
            joined[["event_id", "event_centroid_lon", "event_centroid_lat",
                    "start_time", "start_year", "spatial_block", "site_id"]]
            .copy()
        )
        written += len(joined)
        print(f"  batch {batch_no:>3d}  rows={len(joined):>8,}  "
              f"total={written:>10,}  {time.time() - t0:>7.1f}s")

    print()
    print(f"assembled {written:,} events")

    # --- write ---------------------------------------------------------
    X = pd.concat(x_chunks, ignore_index=True)
    y = pd.concat(y_chunks, ignore_index=True)
    meta_all = pd.concat(meta_chunks, ignore_index=True)

    db.assert_no_location_features(list(X.columns))

    X.to_parquet(OUT_DIR / "X.parquet", index=False, compression="snappy")
    y.to_parquet(OUT_DIR / "y.parquet", index=False, compression="snappy")
    meta_all.to_parquet(OUT_DIR / "meta.parquet", index=False,
                        compression="snappy")
    print(f"Wrote {OUT_DIR}/X.parquet    {len(X):,} x {len(X.columns)}")
    print(f"Wrote {OUT_DIR}/y.parquet    {len(y):,}")
    print(f"Wrote {OUT_DIR}/meta.parquet {len(meta_all):,}")

    # --- QC ------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    report = lab.label_report(y["label"], y["label_rule"])
    print(f"  events            : {report['n_events']:,}")
    for cls in lab.LABEL_CLASSES:
        n = report["counts"][cls]
        print(f"  {cls:20s} {n:>10,}  {report['fractions'][cls]:>7.2%}")
    print(f"  supervised        : {report['n_supervised']:,} "
          f"({report['supervised_fraction']:.2%})")
    print(f"  UNKNOWN kept      : {report['n_unknown']:,}")

    plan = td.training_plan(list(X.columns), report["counts"])
    print()
    print(f"  model A features  : {plan['model_a']['n']}")
    print(f"  model B features  : {plan['model_b']['n']}  (non-leak)")
    print(f"  label-derived in B: {plan['leakage_report']['label_derived_in_b']}")
    print(f"  headline split    : {plan['headline_split']}")
    print(f"  review threshold  : {db.REVIEW_CONFIDENCE}")

    dense = [
        c for c in X.columns if X[c].notna().mean() > 0.5
    ]
    print()
    print(f"  features >50% populated: {len(dense)} of {len(X.columns)}")
    print(f"  MISSING_FEATURE_FRACTION mean: "
          f"{float(X['MISSING_FEATURE_FRACTION'].mean()):.3f}")

    manifest = {
        "n_events": int(len(X)),
        "features": list(X.columns),
        "label_counts": report["counts"],
        "label_rules": report["rules"],
        "model_a_n": plan["model_a"]["n"],
        "model_b_n": plan["model_b"]["n"],
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "nan_fraction": {c: float(X[c].isna().mean()) for c in X.columns},
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/manifest.json")
    print()
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import training_design as td

INSPECT_ONLY = True
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            path = SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            writers[key] = pq.ParquetWriter(path, compression="snappy")
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        frame = frame.assign(_part=parts)
        for part in range(N_PARTITIONS):
            piece = frame[frame["_part"] == part].drop(columns=["_part"])
            if piece.empty:
                continue
            shard_writer(kind, part).write_table(
                pa.Table.from_pandas(piece, preserve_index=False)
            )
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    writers.clear()
    print(f"  shard dir: {SHARD_DIR}")

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    rule_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = pq.ParquetWriter(
                    OUT_DIR / f"{kind}.parquet", compression="snappy"
                )
            store[kind].write_table(
                pa.Table.from_pandas(frame, preserve_index=False)
            )
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {n / total:>7.2%}")
    print(f"    {'supervised':20s} "
          f"{total - label_counts.get(lab.UNKNOWN, 0):>10,}  "
          f"{1 - label_counts.get(lab.UNKNOWN, 0) / total:>7.2%}")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet" for k, w in writers.items() if not w.opened
    ]
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    print(f"  shard dir: {SHARD_DIR}")

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    rule_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {n / total:>7.2%}")
    print(f"    {'supervised':20s} "
          f"{total - label_counts.get(lab.UNKNOWN, 0):>10,}  "
          f"{1 - label_counts.get(lab.UNKNOWN, 0) / total:>7.2%}")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

t0 = time.time()
X = pd.read_parquet(DATASET_DIR / "X.parquet")
y = pd.read_parquet(DATASET_DIR / "y.parquet")
meta = pd.read_parquet(DATASET_DIR / "meta.parquet")
print(f"loaded X {X.shape}  y {y.shape}  meta {meta.shape}  "
      f"({time.time() - t0:.1f}s)")

# Hard gate: no coordinate may be in the model.
db.assert_no_location_features(list(X.columns))
print("location check: PASS")

labels = y["label"].to_numpy()
weights = y["sample_weight"].to_numpy()

# The headline target is the class we can actually evidence today. The other
# five classes need land cover and weather, so they remain UNKNOWN until the
# GEE groups land -- the brief's escalation path starts with what is measurable.
TARGET = "INDUSTRIAL_FLARE"
print()
print(f"target class: {TARGET}")
print("class counts:")
for cls in lab.LABEL_CLASSES:
    print(f"  {cls:20s} {int((labels == cls).sum()):>10,}")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
parts = td.make_split(meta, kind=td.SPLIT_COMBINED, seed=0)
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
td.assert_no_time_leakage(meta[is_train], meta[is_test])
print("  leakage checks: PASS")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------

non_leak = [c for c in X.columns if c not in td.LABEL_DERIVED]
label_derived = [c for c in X.columns if c in td.LABEL_DERIVED]
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X.loc[is_train, non_leak], y_bin[is_train], weights[is_train],
    X.loc[is_test, non_leak], y_bin[is_test], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[is_test], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[is_test], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {X.shape[1]} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X.loc[is_train, list(X.columns)], y_bin[is_train], weights[is_train],
    X.loc[is_test, list(X.columns)], y_bin[is_test], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[is_test], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[is_test], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

dist = X["DIST_NEAREST_FACILITY_LOG"].to_numpy()
dist_m = np.where(np.isfinite(dist), np.power(10.0, dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    dist_m[is_train], y_bin[is_train]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
print("PER-CLASS METRICS (all classes, at the operating threshold)")
print("=" * 74)
for key in ("B", "A"):
    pred_cls = np.full(len(y_bin[is_test]), lab.UNKNOWN, dtype=object)
    confident = prob_b if key == "B" else prob_a
    accepted = confident >= db.REVIEW_CONFIDENCE
    pred_cls[accepted] = TARGET
    report = per_class_report(
        labels[is_test], pred_cls,
        [TARGET, lab.INDUSTRIAL_FIRE, lab.MINING, lab.AGRICULTURAL,
         lab.WILDFIRE, lab.UNKNOWN],
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(X.columns), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
print("Models are in memory only. Nothing has been written to Drive except")
print("the two QC files above.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    rule_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "fire_labels": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# COLAB CELL -- Phase 0 gap analysis + rule diagnostics
#
# Run this INSTEAD of P7. It answers, with numbers:
#   1. which of the 141 specified features are actually present
#   2. which are present but mostly NaN (kept -- LightGBM routes NaN natively)
#   3. which are missing entirely, and how each one can be obtained
#   4. why every event came out `no_rule` -- the blocking condition, named
#   5. why FRP_ANOMALY_VS_SITE was 99.5% NaN, and what grid fixes it
#
# Reads X.parquet / y.parquet / meta.parquet. Writes two CSVs to qc/.
# ============================================================================

import inspect
import json
from pathlib import Path

import numpy as np
import pandas as pd

import colab_utils as cu
import feature_gap as fg
import feature_ledger as fl
import training_design as td
import weak_labels as wl

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

print("=" * 78)
print("PHASE 0 -- GAP ANALYSIS AND RULE DIAGNOSTICS")
print("=" * 78)
print(f"ledger {fl.LEDGER_VERSION}   spec size {len(fl.FEATURES_141)}")
print()
print("MODULE STAMPS -- if these differ from your repo, you are running a")
print("stale pasted cell, and its output is missing whatever changed.")
print("-" * 78)
for module in (fl, fg, wl, td, cu):
    try:
        size = Path(inspect.getfile(module)).stat().st_size
        print(f"  {module.__name__:<20s} {size:>9,} bytes")
    except Exception as exc:                       # noqa: BLE001
        print(f"  {module.__name__:<20s} !! {exc}")
print("-" * 78)

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"
QC_DIR.mkdir(parents=True, exist_ok=True)

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{name} missing. Run colab_p6_dataset.py first."
        )

print()
print("loading X, y, meta ...")
X = pd.read_parquet(DATASET_DIR / "X.parquet")
y = pd.read_parquet(DATASET_DIR / "y.parquet")
meta = pd.read_parquet(DATASET_DIR / "meta.parquet")
print(f"  X {X.shape}   y {y.shape}   meta {meta.shape}")

# ===========================================================================
# 1. THE GAP
# ===========================================================================

print()
print("=" * 78)
print("1. FEATURE GAP vs THE 141-FEATURE SPEC")
print("=" * 78)

report = fg.gap_report(X)
summary = fg.summarise(report)

print(f"  PRESENT (usable) : {summary['n_usable']:>4}")
print(f"  SPARSE (>90% NaN): {summary['n_sparse']:>4}   kept -- LightGBM "
      "routes NaN natively")
print(f"  MISSING          : {summary['n_missing']:>4}")
print(f"  coverage         : {summary['coverage_frac']:>6.1%} of the spec")
print()
print("  SPARSE columns (weak signal, not broken -- do NOT impute):")
sparse = report[report["status"] == "SPARSE"].sort_values("nan_fraction")
for _, r in sparse.iterrows():
    print(f"    {r['feature']:32s} {r['nan_fraction']:>7.1%} NaN")

print()
print("  MISSING features by route to obtain them:")
for route, count in sorted(summary["obtain_for_missing"].items(),
                           key=lambda kv: -kv[1]):
    print(f"    {route:18s} {count:>4}")

# `missing_by_route` keys are (route, group) pairs.
grouped = fg.missing_by_route(report)
for route in ("DERIVE", "EXTERNAL_RASTER", "EXTERNAL_TABLE"):
    keys = sorted(k for k in grouped if k[0] == route)
    if not keys:
        continue
    print()
    print(f"  --- {route} ---")
    for route_key, group in keys:
        feats = grouped[(route_key, group)]
        print(f"    {group:24s} {len(feats):>3d}  e.g. {', '.join(feats[:3])}")

# The DERIVE list is short enough to print in full, and it is the work that
# needs no download at all -- so it is the first thing worth doing.
derive_feats = sorted(
    f for f in report[report["obtain"] == "DERIVE"]["feature"]
)
if derive_feats:
    print()
    print(f"  DERIVE in full ({len(derive_feats)} -- no downloads needed):")
    for i in range(0, len(derive_feats), 3):
        print("    " + "  ".join(f"{name:30s}"
                                for name in derive_feats[i:i + 3]))

# ===========================================================================
# 2. WHY IS EVERYTHING no_rule?
# ===========================================================================

print()
print("=" * 78)
print("2. WHY EVERY EVENT CAME OUT no_rule -- per-condition pass counts")
print("=" * 78)

if "label_weak" in y.columns:
    lab_col = "label_weak"
    print(f"  using weak labels from column '{lab_col}'")
else:
    lab_col = "label"
    print(f"  column 'label_weak' absent -- using '{lab_col}'.")
    print("  That means colab_p6_dataset.py ran an older pasted cell; the weak")
    print("  hierarchy and the missing-input diagnostics were not applied.")

labelling = X.copy()
for column in ("DIST_REFINERY_LOG", "DIST_NEAREST_FACILITY_LOG",
               "FACILITY_COUNT_5KM", "DIST_POWERPLANT_LOG",
               "DIST_PETROCHEMICAL_LOG", "DIST_STEEL_LOG",
               "DIST_CEMENT_LOG", "DIST_MINE_LOG", "AREA_GROWTH_RATE",
               "SPREAD_SPEED_MEDIAN", "DIRECTION_CONSISTENCY",
               "TIME_TO_PEAK", "FRP_PERSISTENCE", "DETECTION_COUNT_LOG",
               "INSIDE_FACILITY", "KNOWN_FLARE"):
    if column in X.columns:
        labelling[column] = X[column]

conditions = wl.condition_pass_report(labelling, wl.WIDE)
for cls in ("INDUSTRIAL_FLARE", "INDUSTRIAL_FIRE", "MINING", "AGRICULTURAL",
            "WILDFIRE"):
    block = conditions[conditions["rule"] == cls]
    print()
    print(f"  {cls}")
    print(block[["condition", "n_pass", "pass_frac"]].to_string(index=False))

print()
print("  Label distribution actually produced:")
counts = y[lab_col].value_counts()
for cls, n in counts.items():
    print(f"    {cls:22s} {n:>10,}  {100.0 * n / max(len(y), 1):>6.2f}%")

# ===========================================================================
# 3. WHY FRP_ANOMALY_VS_SITE WAS 99.5% NaN
# ===========================================================================

print()
print("=" * 78)
print("3. SITE GRID -- why the anomaly column is nearly empty")
print("=" * 78)

lon = meta["event_centroid_lon"].to_numpy()
lat = meta["event_centroid_lat"].to_numpy()
table, chosen = fg.recommend_site_deg(lon, lat)
print(table.to_string(index=False))
print()
import dataset_builder as db
print(f"  currently used: SITE_DEG = {db.SITE_DEG} deg")
print(f"  recommended   : {chosen} deg")
if chosen != db.SITE_DEG:
    print(f"  -> change SITE_DEG in dataset_builder.py to {chosen} and re-run P6")
    print("     to populate FRP_ANOMALY_VS_SITE")
print()
if "FRP_ANOMALY_VS_SITE" in X.columns:
    print(f"  FRP_ANOMALY_VS_SITE populated: "
          f"{X['FRP_ANOMALY_VS_SITE'].notna().mean():.2%}")
print("  A per-site baseline needs company: a cell holding exactly one event")
print("  has nothing to compare against, so the ratio is undefined by design.")
print("  Coarsen the grid, or define sites by proximity to a facility.")

# ===========================================================================
# 4. WHAT MODEL B CAN STILL USE
# ===========================================================================

print()
print("=" * 78)
print("4. LEAKAGE PARTITION -- what the scientific model may use")
print("=" * 78)
model_a, model_b, leakage = td.split_feature_sets(list(X.columns))
print(f"  model A (operational) : {len(model_a):>3} features")
print(f"  model B (scientific) : {len(model_b):>3} features")
print(f"  excluded as label inputs: "
      f"{leakage['n_label_derived_in_a']} industrial + "
      f"{leakage['n_weak_rule_inputs_in_a']} weak-rule")
print()
print("  model B keeps:")
for name in model_b:
    print(f"    {name}")
if len(model_b) <= 12:
    print()
    print("  !! Model B is nearly empty. A result from it would be thin evidence,")
    print("     and the honest headline stays model A plus an independent")
    print("     hand-labelled sample.")

# ===========================================================================
# 5. WRITE
# ===========================================================================

gap_path = QC_DIR / "p0_feature_gap.csv"
cond_path = QC_DIR / "p0_rule_conditions.csv"
site_path = QC_DIR / "p0_site_occupancy.csv"
report.to_csv(gap_path, index=False)
conditions.to_csv(cond_path, index=False)
table.to_csv(site_path, index=False)

print()
print(f"Wrote {gap_path}")
print(f"Wrote {cond_path}")
print(f"Wrote {site_path}")
print()
print("NEXT, in this order")
print("  1. Implement the DERIVE rows first -- no downloads, and they are the")
print("     features the rule engine is waiting on.")
print("  2. Join a real flare catalogue (VIIRS Nightfire / GGFR). That is what")
print("     makes KNOWN_FLARE non-null and gives INDUSTRIAL_FLARE a seed.")
print("  3. One Earth Engine pass for land cover: it unblocks the AGRICULTURAL")
print("     and WILDFIRE rules, which currently have no discriminating input.")
print("  4. Re-run P6, then P7.")

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
#: Every module the pipeline imports. A module missing from this list is not
#: reported until some later cell dies with a bare ImportError, which is how
#: cluster_tier / weak_labels / dataset_builder went missing twice.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "cluster_tier": None,
    "fire_labels": None,
    "weak_labels": None,
    "dataset_builder": None,
    "training_design": None,
    "feature_gap": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import cluster_tier as ct
import dataset_builder as db
import feature_gap as fgap
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf
import training_design as td
import weak_labels as weak


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial block does not match the ledger")
if ct.N_CLUSTER_TIER + len(weak.LABEL_INPUT_FEATURES) < 1:
    problems.append("cluster_tier / weak_labels look empty")
if len(fl.FEATURES_141) != 141:
    problems.append(f"ledger has {len(fl.FEATURES_141)} features, expected 141")
if not hasattr(db, "SITE_DEG") or not hasattr(fgap, "gap_report"):
    problems.append("dataset_builder / feature_gap are stale")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print(f"  weak labels : {len(weak.LABEL_INPUT_FEATURES)} rule inputs published")
print(f"  site grid   : {db.SITE_DEG} deg")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
WEAK_VARIANT = [weak.WIDE, weak.TIGHT]
WEAK_VARIANT_NAMES = ["wide(1-10 km)", "tight(2/5/10 m)"]

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        weak_out = weak.assign_weak_labels(joined, WEAK_VARIANT[0])
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_weak"] = weak_out["label"]
        joined["label_weak_rule"] = weak_out["rule"]
        joined["label_weak_source"] = weak_out["label_source"]
        for cls, n_ in Counter(weak_out["label"]).items():
            weak_counts[cls] += n_
        for entry, n_missing in Counter(
                label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "label_weak", "label_weak_rule", "label_weak_source"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS (industrial-context hierarchy; manufactured from")
    print("  features the model sees, so never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm, n in missing_counter.most_common(20):
            print(f"    {nm:32s} {n:>10,} events blocked")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: int(v) for k, v in missing_counter.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_variant": WEAK_VARIANT_NAMES[0],
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

In [ ]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
#: Every module the pipeline imports. A module missing from this list is not
#: reported until some later cell dies with a bare ImportError, which is how
#: cluster_tier / weak_labels / dataset_builder went missing twice.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "cluster_tier": None,
    "fire_labels": None,
    "weak_labels": None,
    "dataset_builder": None,
    "training_design": None,
    "feature_gap": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import cluster_tier as ct
import dataset_builder as db
import feature_gap as fgap
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf
import training_design as td
import weak_labels as weak


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial block does not match the ledger")
if ct.N_CLUSTER_TIER + len(weak.LABEL_INPUT_FEATURES) < 1:
    problems.append("cluster_tier / weak_labels look empty")
if len(fl.FEATURES_141) != 141:
    problems.append(f"ledger has {len(fl.FEATURES_141)} features, expected 141")
if not hasattr(db, "SITE_DEG") or not hasattr(fgap, "gap_report"):
    problems.append("dataset_builder / feature_gap are stale")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print(f"  weak labels : {len(weak.LABEL_INPUT_FEATURES)} rule inputs published")
print(f"  site grid   : {db.SITE_DEG} deg")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

In [ ]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
WEAK_VARIANT = [weak.WIDE, weak.TIGHT]
WEAK_VARIANT_NAMES = ["wide(1-10 km)", "tight(2/5/10 m)"]

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        weak_out = weak.assign_weak_labels(joined, WEAK_VARIANT[0])
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_weak"] = weak_out["label"]
        joined["label_weak_rule"] = weak_out["rule"]
        joined["label_weak_source"] = weak_out["label_source"]
        for cls, n_ in Counter(weak_out["label"]).items():
            weak_counts[cls] += n_
        for entry, n_missing in Counter(
                label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "label_weak", "label_weak_rule", "label_weak_source"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS (industrial-context hierarchy; manufactured from")
    print("  features the model sees, so never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm, n in missing_counter.most_common(20):
            print(f"    {nm:32s} {n:>10,} events blocked")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: int(v) for k, v in missing_counter.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_variant": WEAK_VARIANT_NAMES[0],
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()